# MediFlow 4차 · 실제 촬영 재대조와 조명 강도 재설정

**GPU 선택 → 모두 실행 → Google Drive 연결**로 실행합니다.
기존 공통 ZIP v3·원본 데이터 ZIP·1차와 2차 결과를 재사용합니다.
새 실행 ZIP 업로드나 Target 복호화 키 입력은 필요 없습니다.

피부 초기 B0·224, 얼굴 선정 PMG·B0·256, 두피 선정 B1·384의 1차 학습 전 상태에서
각 5 epoch, 총 세 분야를 순서대로 학습합니다. 이전 확대 후보는 학습하지 않았습니다.

실제 원본 124장과 유지된 Train 표본 461장을 다시 측정해 밝기·대비·방향성 조명을 맞췄습니다.
조명 조정 후 색·흐림·미세 대비를 다시 점검했습니다. 네 사람의 촬영 특성 조사 결과이며
임상적인 병변 보존이나 실제 환자 분류 성능을 검증한 설정은 아닙니다.
아래는 실행에 쓰는 실제 설정입니다.
시작할 때 세 분야의 출발 모델·기존 결과·분할과 전체 입력 캐시 공간을 먼저 확인합니다.

| 분야 | 밝기 이동 | 대비 | 좌우 조명 | Lab a | Lab b | 미세 대비 | 흐림 sigma · ROI 64 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| skin | `[67.47361687519953, 71.01787172836876]` | `[0.25, 0.26459406014955544]` | `[0.0, 0.0]` | `[-0.9609375, 0.0]` | `[0.0, 0.92578125]` | `[1, 1]` | `[0, 0]` |
| web_skin | `[-99.7760789846667, -56.06404023208951]` | `[1.0, 1.0]` | `[3.5654401022290427, 64.0]` | `[0, 0]` | `[0, 0]` | `[0.85, 0.9145815899839365]` | `[0, 0.4]` |
| hair | `[15.043113708496094, 26.26080894470215]` | `[1.0, 1.0]` | `[0.0, 0.0]` | `[0, 0]` | `[0, 0]` | `[1, 1]` | `[0, 0]` |

이번 변경은 촬영 변환 설정입니다.
데이터 분할·출발 모델·학습량·시드·학습률·변환 확률은 기존 비교와 같습니다.
실제 사진과의 동일함이나 질환 분류 성능 향상은 검증되지 않았습니다.
이미 확인한 Source Test는 이 노트북에서 다시 추론하지 않습니다.
결과는 학습 전·기존 2차·이번 수정본을 같은 원본·기존 변환·수정 변환 Validation으로 비교합니다.
두피의 대비를 전체 촬영 외형에 강제로 맞추지 않았습니다. 모발 가림·굵기·부위가 섞여 있습니다.

Drive 결과 폴더
`MediFlow_adaptation_runs/4ae6e07eb4af1225/measured_strength_e6bc85bc0a7d`

화면에 epoch·배치·검증·저장을 표시하고 저장된 회차부터 이어 실행합니다.
완료하면 학습곡선과 비교표를 표시하고 `MediFlow_Revised_Camera_Results_20261111.zip`을
다운로드합니다. Drive의 위 결과 폴더에도 같은 이름으로 보관합니다.


In [ ]:
import hashlib
import json
import os
import shutil
import signal
import stat
import subprocess
import sys
import tempfile
import uuid
import zipfile
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
from google.colab import drive, files

def digest(path):
    value = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            value.update(chunk)
    return value.hexdigest()

def unpack_bundle(bundle, destination, expected_hash):
    """Verify the previously supplied bundle before using any of its Python files."""
    if digest(bundle) != expected_hash:
        raise ValueError(
            "선택한 ZIP이 준비된 실행 ZIP과 다릅니다. MediFlow_Adaptation_Common_v3.zip을 선택하세요."
        )
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=False)
    with zipfile.ZipFile(bundle) as archive:
        manifest = json.loads(archive.read("bundle_manifest.json"))
        members = {}
        for info in archive.infolist():
            name = info.orig_filename
            parts = PurePosixPath(name).parts
            if (
                not parts
                or PurePosixPath(name).is_absolute()
                or ".." in parts
                or "\\" in name
                or ":" in name
                or stat.S_ISLNK(info.external_attr >> 16)
                or name.casefold() in members
            ):
                raise ValueError("안전하지 않거나 중복된 ZIP 경로: " + name)
            members[name.casefold()] = info
        files = {info.filename for info in members.values() if not info.is_dir()}
        if files != set(manifest) | {"bundle_manifest.json"}:
            raise ValueError("실행 ZIP의 파일 목록이 검증 목록과 다릅니다.")
        for name, expected in manifest.items():
            body = archive.read(name)
            if hashlib.sha256(body).hexdigest() != expected:
                raise ValueError("실행 파일 검사 실패: " + name)
            path = destination / name
            if not path.resolve().is_relative_to(destination):
                raise ValueError("ZIP 경로가 실행 폴더 밖을 가리킵니다.")
            path.parent.mkdir(parents=True, exist_ok=True)
            with path.open("xb") as stream:
                stream.write(body)
    return destination

drive.mount("/content/drive")
my_drive = Path("/content/drive/MyDrive")
project = my_drive / "mediflow_Project"
bundle_sha = "4ae6e07eb4af122531439519ae2207aad57a1c8dfb8f158152a2093fa65a91fe"
parent = my_drive / "MediFlow_adaptation_runs" / bundle_sha[:16]
out = parent / "measured_strength_e6bc85bc0a7d"
logs = out / "logs"
logs.mkdir(parents=True, exist_ok=True)
run_id = uuid.uuid4().hex[:12]
log_path = logs / f"run_{run_id}.log"
print("세 분야 촬영 변환 수정본 · 각 5 epoch", flush=True)
print("결과 저장 위치", out, flush=True)
with tempfile.TemporaryDirectory(prefix="mediflow_strength_", dir="/content") as temporary:
    work = Path(temporary)
    local_bundle = work / "MediFlow_Adaptation_Common_v3.zip"
    shutil.copyfile(project / "MediFlow_Adaptation_Common_v3.zip", local_bundle)
    runtime = unpack_bundle(local_bundle, work / "runtime", bundle_sha)
    local_bundle.unlink()
    for name, content in {'adaptation_strength.py': b'"""Validation-only follow-up with frozen, stronger measured camera transformations."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport shutil\nimport subprocess\nimport tempfile\nimport uuid\nfrom pathlib import Path\nfrom types import SimpleNamespace\n\nfrom mediflow_datasets import adaptation_stage1_colab as colab\nfrom mediflow_datasets import adaptation_stage2 as previous\nfrom mediflow_datasets import domain_adaptation_batch as batch\nfrom mediflow_datasets.domain_adaptation_io import write_progress\n\n\ndef frozen_inputs(runtime, stage1, stage2, study, domain):\n    """Check required artifacts without importing TensorFlow or copying source archives."""\n    config, plan = previous.starting_plan(runtime, stage1, study["bundle_sha256"])\n    previous.require_stage2(stage2, study["bundle_sha256"])\n    task, spec = config["tasks"][domain], plan[domain]["spec"]\n    entry = study["domains"][domain]\n    reference = previous.read_json(runtime / "references" / f"{domain}.json")\n    profile, weak = entry["profile"], reference["extended_profile"]\n    record = previous.read_json(stage2 / domain / "complete.json")\n    prior_configuration = previous.read_json(stage2 / domain / "run_configuration.json")\n    manifest = stage1 / domain / "source_manifest.json"\n    initial = (runtime / spec["model"]).resolve()\n    if not initial.is_relative_to(runtime.resolve()) or not initial.is_file():\n        raise ValueError(f"{domain}: starting model is missing or outside the runtime")\n    old_model = stage2 / domain / "best.keras"\n\n    def object_hash(value):\n        return hashlib.sha256(\n            json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":")).encode()\n        ).hexdigest()\n\n    if (\n        object_hash(config) != study["configuration_sha256"]\n        or object_hash(reference) != entry["previous_reference_sha256"]\n        or batch.digest(manifest) != entry["source_manifest_sha256"]\n        or batch.digest(stage2 / domain / "source_manifest.json") != entry["source_manifest_sha256"]\n        or batch.digest(initial) != spec["model_sha256"]\n        or batch.digest(old_model) != record["best_model"]["sha256"]\n        or record["best_model"]["sha256"] != entry["previous_best_sha256"]\n        or record["model_config"] != spec\n        or record["stage1_best"] != plan[domain]["stage1_best"]\n        or prior_configuration["profile"] != weak\n        or prior_configuration["training_configuration"] != config\n        or profile["domain"] != domain\n        or profile["schema"] != "measured_camera_extended_v1"\n    ):\n        raise ValueError(f"{domain}: frozen inputs or comparison model changed")\n    return SimpleNamespace(\n        config=config,\n        task=task,\n        spec=spec,\n        entry=entry,\n        reference=reference,\n        profile=profile,\n        weak=weak,\n        manifest=manifest,\n        initial=initial,\n        old_model=old_model,\n    )\n\n\ndef cache_bytes(train_count, validation_count, pair_count, size):\n    # Original Train + original Validation + lighting/expanded/previous Validation.\n    return (train_count + pair_count + 4 * validation_count) * size * size * 3 * 4\n\n\ndef require_cache_space(local, train_count, validation_count, pair_count, size):\n    needed = cache_bytes(train_count, validation_count, pair_count, size) + 1024**3\n    if shutil.disk_usage(local).free < needed:\n        raise RuntimeError(\n            f"\xec\x9e\x85\xeb\xa0\xa5 \xec\xba\x90\xec\x8b\x9c\xc2\xb7\xec\x9d\xb4\xec\xa0\x84 \xec\xa1\xb0\xea\xb1\xb4 \xeb\xb9\x84\xea\xb5\x90 \xec\xba\x90\xec\x8b\x9c \xea\xb3\xb5\xea\xb0\x84 \xeb\xb6\x80\xec\xa1\xb1 (\xec\x97\xac\xec\x9c\xa0 \xed\x8f\xac\xed\x95\xa8 {needed / 1024**3:.2f} GiB \xed\x95\x84\xec\x9a\x94)"\n        )\n    return needed\n\n\ndef startup_preflight(runtime, root, study, project, work, destination):\n    report, sources = {}, {}\n    for domain in colab.DOMAINS:\n        frozen = frozen_inputs(runtime, root / "stage1", root / "stage2", study, domain)\n        manifest = previous.read_json(frozen.manifest)\n        original = manifest["original"]\n        amount = cache_bytes(\n            len(original["train"]),\n            len(original["val"]),\n            len(manifest["skin_preparation_pairs"]),\n            frozen.spec["size"],\n        )\n        completed = (destination / domain / "complete.json").is_file()\n        if not completed:\n            source = batch.find_source(project / "datasets", frozen.task)\n            sources[domain] = source\n            if shutil.disk_usage(work).free < amount + source.stat().st_size + 1024**3:\n                raise RuntimeError(\n                    f"{domain}: \xeb\x8d\xb0\xec\x9d\xb4\xed\x84\xb0 ZIP\xea\xb3\xbc \xec\xa0\x84\xec\xb2\xb4 \xec\x9e\x85\xeb\xa0\xa5 \xec\xba\x90\xec\x8b\x9c\xeb\xa5\xbc \xec\x9c\x84\xed\x95\x9c \xea\xb3\xb5\xea\xb0\x84\xec\x9d\xb4 \xeb\xb6\x80\xec\xa1\xb1\xed\x95\xa9\xeb\x8b\x88\xeb\x8b\xa4."\n                )\n        report[domain] = {\n            "starting_model_sha256": frozen.spec["model_sha256"],\n            "previous_model_sha256": frozen.entry["previous_best_sha256"],\n            "source_manifest_sha256": frozen.entry["source_manifest_sha256"],\n            "classes": frozen.task["classes"],\n            "input_size": frozen.spec["size"],\n            "batch_size": frozen.task["batch_size"],\n            "input_cache_bytes": amount,\n            "source_zip": str(sources[domain]) if domain in sources else None,\n            "completed_result_available": completed,\n        }\n        print(\n            f"[{domain}] \xec\xb6\x9c\xeb\xb0\x9c \xeb\xaa\xa8\xeb\x8d\xb8\xc2\xb7\xea\xb8\xb0\xec\xa1\xb4 \xea\xb2\xb0\xea\xb3\xbc\xc2\xb7\xeb\xb6\x84\xed\x95\xa0 \xed\x99\x95\xec\x9d\xb8 \xc2\xb7 \xec\x9e\x85\xeb\xa0\xa5 \xec\xba\x90\xec\x8b\x9c {amount / 1024**3:.2f} GiB",\n            flush=True,\n        )\n    return report, sources\n\n\ndef run_domain(runtime, stage1, stage2, study_path, domain, archive, output, local):\n    import cv2\n    import keras\n    import numpy as np\n    import tensorflow as tf\n\n    from mediflow_datasets import adaptation_stage1 as train\n    from mediflow_datasets import camera_extended\n\n    study = previous.read_json(study_path)\n    frozen = frozen_inputs(runtime, stage1, stage2, study, domain)\n    config, task, spec = frozen.config, frozen.task, frozen.spec\n    entry, reference, profile = frozen.entry, frozen.reference, frozen.profile\n    weak, manifest, initial, old_model = (\n        frozen.weak,\n        frozen.manifest,\n        frozen.initial,\n        frozen.old_model,\n    )\n    identity = {\n        "study_sha256": batch.digest(study_path),\n        "runner_sha256": batch.digest(__file__),\n        "training_sha256": batch.digest(train.__file__),\n        "camera_transform_sha256": batch.digest(camera_extended.__file__),\n        "source_manifest_sha256": entry["source_manifest_sha256"],\n    }\n    output.mkdir(parents=True, exist_ok=True)\n    marker = output / "identity.json"\n    if marker.exists() and previous.read_json(marker) != identity:\n        raise ValueError("Output belongs to another study or code version")\n    write_progress(marker, identity)\n    complete = output / "complete.json"\n    if complete.exists():\n        saved = previous.read_json(complete)\n        if batch.digest(output / "best.keras") != saved["best_model"]["sha256"]:\n            raise ValueError("Completed checkpoint does not match its report")\n        print(f"[{domain}] \xec\x99\x84\xeb\xa3\x8c \xeb\xb6\x84\xec\x95\xbc \xec\x9e\xac\xec\x82\xac\xec\x9a\xa9", flush=True)\n        return\n    cv2.setNumThreads(1)\n    rows, val, pairs = train.prepare_source(domain, archive, task, reference, output)\n    if previous.read_json(output / "source_manifest.json") != previous.read_json(manifest):\n        raise ValueError("Original Train/Validation/Test split changed")\n    write_progress(\n        output / "run_configuration.json",\n        {\n            "identity": identity,\n            "study": study,\n            "model": spec,\n            "training_configuration": config,\n            "profile": profile,\n            "previous_profile": weak,\n            "tensorflow": tf.__version__,\n            "keras": keras.__version__,\n            "numpy": np.__version__,\n            "opencv": cv2.__version__,\n            "gpu": [\n                tf.config.experimental.get_device_details(d)\n                for d in tf.config.list_physical_devices("GPU")\n            ],\n            "source_test_evaluated": False,\n            "prior_source_test_inspected": True,\n        },\n    )\n    size, batch_size = spec["size"], task["batch_size"]\n    require_cache_space(local, len(rows), len(val), len(pairs), size)\n    cache, shifted, pair_indexes = train.image_cache(\n        local, archive, rows, val, pairs, task, size, profile["lighting"], expanded=profile\n    )\n    old_validation = np.lib.format.open_memmap(\n        local / "previous_validation.npy",\n        mode="w+",\n        dtype=np.float32,\n        shape=(len(val), size, size, 3),\n    )\n    for index, row in enumerate(val):\n        old_validation[index] = camera_extended.transform(\n            cache[len(rows) + index], weak, "expanded", train.exp.item_seed(row, 1707)\n        )\n    old_validation.flush()\n    shifted["weak"] = old_validation\n    keras.utils.set_random_seed(config["seed"])\n    model = keras.models.load_model(initial, compile=False)\n    train.exp.configure_partial(model)\n    best = train.phase(\n        model,\n        (config, task, rows, val, profile, cache, shifted, pair_indexes),\n        output / "expanded",\n        local / "saving",\n        spec,\n        "expanded",\n        config["epochs"],\n        config["learning_rate"],\n    )\n    saved = train.export_model(model, output, "best", local / "saving")\n    datasets = train.validation_pipelines(cache, shifted, len(rows), val, batch_size, size)\n    metrics = {"stronger_expanded": {condition: best[condition] for condition in datasets}}\n    del model\n    for name, path in (("initial", initial), ("previous_expanded", old_model)):\n        keras.backend.clear_session()\n        model = keras.models.load_model(path, compile=False)\n        metrics[name] = train.validate(\n            train.predictor(model, spec),\n            datasets,\n            val,\n            len(task["classes"]),\n            f"[{domain} \xc2\xb7 {name}]",\n        )\n        del model\n    write_progress(\n        complete,\n        {\n            "best_model": saved,\n            "best_epoch": best["epoch"],\n            "model_config": spec,\n            "validation": metrics,\n            "source_test_evaluated": False,\n            "target_accuracy": None,\n            "historical_checkpoint_independence_verified": False,\n            "epoch_selection": "max min(clean,strong expanded) Macro-F1, then clean",\n        },\n    )\n    print(f"[{domain}] \xed\x9b\x84\xec\x86\x8d \xed\x95\x99\xec\x8a\xb5 \xec\x99\x84\xeb\xa3\x8c \xc2\xb7 \xec\xb5\x9c\xec\xa0\x81 {best[\'epoch\']}\xed\x9a\x8c\xec\xb0\xa8", flush=True)\n\n\ndef comparison(output):\n    records = {\n        domain: previous.read_json(output / domain / "complete.json") for domain in colab.DOMAINS\n    }\n    selected = {}\n    arms = ("initial", "previous_expanded", "stronger_expanded")\n    with (output / "validation_comparison.csv").open("w", encoding="utf-8-sig", newline="") as f:\n        writer = csv.writer(f)\n        writer.writerow(["domain", "model", "condition", "accuracy", "macro_f1"])\n        for domain, record in records.items():\n            metrics = record["validation"]\n            selected[domain] = max(\n                arms,\n                key=lambda arm: (\n                    min(metrics[arm][c]["macro_f1"] for c in ("clean", "weak", "expanded")),\n                    metrics[arm]["clean"]["macro_f1"],\n                    -arms.index(arm),\n                ),\n            )\n            for arm in arms:\n                for condition, result in metrics[arm].items():\n                    writer.writerow(\n                        [domain, arm, condition, result["accuracy"], result["macro_f1"]]\n                    )\n    write_progress(\n        output / "selection.json",\n        {\n            "selected_on_validation": selected,\n            "criteria": (\n                "max worst(clean, previous expanded, stronger expanded) F1; clean; earlier arm"\n            ),\n            "source_test_evaluated": False,\n            "prior_source_test_inspected": True,\n        },\n    )\n\n\ndef plots(output):\n    import matplotlib\n\n    matplotlib.use("Agg")\n    from matplotlib import pyplot as plt\n\n    fig, axes = plt.subplots(3, 2, figsize=(13, 10), layout="constrained")\n    try:\n        for row, domain in enumerate(colab.DOMAINS):\n            history = previous.read_json(output / domain / "expanded/history.json")\n            axes[row, 0].plot(\n                [r["epoch"] for r in history],\n                [r["train_loss_mean_per_update"] for r in history],\n                marker="o",\n            )\n            axes[row, 0].set_title(f"{domain} \xc2\xb7 training loss")\n            for condition in ("clean", "weak", "expanded"):\n                axes[row, 1].plot(\n                    [r["epoch"] for r in history],\n                    [r[condition]["macro_f1"] for r in history],\n                    marker="o",\n                    label=condition,\n                )\n            axes[row, 1].set_title(f"{domain} \xc2\xb7 Validation Macro-F1")\n            axes[row, 1].legend()\n            for axis in axes[row]:\n                axis.set_xlabel("Epoch")\n                axis.grid(alpha=0.2)\n        fig.suptitle("Measured stronger camera adaptation \xc2\xb7 Validation only", fontsize=16)\n        (output / "plots").mkdir(exist_ok=True)\n        fig.savefig(output / "plots/learning_curves.png", dpi=180)\n    finally:\n        plt.close(fig)\n\n\ndef launch(runtime, work, project, study_path, my_drive):\n    import fcntl\n\n    study = previous.read_json(study_path)\n    study_sha = batch.digest(study_path)\n    root = my_drive / "MediFlow_adaptation_runs" / study["bundle_sha256"][:16]\n    destination = root / ("measured_strength_" + study_sha[:12])\n    destination.mkdir(parents=True, exist_ok=True)\n    with (work.parent / f"mediflow_strength_{study_sha}.lock").open("a") as lock:\n        fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)\n        readiness, sources = startup_preflight(runtime, root, study, project, work, destination)\n        write_progress(destination / "startup_preflight.json", readiness)\n        python, versions = batch.prepare_python(work, executor=colab.run_live)\n        if not versions["gpu"]:\n            raise RuntimeError("Colab\xec\x97\x90\xec\x84\x9c GPU \xeb\x9f\xb0\xed\x83\x80\xec\x9e\x84\xec\x9d\x84 \xec\x84\xa0\xed\x83\x9d\xed\x95\x98\xec\x84\xb8\xec\x9a\x94.")\n        env = batch.python_environment(python, runtime)\n        env.update(MPLBACKEND="Agg", PYTHONIOENCODING="utf-8", PYTHONDONTWRITEBYTECODE="1")\n        if subprocess.run(\n            [str(python), "-c", "import matplotlib"], env=env, capture_output=True\n        ).returncode:\n            colab.run_live(\n                [str(python), "-m", "pip", "install", "matplotlib>=3.8,<4"],\n                env,\n                work / "matplotlib_install.log",\n            )\n        write_progress(\n            destination / "study.json",\n            {"study_sha256": study_sha, "study": study, "environment": versions},\n        )\n        code = destination / "code"\n        code.mkdir(exist_ok=True)\n        for name in ("adaptation_strength.py", "adaptation_stage1.py", "camera_extended.py"):\n            (code / name).write_bytes((runtime / "src/mediflow_datasets" / name).read_bytes())\n        failures = {}\n        command = [str(python), "-u", "-m", "mediflow_datasets.adaptation_strength"]\n        for domain in colab.DOMAINS:\n            log = destination / f"{domain}_{uuid.uuid4().hex[:12]}.log"\n            try:\n                with tempfile.TemporaryDirectory(prefix=f"{domain}_", dir=work) as temp:\n                    local = Path(temp)\n                    archive = local / "source.zip"\n                    if not (destination / domain / "complete.json").exists():\n                        colab.copy_source(sources[domain], archive)\n                    colab.run_live(\n                        command\n                        + [\n                            "--domain",\n                            domain,\n                            "--runtime",\n                            str(runtime),\n                            "--stage1",\n                            str(root / "stage1"),\n                            "--stage2",\n                            str(root / "stage2"),\n                            "--study",\n                            str(study_path),\n                            "--archive",\n                            str(archive),\n                            "--output",\n                            str(destination / domain),\n                            "--local",\n                            str(local),\n                        ],\n                        env,\n                        log,\n                    )\n            except Exception as error:\n                failures[domain] = {"error": str(error), "log": log.name}\n                print(f"[{domain}] \xeb\xaf\xb8\xec\x99\x84\xeb\xa3\x8c \xc2\xb7 {error}", flush=True)\n        write_progress(\n            destination / "summary.json",\n            {\n                "completed_domains": [d for d in colab.DOMAINS if d not in failures],\n                "failures": failures,\n                "source_test_evaluated": False,\n            },\n        )\n        if failures:\n            raise RuntimeError(f"\xeb\xaf\xb8\xec\x99\x84\xeb\xa3\x8c \xeb\xb6\x84\xec\x95\xbc\xea\xb0\x80 \xec\x9e\x88\xec\x8a\xb5\xeb\x8b\x88\xeb\x8b\xa4. {destination}")\n        comparison(destination)\n        colab.run_live(command + ["--plots", str(destination)], env, work / "plots.log")\n    return destination\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--launch", nargs=5)\n    parser.add_argument("--plots", type=Path)\n    parser.add_argument("--domain", choices=colab.DOMAINS)\n    for name in ("runtime", "stage1", "stage2", "study", "archive", "output", "local"):\n        parser.add_argument("--" + name, type=Path)\n    args = parser.parse_args()\n    if args.launch:\n        launch(*(Path(arg) for arg in args.launch))\n    elif args.plots:\n        plots(args.plots)\n    else:\n        run_domain(\n            args.runtime,\n            args.stage1,\n            args.stage2,\n            args.study,\n            args.domain,\n            args.archive,\n            args.output,\n            args.local,\n        )\n\n\nif __name__ == "__main__":\n    main()\n', 'adaptation_stage1.py': b'"""Compare two eligible starting models per domain with the same lighting treatment."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport math\nimport os\nimport shutil\nimport time\nimport uuid\nimport zipfile\nfrom collections import Counter\nfrom concurrent.futures import ThreadPoolExecutor\nfrom dataclasses import asdict\nfrom pathlib import Path\n\nimport keras\nimport numpy as np\nimport tensorflow as tf\n\nfrom mediflow_datasets import camera_extended, domain_alignment\nfrom mediflow_datasets import domain_adaptation_data as data\nfrom mediflow_datasets import domain_adaptation_experiment as exp\nfrom mediflow_datasets import domain_adaptation_resolution as resolution\nfrom mediflow_datasets.candidate_reproduction import ROOT\nfrom mediflow_datasets.domain_adaptation_io import write_progress\nfrom mediflow_datasets.domain_adaptation_training import object_hash\n\nDOMAINS = ("skin", "web_skin", "hair")\nARMS = ("basic", "selected")\nWORKERS = min(4, os.cpu_count() or 1)\n\n\ndef show(label, step, total, extra=""):\n    print(f"{label} | {step}/{total} ({100 * step / total:.1f}%) {extra}", flush=True)\n\n\ndef prepare_source(domain, archive_path, task, reference, out):\n    classes = task["classes"]\n    print(f"[{domain}] \xeb\x8d\xb0\xec\x9d\xb4\xed\x84\xb0 ZIP \xed\x95\xb4\xec\x8b\x9c\xc2\xb7\xed\x81\xb4\xeb\x9e\x98\xec\x8a\xa4 \xed\x99\x95\xec\x9d\xb8", flush=True)\n    source, sha = data.source_index(archive_path, classes, task["source_sha256"])\n    retained = (\n        resolution.filtered_source(source, reference["quarantine"], classes, sha)\n        if reference.get("quarantine")\n        else {k: v for k, v in source.items() if k[0] == "original"}\n    )\n    # Skin\'s historical distinction is Original versus Augmented preparation. Use only\n    # linked descendants of retained originals. Adaptation itself uses identical originals.\n    train = retained["original", "train"]\n    originals = {r.path: r for r in train}\n    prepare_augmented = domain == "skin" and any(\n        spec.get("skin_augmented_preparation") for spec in task.get("models", {}).values()\n    )\n    descendants = [\n        r\n        for r in source["augmented", "train"]\n        if prepare_augmented\n        and r.origin in originals\n        and r.label == originals[r.origin].label\n        and r.path.rsplit("/", 1)[-1] != r.origin.rsplit("/", 1)[-1]\n    ]\n    # For the saved Hair model, include its previous augmented source in the leakage scan.\n    scanned = (\n        [r for rows in source.values() for r in rows]\n        if domain == "hair"\n        else [r for rows in retained.values() for r in rows] + descendants\n    )\n    label = f"[{domain}] \xec\xa4\x91\xeb\xb3\xb5\xc2\xb7\xeb\xb6\x84\xed\x95\xa0 \xed\x99\x95\xec\x9d\xb8 (\xec\x82\xac\xec\xa7\x84)"\n    entries = []\n    with zipfile.ZipFile(archive_path) as archive, ThreadPoolExecutor(WORKERS) as pool:\n\n        def signature(row):\n            return row, *data.image_signatures(archive.read(row.path))\n\n        for start in range(0, len(scanned), 128):\n            entries.extend(pool.map(signature, scanned[start : start + 128]))\n            done = len(entries)\n            if start == 0 or done % 1024 == 0 or done == len(scanned):\n                show(label, done, len(scanned))\n    report = data.audit_groups(entries)\n    dropped = set()\n    if descendants:\n        involved = {p for pair in report["near_duplicate_review"] for p in pair}\n        involved.update(\n            p for leak in report["blocking_leaks_or_conflicts"] for p in leak.get("paths", ())\n        )\n        dropped = {r.path for r in descendants if r.path in involved}\n        entries = [entry for entry in entries if entry[0].path not in dropped]\n        report = data.audit_groups(entries)\n    report["excluded_augmented_descendants"] = sorted(dropped)\n    report["source_target_pixel_overlap"] = len(\n        {entry[2] for entry in entries} & set(reference["pixel_hashes"])\n    )\n    report["source_sha256"] = sha\n    report["retained_counts"] = {split: len(retained["original", split]) for split in data.SPLITS}\n    if domain == "hair":\n        consistency = {}\n        for split in data.SPLITS:\n            original, augmented = (\n                Counter(\n                    (row.label, pixel)\n                    for row, _, pixel, _ in entries\n                    if row.split == split and row.variant == variant\n                )\n                for variant in ("original", "augmented")\n            )\n            consistency[split] = (\n                not (original - augmented) if split == "train" else original == augmented\n            )\n        report["original_augmented_consistency"] = consistency\n    write_progress(out / "data_audit.json", report)\n    if (\n        report["blocking_leaks_or_conflicts"]\n        or report["near_duplicate_review"]\n        or report["source_target_pixel_overlap"]\n        or not all(report.get("original_augmented_consistency", {}).values())\n    ):\n        raise ValueError(f"{domain} \xeb\xb6\x84\xed\x95\xa0 \xec\xa0\x90\xea\xb2\x80 \xec\x8b\xa4\xed\x8c\xa8. data_audit.json\xec\x9d\x84 \xed\x99\x95\xec\x9d\xb8\xed\x95\x98\xec\x84\xb8\xec\x9a\x94.")\n    pixel_by_path = {row.path: pixel for row, _, pixel, _ in entries}\n    paired = {}\n    for row in descendants:\n        if row.path not in dropped and pixel_by_path[row.path] != pixel_by_path[row.origin]:\n            paired.setdefault(row.origin, row)\n    if prepare_augmented and {r.label for r in paired.values()} != set(range(len(classes))):\n        raise ValueError("\xed\x94\xbc\xeb\xb6\x80 \xec\x84\xa0\xec\xa0\x95 \xeb\xb0\xa9\xec\x8b\x9d \xec\x9e\xac\xed\x98\x84\xec\x97\x90 \xed\x95\x84\xec\x9a\x94\xed\x95\x9c \xed\x81\xb4\xeb\x9e\x98\xec\x8a\xa4\xeb\xb3\x84 \xed\x8c\x8c\xec\x83\x9d\xeb\xb3\xb8\xec\x9d\xb4 \xed\x99\x95\xec\x9d\xb8\xeb\x90\x98\xec\xa7\x80 \xec\x95\x8a\xec\x95\x98\xec\x8a\xb5\xeb\x8b\x88\xeb\x8b\xa4.")\n    write_progress(\n        out / "source_manifest.json",\n        {\n            "original": {\n                split: [asdict(r) for r in retained["original", split]] for split in data.SPLITS\n            },\n            "skin_preparation_pairs": {parent: asdict(row) for parent, row in paired.items()},\n        },\n    )\n    return train, retained["original", "val"], paired\n\n\ndef image_cache(folder, archive_path, train, val, pairs, task, size, profile, *, expanded=None):\n    """Cache exact float32 model inputs on local SSD once for each input resolution."""\n    rows = [*train, *val, *pairs.values()]\n    conditions = ("lighting", "expanded") if expanded is not None else ("lighting",)\n    size_bytes = (len(rows) + len(conditions) * len(val)) * size * size * 3 * 4\n    if shutil.disk_usage(folder).free < size_bytes + 1024**3:\n        raise RuntimeError(\n            f"{task[\'domain\']} \xec\x9e\x85\xeb\xa0\xa5 \xec\xba\x90\xec\x8b\x9c \xea\xb3\xb5\xea\xb0\x84 \xeb\xb6\x80\xec\xa1\xb1 (\xed\x95\x84\xec\x9a\x94 {size_bytes / 1e9:.1f} GB)."\n        )\n    shape = (size, size, 3)\n    cache = np.lib.format.open_memmap(\n        folder / f"inputs_{size}.npy", mode="w+", dtype=np.float32, shape=(len(rows), *shape)\n    )\n    shifted = {\n        name: np.lib.format.open_memmap(\n            folder / f"val_{name}_{size}.npy", mode="w+", dtype=np.float32, shape=(len(val), *shape)\n        )\n        for name in conditions\n    }\n    with zipfile.ZipFile(archive_path) as archive, ThreadPoolExecutor(WORKERS) as pool:\n\n        def prepare(item):\n            i, row = item\n            with tf.device("/CPU:0"):\n                image = exp.image_tensor(archive.read(row.path), size, task["input_policy"])\n            cache[i] = image\n            if row.split == "val":\n                seed = exp.item_seed(row, 1707)\n                shifted["lighting"][i - len(train)] = exp.transform_image(\n                    image, profile, "target_guided", seed\n                )\n                if expanded is not None:\n                    shifted["expanded"][i - len(train)] = camera_extended.transform(\n                        image, expanded, "expanded", seed\n                    )\n\n        for start in range(0, len(rows), 128):\n            list(pool.map(prepare, enumerate(rows[start : start + 128], start)))\n            end = min(start + 128, len(rows))\n            if start == 0 or end % 512 == 0 or end == len(rows):\n                show(f"[{task[\'domain\']} \xc2\xb7 {size}px] \xec\x9e\x85\xeb\xa0\xa5 \xec\xba\x90\xec\x8b\x9c \xec\xa4\x80\xeb\xb9\x84 (\xec\x82\xac\xec\xa7\x84)", end, len(rows))\n    cache.flush()\n    for images in shifted.values():\n        images.flush()\n    lookup = {row.path: i for i, row in enumerate(rows)}\n    paired_indexes = [\n        lookup[pairs[r.path].path] if r.path in pairs else i for i, r in enumerate(train)\n    ]\n    return cache, shifted if expanded is not None else shifted["lighting"], paired_indexes\n\n\ndef pipeline(images, indexes, labels, batch, size, transform=None):\n    def read(index):\n        image = images[int(index)]\n        return image if transform is None else transform(int(index), image)\n\n    def get(index, label):\n        image = tf.numpy_function(read, [index], tf.float32)\n        return tf.ensure_shape(image, (size, size, 3)), label\n\n    ds = tf.data.Dataset.from_tensor_slices(\n        (np.asarray(indexes, np.int32), np.asarray(labels, np.int32))\n    )\n    options = tf.data.Options()\n    options.threading.private_threadpool_size = WORKERS\n    options.threading.max_intra_op_parallelism = 1\n    return ds.with_options(options).map(get, num_parallel_calls=WORKERS).batch(batch).prefetch(2)\n\n\ndef model_for(spec, count, weights):\n    if spec["architecture"] == "pmg":\n        from mediflow_datasets.web_skin_paper_suite import build_pmg_model\n\n        return build_pmg_model(count, spec["size"], weights=weights)\n    size = spec["size"]\n    backbone = keras.applications.EfficientNetB0(\n        include_top=False, weights=weights, input_shape=(size, size, 3)\n    )\n    backbone.trainable = False\n    inputs = keras.Input((size, size, 3))\n    features = keras.layers.GlobalAveragePooling2D()(backbone(inputs, training=False))\n    outputs = keras.layers.Dense(count, activation="softmax")(keras.layers.Dropout(0.3)(features))\n    return keras.Model(inputs, outputs)\n\n\ndef predictor(model, spec):\n    pmg = spec["architecture"] == "pmg"\n    size = spec["size"]\n\n    @tf.function(input_signature=[tf.TensorSpec([None, size, size, 3], tf.float32)])\n    def predict(images):\n        outputs = model(images, training=False)\n        scores = tf.nn.softmax(tf.add_n(outputs), axis=-1) if pmg else outputs\n        return tf.debugging.check_numerics(scores, "validation scores")\n\n    return predict\n\n\ndef steps(model, spec, optimizer, count, *, alignment=None):\n    pmg = spec["architecture"] == "pmg"\n    loss_fn = keras.losses.CategoricalCrossentropy(\n        from_logits=pmg, label_smoothing=spec["label_smoothing"]\n    )\n    variables = model.trainable_variables\n    optimizer.build(variables)\n    if pmg:\n        from mediflow_datasets.web_skin_paper_suite import _jigsaw\n    if alignment is not None:\n        adapter = "sum_logits_softmax" if pmg else "single_softmax"\n        extractor = exp.feature_model(model, adapter)\n        features = keras.Model(model.input, extractor.outputs[-1])\n\n    def update(images, labels, branch=None, factor=1.0, target=None):\n        with tf.GradientTape() as tape:\n            if target is None:\n                outputs = model(images, training=True)\n                logits = outputs[branch] if pmg else outputs\n                mmd = coral = tf.constant(0.0)\n            else:\n                outputs = extractor(images, training=True)\n                logits = outputs[branch if pmg else 0]\n                mmd, coral = domain_alignment.losses(\n                    outputs[-1], features(target, training=False), alignment["reference"]\n                )\n            ce = factor * loss_fn(labels, logits)\n            loss = ce\n            if target is not None:\n                loss = (\n                    loss + alignment["weights"]["mmd"] * mmd + alignment["weights"]["coral"] * coral\n                )\n            loss = tf.debugging.check_numerics(loss, "training loss")\n        grads = tape.gradient(loss, variables)\n        pairs = [\n            (grad, var) for grad, var in zip(grads, variables, strict=True) if grad is not None\n        ]\n        clipped, norm = tf.clip_by_global_norm([g for g, _ in pairs], 1.0)\n        with tf.control_dependencies([tf.debugging.check_numerics(norm, "gradient norm")]):\n            optimizer.apply_gradients(zip(clipped, [v for _, v in pairs], strict=True))\n        return loss, ce, mmd, coral\n\n    signature = [\n        tf.TensorSpec([None, spec["size"], spec["size"], 3], tf.float32),\n        tf.TensorSpec([None], tf.int32),\n    ]\n\n    def update_batch(images, indices, target=None):\n        labels = tf.one_hot(indices, count)\n        if pmg:\n            branch_losses = [\n                update(_jigsaw(images, grid), labels, i)[0] for i, grid in enumerate((8, 4, 2))\n            ]\n            loss, ce, mmd, coral = update(images, labels, 3, 2.0, target)\n            total = tf.add_n(branch_losses)\n            return (total + loss) / 4, (total + ce) / 4, mmd, coral\n        return update(images, labels, target=target)\n\n    @tf.function(input_signature=signature)\n    def train_step(images, indices):\n        return update_batch(images, indices)[0]\n\n    if alignment is not None:\n        aligned_step = tf.function(update_batch, input_signature=[*signature, signature[0]])\n        return train_step, predictor(model, spec), aligned_step\n\n    return train_step, predictor(model, spec)\n\n\ndef epoch_records(folder, epochs):\n    records = sorted(folder.glob("epoch_*.json"))\n    history = [data.read_json(p) for p in records]\n    if len(history) > epochs or [r["epoch"] for r in history] != list(range(1, len(history) + 1)):\n        raise ValueError(f"Incomplete or inconsistent epoch records: {folder}")\n    return history\n\n\ndef save_epoch(checkpoint, folder, local, entry):\n    attempt = "snapshot_" + uuid.uuid4().hex\n    destination = folder / attempt\n    destination.mkdir()\n    local.mkdir(exist_ok=True)\n    local_prefix = checkpoint.write(str(local / "state"))\n    record = {**entry, "snapshot": attempt, "files": {}}\n    for path in sorted(Path(local_prefix).parent.glob("state.*")):\n        shutil.copyfile(path, destination / path.name)\n        record["files"][path.name] = data.sha256(path)\n    if set(record["files"]) != {"state.index", "state.data-00000-of-00001"}:\n        raise RuntimeError("Unexpected checkpoint shard layout")\n    write_progress(folder / f"epoch_{entry[\'epoch\']:03}.json", record)\n    return record\n\n\ndef restore_epoch(checkpoint, folder, record):\n    for name, sha in record["files"].items():\n        if data.sha256(data.local_path(folder, f"{record[\'snapshot\']}/{name}")) != sha:\n            raise ValueError("Epoch checkpoint failed integrity verification")\n    checkpoint.read(str(folder / record["snapshot"] / "state")).assert_consumed()\n\n\ndef publish_model(source, directory, name):\n    temporary = directory / (uuid.uuid4().hex + ".keras.tmp")\n    shutil.copyfile(source, temporary)\n    destination = directory / f"{name}.keras"\n    temporary.replace(destination)\n    return {"file": destination.name, "sha256": data.sha256(source)}\n\n\ndef export_model(model, directory, name, local):\n    local.mkdir(exist_ok=True)\n    scratch = local / f"{name}.keras"\n    model.save(scratch)\n    return publish_model(scratch, directory, name)\n\n\ndef validation_pipelines(cache, shifted, n, val, batch, size):\n    labels = [row.label for row in val]\n    return {\n        "clean": pipeline(cache, range(n, n + len(val)), labels, batch, size),\n        **{\n            name: pipeline(images, range(len(val)), labels, batch, size)\n            for name, images in shifted.items()\n        },\n    }\n\n\ndef validate(predict, datasets, rows, class_count, label):\n    metrics = {}\n    for name, ds in datasets.items():\n        print(f"{label} Validation {name} \xc2\xb7 {len(rows):,}\xec\x9e\xa5", flush=True)\n        scores = np.concatenate([predict(images).numpy() for images, _ in ds])\n        metrics[name] = exp.classification(rows, scores, class_count)\n    return metrics\n\n\ndef phase(model, context, folder, local, spec, phase_name, epochs, rate, *, alignment=None):\n    config, task, train, val, profile, cache, shifted, pairs = context\n    folder.mkdir(exist_ok=True)\n    history = epoch_records(folder, epochs)\n    optimizer = keras.optimizers.Adam(rate)\n    step, predict, *extra_steps = steps(\n        model, spec, optimizer, len(task["classes"]), alignment=alignment\n    )\n    if (phase_name == "aligned") != (alignment is not None):\n        raise ValueError("Alignment inputs must be present exactly for the aligned phase")\n    checkpoint = tf.train.Checkpoint(model=model, optimizer=optimizer)\n    tag = f"[{task[\'domain\']} \xc2\xb7 {spec[\'role\']} \xc2\xb7 {phase_name}]"\n    if history:\n        restore_epoch(checkpoint, folder, history[-1])\n        print(f"{tag} Drive\xec\x97\x90\xec\x84\x9c {len(history)}/{epochs} epoch \xeb\xb3\xb5\xec\x9b\x90", flush=True)\n    n, batch, size = len(train), task["batch_size"], spec["size"]\n    expanded = phase_name in ("expanded", "aligned")\n    condition = "expanded" if expanded else "lighting"\n    datasets = validation_pipelines(\n        cache, shifted if expanded else {"lighting": shifted}, n, val, batch, size\n    )\n    for epoch in range(len(history), epochs):\n        started = time.monotonic()\n        keras.utils.set_random_seed(config["seed"] + epoch)\n        order = np.random.default_rng(config["seed"] + epoch).permutation(n)\n        indexes = order\n        if phase_name.startswith("prepare") and spec.get("skin_augmented_preparation"):\n            indexes = [pairs[i] if (i + epoch) % 2 == 0 else i for i in order]\n        transform = None\n        if expanded or phase_name == "lighting":\n            seeds = [exp.item_seed(row, config["seed"], epoch) for row in train]\n            apply = [exp.augmentation_applies(s, config["augmentation_probability"]) for s in seeds]\n\n            def transform(i, image, seeds=seeds, apply=apply):\n                if not apply[i]:\n                    return image\n                if expanded:\n                    return camera_extended.transform(image, profile, "expanded", seeds[i])\n                return exp.transform_image(image, profile, "target_guided", seeds[i])\n\n        label = f"{tag} Epoch {epoch + 1}/{epochs}"\n        batches = math.ceil(n / batch)\n        print(f"{label} \xec\x8b\x9c\xec\x9e\x91 \xc2\xb7 \xed\x95\x99\xec\x8a\xb5 {n:,}\xec\x9e\xa5 \xc2\xb7 {batches} \xeb\xb0\xb0\xec\xb9\x98", flush=True)\n        train_data = pipeline(\n            cache, indexes, [train[i].label for i in order], batch, size, transform\n        )\n        losses = []\n        aligned_metrics = []\n        schedule = {}\n        if alignment is not None:\n            eligible = range(batches - int(n % batch == 1))\n            schedule = domain_alignment.balanced_schedule(\n                alignment["rows"], eligible, config["seed"], epoch\n            )\n            if not schedule:\n                raise ValueError("No Source batch is eligible for feature alignment")\n        for i, (images, labels) in enumerate(train_data, 1):\n            if i - 1 in schedule:\n                target = alignment["images"][schedule[i - 1]]\n                loss, ce, mmd, coral = map(float, extra_steps[0](images, labels, target))\n                losses.append(loss)\n                aligned_metrics.append((ce, mmd, coral))\n            else:\n                losses.append(float(step(images, labels)))\n            if i == 1 or i == batches or i % max(5, batches // 15) == 0:\n                elapsed = time.monotonic() - started\n                remaining = elapsed * (batches / i - 1)\n                show(\n                    label + " \xed\x95\x99\xec\x8a\xb5",\n                    i,\n                    batches,\n                    f"loss={np.mean(losses):.5f} \xc2\xb7 \xeb\x82\xa8\xec\x9d\x80 \xed\x95\x99\xec\x8a\xb5 \xec\x95\xbd {remaining:.0f}\xec\xb4\x88",\n                )\n        train_seconds = time.monotonic() - started\n        metrics = validate(predict, datasets, val, len(task["classes"]), label)\n        validation_seconds = time.monotonic() - started - train_seconds\n        entry = {\n            "epoch": epoch + 1,\n            "train_loss_mean_per_update": float(np.mean(losses)),\n            "training_batches": batches,\n            "optimizer_updates": batches * (4 if spec["architecture"] == "pmg" else 1),\n            "train_seconds": train_seconds,\n            "validation_seconds": validation_seconds,\n            **metrics,\n        }\n        if aligned_metrics:\n            ce, mmd, coral = np.asarray(aligned_metrics).mean(0).tolist()\n            entry["alignment"] = {\n                "updates": len(aligned_metrics),\n                "target_images_used": sum(map(len, schedule.values())),\n                "classification_loss_on_aligned_batches": ce,\n                "mmd_mean": mmd,\n                "coral_mean": coral,\n            }\n            print(\n                f"{label} \xec\xa0\x95\xeb\xa0\xac {len(aligned_metrics)}\xeb\xb0\xb0\xec\xb9\x98 \xc2\xb7 MMD={mmd:.6g} \xc2\xb7 CORAL={coral:.6g}",\n                flush=True,\n            )\n        print(f"{label} Drive \xec\xb2\xb4\xed\x81\xac\xed\x8f\xac\xec\x9d\xb8\xed\x8a\xb8 \xec\xa0\x80\xec\x9e\xa5", flush=True)\n        history.append(save_epoch(checkpoint, folder, local, entry))\n        print(\n            f"{label} \xec\x99\x84\xeb\xa3\x8c \xc2\xb7 loss={entry[\'train_loss_mean_per_update\']:.5f} \xc2\xb7 "\n            f"\xec\x9b\x90\xeb\xb3\xb8 Val F1={metrics[\'clean\'][\'macro_f1\']:.5f} \xc2\xb7 "\n            f"\xec\xa1\xb0\xeb\xaa\x85 Val F1={metrics[\'lighting\'][\'macro_f1\']:.5f}"\n            + (f" \xc2\xb7 \xed\x99\x95\xec\x9e\xa5 Val F1={metrics[\'expanded\'][\'macro_f1\']:.5f}" if expanded else ""),\n            flush=True,\n        )\n    best = max(history, key=lambda r: score(r, expanded or phase_name == "lighting", condition))\n    restore_epoch(checkpoint, folder, best)\n    write_progress(folder / "history.json", history)\n    return best\n\n\ndef score(record, lighting=True, condition="lighting"):\n    clean, shifted = record["clean"]["macro_f1"], record[condition]["macro_f1"]\n    return (min(clean, shifted), clean) if lighting else (clean,)\n\n\ndef trial(context, output, local, spec, job):\n    config, task = context[:2]\n    output.mkdir(exist_ok=True)\n    completed = output / "complete.json"\n    if completed.exists():\n        saved = data.read_json(completed)\n        verify_models(output, saved)\n        print(f"[{job}/6 \xc2\xb7 {task[\'domain\']} \xc2\xb7 {spec[\'role\']}] \xec\x99\x84\xeb\xa3\x8c \xea\xb2\xb0\xea\xb3\xbc \xec\x9e\xac\xec\x82\xac\xec\x9a\xa9", flush=True)\n        return saved\n    print(f"\\n[{job}/6 \xc2\xb7 {task[\'domain\']} \xc2\xb7 {spec[\'role\']}] \xec\x8b\xa4\xed\x97\x98 \xec\x8b\x9c\xec\x9e\x91", flush=True)\n    keras.backend.clear_session()\n    keras.utils.set_random_seed(config["seed"])\n    count = len(task["classes"])\n    initial_path = output / "initial.json"\n    if initial_path.exists():\n        initial = data.read_json(initial_path)\n        path = data.local_path(output, initial["file"])\n        if data.sha256(path) != initial["sha256"]:\n            raise ValueError("Starting checkpoint changed")\n        model = keras.models.load_model(path, compile=False)\n    else:\n        if spec.get("model"):\n            path = data.local_path(ROOT, spec["model"])\n            if data.sha256(path) != spec["model_sha256"]:\n                raise ValueError("Selected checkpoint changed")\n            model = keras.models.load_model(path, compile=False)\n            initial = publish_model(path, output, "initial")\n            print(f"[{task[\'domain\']} \xc2\xb7 {spec[\'role\']}] \xec\xa0\x80\xec\x9e\xa5 \xea\xb0\x80\xec\xa4\x91\xec\xb9\x98\xeb\xa1\x9c \xec\xa1\xb0\xeb\xaa\x85 \xec\xa0\x81\xec\x9d\x91 \xec\x8b\x9c\xec\x9e\x91", flush=True)\n        else:\n            weights = None if (output / "prepare_head/epoch_001.json").exists() else "imagenet"\n            print(f"[{task[\'domain\']} \xc2\xb7 {spec[\'role\']}] \xed\x98\x84\xec\x9e\xac Train\xec\x9c\xbc\xeb\xa1\x9c \xec\xb6\x9c\xeb\xb0\x9c \xeb\xaa\xa8\xeb\x8d\xb8 \xec\xa4\x80\xeb\xb9\x84", flush=True)\n            model = model_for(spec, count, weights)\n            phase(\n                model,\n                context,\n                output / "prepare_head",\n                local,\n                spec,\n                "prepare_head",\n                config["prepare_head_epochs"],\n                config["prepare_head_lr"],\n            )\n            exp.configure_partial(model)\n            phase(\n                model,\n                context,\n                output / "prepare_partial",\n                local,\n                spec,\n                "prepare_partial",\n                config["prepare_partial_epochs"],\n                config["prepare_partial_lr"],\n            )\n            initial = export_model(model, output, "initial", local)\n        write_progress(initial_path, initial)\n    if tuple(model.input_shape[1:]) != (spec["size"], spec["size"], 3):\n        raise ValueError("Initial input shape differs from configuration")\n    if len(model.outputs) != (4 if spec["architecture"] == "pmg" else 1) or any(\n        layer.shape[-1] != count for layer in model.outputs\n    ):\n        raise ValueError("Initial output contract differs from class mapping")\n    exp.configure_partial(model)\n    best = phase(\n        model,\n        context,\n        output / "lighting",\n        local,\n        spec,\n        "lighting",\n        config["epochs"],\n        config["learning_rate"],\n    )\n    saved = {\n        "initial": initial,\n        "best_model": export_model(model, output, "best", local),\n        "model_config": spec,\n        "validation": best,\n        "source_test_evaluated": False,\n        "validation_scope": config.get("validation_scope", "current_source_validation"),\n    }\n    write_progress(completed, saved)\n    return saved\n\n\ndef verify_models(folder, result):\n    for key in ("initial", "best_model"):\n        item = result[key]\n        if data.sha256(data.local_path(folder, item["file"])) != item["sha256"]:\n            raise ValueError(f"Completed checkpoint changed: {folder}/{item[\'file\']}")\n\n\ndef run_domain(domain, source_zip, output, local):\n    config = data.read_json(ROOT / "configs/adaptation_stage1.json")\n    task = config["tasks"][domain]\n    if config.get("initialization_policy") == "saved_only" and any(\n        not spec.get("model") or not spec.get("model_sha256") for spec in task["models"].values()\n    ):\n        raise ValueError("A starting checkpoint is missing from the saved-only bundle")\n    if config.get("validation_scope"):\n        print(\n            f"[{domain}] \xea\xb3\xbc\xea\xb1\xb0 \xed\x95\x99\xec\x8a\xb5 \xec\x9d\xb4\xeb\xa0\xa5\xec\x9d\x98 \xea\xb2\xb9\xec\xb9\xa8 \xec\x97\xac\xeb\xb6\x80 \xeb\xaf\xb8\xed\x99\x95\xec\x9d\xb8 \xc2\xb7 Validation\xec\x9d\x80 \xed\x83\x90\xec\x83\x89 \xeb\xb9\x84\xea\xb5\x90\xec\x9a\xa9", flush=True\n        )\n    reference = data.read_json(ROOT / f"references/{domain}.json")\n    profile = reference["extended_profile"]["lighting"]\n    output.mkdir(parents=True, exist_ok=True)\n    identity = {\n        "config_sha256": object_hash(config),\n        "reference_sha256": object_hash(reference),\n        "code_sha256": data.sha256(Path(__file__)),\n    }\n    marker = output / "identity.json"\n    if marker.exists() and data.read_json(marker) != identity:\n        raise ValueError("Different inputs cannot resume this output")\n    write_progress(marker, identity)\n    write_progress(\n        output / "run_configuration.json",\n        {\n            "identity": identity,\n            "configuration": config,\n            "profile": profile,\n            "tensorflow": tf.__version__,\n            "keras": keras.__version__,\n            "numpy": np.__version__,\n            "gpu": [\n                tf.config.experimental.get_device_details(device)\n                for device in tf.config.list_physical_devices("GPU")\n            ],\n            "code_provenance": data.read_json(ROOT / "bundle_provenance.json"),\n        },\n    )\n    if all((output / role / "complete.json").is_file() for role in ARMS):\n        results = {role: data.read_json(output / role / "complete.json") for role in ARMS}\n        for role, result in results.items():\n            verify_models(output / role, result)\n        print(f"[{domain}] \xeb\x91\x90 \xed\x95\x99\xec\x8a\xb5 \xeb\xaa\xa8\xeb\x91\x90 \xec\x99\x84\xeb\xa3\x8c\xeb\x90\xa8", flush=True)\n        return results\n    train, val, pairs = prepare_source(domain, source_zip, task, reference, output)\n    caches = {}\n    results = {}\n    for i, role in enumerate(ARMS, 1):\n        existing = output / role / "complete.json"\n        if existing.is_file():\n            results[role] = data.read_json(existing)\n            verify_models(output / role, results[role])\n            print(\n                f"[{DOMAINS.index(domain) * 2 + i}/6 \xc2\xb7 {domain} \xc2\xb7 {role}] \xec\x99\x84\xeb\xa3\x8c \xea\xb2\xb0\xea\xb3\xbc \xec\x9e\xac\xec\x82\xac\xec\x9a\xa9",\n                flush=True,\n            )\n            continue\n        spec = {**task["models"][role], "role": role}\n        size = spec["size"]\n        if size not in caches:\n            caches[size] = image_cache(local, source_zip, train, val, pairs, task, size, profile)\n        context = (config, task, train, val, profile, *caches[size])\n        results[role] = trial(\n            context, output / role, local / "saving", spec, DOMAINS.index(domain) * 2 + i\n        )\n    return results\n\n\ndef write_selection(root):\n    selected, table = {}, []\n    for domain in DOMAINS:\n        records = {role: data.read_json(root / domain / role / "complete.json") for role in ARMS}\n        for role, record in records.items():\n            verify_models(root / domain / role, record)\n            table.append(\n                {\n                    "domain": domain,\n                    "role": role,\n                    "best_epoch": record["validation"]["epoch"],\n                    "clean_macro_f1": record["validation"]["clean"]["macro_f1"],\n                    "lighting_macro_f1": record["validation"]["lighting"]["macro_f1"],\n                    "clean_accuracy": record["validation"]["clean"]["accuracy"],\n                    "lighting_accuracy": record["validation"]["lighting"]["accuracy"],\n                    "validation_scope": record.get("validation_scope", "current_source_validation"),\n                }\n            )\n        role = max(ARMS, key=lambda name: score(records[name]["validation"]))\n        record = records[role]\n        selected[domain] = {\n            "role": role,\n            "model_config": record["model_config"],\n            "initial_checkpoint": {\n                "path": f"{domain}/{role}/initial.keras",\n                "sha256": record["initial"]["sha256"],\n            },\n            "stage1_checkpoint": {\n                "path": f"{domain}/{role}/best.keras",\n                "sha256": record["best_model"]["sha256"],\n            },\n        }\n    selection = {\n        "schema": "mediflow_adaptation_stage1_selection_v1",\n        "selected": selected,\n        "criteria": "Maximize worst Val Macro-F1. Break ties by clean Val, then basic.",\n        "source_test_evaluated": False,\n        "target_accuracy": None,\n    }\n    if any(row["validation_scope"].startswith("exploratory_") for row in table):\n        selection["selection_scope"] = "provisional_followup_candidate"\n        selection["historical_train_evaluation_independence_verified"] = False\n    path = root / "selection.json"\n    if path.exists() and data.read_json(path) != selection:\n        raise ValueError("Existing Validation selection differs")\n    with (root / "validation_comparison.csv").open("w", encoding="utf-8-sig", newline="") as f:\n        writer = csv.DictWriter(f, fieldnames=table[0].keys())\n        writer.writeheader()\n        writer.writerows(table)\n    write_progress(path, selection)\n    for row in table:\n        print(row, flush=True)\n    print("1\xec\xb0\xa8 \xec\x99\x84\xeb\xa3\x8c \xc2\xb7 \xeb\xb6\x84\xec\x95\xbc\xeb\xb3\x84 \xec\x84\xa0\xed\x83\x9d:", {d: s["role"] for d, s in selected.items()}, flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--select", type=Path)\n    parser.add_argument("--domain", choices=DOMAINS)\n    parser.add_argument("--source-zip", type=Path)\n    parser.add_argument("--output", type=Path)\n    parser.add_argument("--local", type=Path)\n    args = parser.parse_args()\n    if args.select:\n        write_selection(args.select)\n    elif all((args.domain, args.source_zip, args.output, args.local)):\n        if not tf.config.list_physical_devices("GPU"):\n            raise RuntimeError("TensorFlow GPU\xeb\xa5\xbc \xec\x82\xac\xec\x9a\xa9\xed\x95\xa0 \xec\x88\x98 \xec\x97\x86\xec\x8a\xb5\xeb\x8b\x88\xeb\x8b\xa4.")\n        run_domain(args.domain, args.source_zip, args.output, args.local)\n    else:\n        parser.error("Provide --select or all domain run parameters")\n\n\nif __name__ == "__main__":\n    main()\n', 'adaptation_stage1_colab.py': b'"""Drive-based stage 1 launcher with live output and scoped interruption."""\n\nfrom __future__ import annotations\n\nimport json\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport uuid\nfrom pathlib import Path\n\nfrom mediflow_datasets import domain_adaptation_batch as batch\nfrom mediflow_datasets.domain_adaptation_io import write_progress\n\nDOMAINS = ("skin", "web_skin", "hair")\n\n\ndef run_live(command, env, log_path, *, input_text=None):\n    with log_path.open("x", encoding="utf-8") as log:\n        process = subprocess.Popen(\n            command,\n            env=env,\n            stdin=subprocess.PIPE if input_text is not None else None,\n            stdout=subprocess.PIPE,\n            stderr=subprocess.STDOUT,\n            encoding="utf-8",\n            errors="replace",\n            bufsize=1,\n        )\n        try:\n            if input_text is not None:\n                with process.stdin:\n                    process.stdin.write(input_text)\n            for line in process.stdout:\n                log.write(line)\n                log.flush()\n                print(line, end="", flush=True)\n            if process.wait():\n                raise subprocess.CalledProcessError(process.returncode, command)\n        finally:\n            process.stdout.close()\n            if process.poll() is None:\n                process.terminate()\n                try:\n                    process.wait(timeout=10)\n                except subprocess.TimeoutExpired:\n                    process.kill()\n                    process.wait()\n\n\ndef copy_source(source, destination):\n    total = source.stat().st_size\n    if shutil.disk_usage(destination.parent).free < total + 1024**3:\n        raise RuntimeError("Colab \xec\x9e\x84\xec\x8b\x9c \xeb\x94\x94\xec\x8a\xa4\xed\x81\xac\xec\x97\x90 \xeb\x8d\xb0\xec\x9d\xb4\xed\x84\xb0 ZIP\xec\x9d\x84 \xeb\xb3\xb5\xec\x82\xac\xed\x95\xa0 \xea\xb3\xb5\xea\xb0\x84\xec\x9d\xb4 \xeb\xb6\x80\xec\xa1\xb1\xed\x95\xa9\xeb\x8b\x88\xeb\x8b\xa4.")\n    with source.open("rb") as reader, destination.open("xb") as writer:\n        done, last = 0, -1\n        while block := reader.read(16 * 1024**2):\n            writer.write(block)\n            done += len(block)\n            progress = min(100, 100 * done // total)\n            if progress >= last + 5 or done == total:\n                print(f"  Drive \xe2\x86\x92 Colab \xeb\xa1\x9c\xec\xbb\xac \xeb\xb3\xb5\xec\x82\xac {progress}% ({done / 1e9:.2f} GB)", flush=True)\n                last = progress\n    if done != total:\n        raise OSError("Drive ZIP \xeb\xb3\xb5\xec\x82\xac\xea\xb0\x80 \xec\xa4\x91\xeb\x8b\xa8\xeb\x90\x90\xec\x8a\xb5\xeb\x8b\x88\xeb\x8b\xa4.")\n\n\ndef run(runtime, work, project, bundle_sha, my_drive):\n    config = json.loads((runtime / "configs/adaptation_stage1.json").read_text(encoding="utf-8"))\n    python, versions = batch.prepare_python(work)\n    if not versions["gpu"]:\n        raise RuntimeError("Colab \xeb\x9f\xb0\xed\x83\x80\xec\x9e\x84 \xec\x9c\xa0\xed\x98\x95\xec\x97\x90\xec\x84\x9c GPU\xeb\xa5\xbc \xec\x84\xa0\xed\x83\x9d\xed\x95\x9c \xeb\x92\xa4 \xeb\x8b\xa4\xec\x8b\x9c \xec\x8b\xa4\xed\x96\x89\xed\x95\x98\xec\x84\xb8\xec\x9a\x94.")\n    env = batch.python_environment(python, runtime)\n    env.update(PYTHONIOENCODING="utf-8", PYTHONDONTWRITEBYTECODE="1")\n    destination = my_drive / "MediFlow_adaptation_runs" / bundle_sha[:16] / "stage1"\n    destination.mkdir(parents=True, exist_ok=True)\n    marker = destination / "bundle.json"\n    if marker.is_file() and json.loads(marker.read_text(encoding="utf-8"))["sha256"] != bundle_sha:\n        raise RuntimeError("\xec\x9d\xb4 \xea\xb2\xb0\xea\xb3\xbc \xed\x8f\xb4\xeb\x8d\x94\xeb\x8a\x94 \xeb\x8b\xa4\xeb\xa5\xb8 \xeb\xb2\x84\xec\xa0\x84\xec\x9d\x98 \xea\xb3\xb5\xed\x86\xb5 ZIP\xec\x97\x90 \xec\x97\xb0\xea\xb2\xb0\xeb\x8f\xbc \xec\x9e\x88\xec\x8a\xb5\xeb\x8b\x88\xeb\x8b\xa4.")\n    write_progress(marker, {"sha256": bundle_sha, "stage": 1, "environment": versions})\n    print("1\xec\xb0\xa8 \xea\xb2\xb0\xea\xb3\xbc \xec\xa0\x80\xec\x9e\xa5:", destination, flush=True)\n    failures = {}\n    for domain in DOMAINS:\n        output = destination / domain\n        log = destination / f"{domain}_{uuid.uuid4().hex[:10]}.log"\n        command = [str(python), "-m", "mediflow_datasets.adaptation_stage1"]\n        try:\n            with tempfile.TemporaryDirectory(prefix=f"{domain}_", dir=work) as folder:\n                local = Path(folder)\n                data_zip = local / "source.zip"\n                if not all(\n                    (output / arm / "complete.json").is_file() for arm in ("basic", "selected")\n                ):\n                    print(f"\\n[{domain}] Drive \xeb\x8d\xb0\xec\x9d\xb4\xed\x84\xb0\xec\x85\x8b \xec\xa4\x80\xeb\xb9\x84", flush=True)\n                    source = batch.find_source(project / "datasets", config["tasks"][domain])\n                    copy_source(source, data_zip)\n                run_live(\n                    command\n                    + [\n                        "--domain",\n                        domain,\n                        "--source-zip",\n                        str(data_zip),\n                        "--output",\n                        str(output),\n                        "--local",\n                        str(local),\n                    ],\n                    env,\n                    log,\n                )\n        except Exception as error:\n            failures[domain] = {"error": str(error), "log": log.name if log.is_file() else None}\n            print(f"[{domain}] \xeb\xaf\xb8\xec\x99\x84\xeb\xa3\x8c: {error}", flush=True)\n    completed = [\n        f"{d}/{a}"\n        for d in DOMAINS\n        for a in ("basic", "selected")\n        if (destination / d / a / "complete.json").is_file() and d not in failures\n    ]\n    write_progress(\n        destination / "summary.json",\n        {"completed_trials": completed, "failures": failures, "selection_written": False},\n    )\n    if failures:\n        raise RuntimeError(\n            f"\xeb\xaf\xb8\xec\x99\x84\xeb\xa3\x8c \xeb\xb6\x84\xec\x95\xbc\xea\xb0\x80 \xec\x9e\x88\xec\x8a\xb5\xeb\x8b\x88\xeb\x8b\xa4. \xec\x99\x84\xeb\xa3\x8c\xeb\x90\x9c \xea\xb8\xb0\xeb\xa1\x9d\xec\x9d\x80 Drive\xec\x97\x90 \xeb\xb3\xb4\xec\xa1\xb4\xed\x96\x88\xec\x8a\xb5\xeb\x8b\x88\xeb\x8b\xa4: {destination}"\n        )\n    run_live(\n        [str(python), "-m", "mediflow_datasets.adaptation_stage1", "--select", str(destination)],\n        env,\n        destination / f"selection_{uuid.uuid4().hex[:10]}.log",\n    )\n    write_progress(\n        destination / "summary.json",\n        {"completed_trials": completed, "failures": failures, "selection_written": True},\n    )\n    print("\xeb\xb9\x84\xea\xb5\x90\xed\x91\x9c:", destination / "validation_comparison.csv", flush=True)\n    return destination\n\n\ndef launch(runtime, work, project, bundle_sha, my_drive):\n    import fcntl\n\n    # Keep two cells in the same Colab VM from touching the same run concurrently.\n    lock = Path("/content") / f"mediflow_stage1_{bundle_sha}.lock"\n    with lock.open("a") as handle:\n        try:\n            fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)\n        except BlockingIOError as error:\n            raise RuntimeError("\xea\xb0\x99\xec\x9d\x80 Colab \xec\x84\xb8\xec\x85\x98\xec\x97\x90\xec\x84\x9c \xec\x9d\xb4\xeb\xaf\xb8 \xec\x8b\xa4\xed\x96\x89 \xec\xa4\x91\xec\x9e\x85\xeb\x8b\x88\xeb\x8b\xa4.") from error\n        return run(runtime, work, project, bundle_sha, my_drive)\n\n\nif __name__ == "__main__":\n    launch(*(Path(arg) if index != 3 else arg for index, arg in enumerate(sys.argv[1:])))\n', 'adaptation_stage2.py': b'"""Train three expanded camera treatments from the pinned Stage 1 starting models."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport os\nimport subprocess\nimport sys\nimport tempfile\nimport uuid\nfrom pathlib import Path\n\nfrom mediflow_datasets import adaptation_stage1_colab as colab\nfrom mediflow_datasets import domain_adaptation_batch as batch\nfrom mediflow_datasets.domain_adaptation_io import write_progress\n\n\ndef read_json(path):\n    return json.loads(path.read_text(encoding="utf-8"))\n\n\ndef starting_plan(runtime, stage1, bundle_sha):\n    config = read_json(runtime / "configs/adaptation_stage1.json")\n    previous = read_json(stage1 / "bundle.json")\n    summary = read_json(stage1 / "summary.json")\n    selected = read_json(stage1 / "selection.json")\n    expected = {f"{d}/{role}" for d in colab.DOMAINS for role in ("basic", "selected")}\n    if (\n        previous["sha256"] != bundle_sha\n        or previous["stage"] != 1\n        or set(summary["completed_trials"]) != expected\n        or summary["failures"]\n        or not summary["selection_written"]\n        or selected["schema"] != "mediflow_adaptation_stage1_selection_v1"\n    ):\n        raise ValueError("\xea\xb3\xb5\xed\x86\xb5 ZIP v3\xec\x99\x80 \xec\x9d\xbc\xec\xb9\x98\xed\x95\x98\xeb\x8a\x94 1\xec\xb0\xa8 \xec\x99\x84\xeb\xa3\x8c \xea\xb2\xb0\xea\xb3\xbc\xea\xb0\x80 \xed\x95\x84\xec\x9a\x94\xed\x95\xa9\xeb\x8b\x88\xeb\x8b\xa4.")\n    plan = {}\n    for domain in colab.DOMAINS:\n        task = config["tasks"][domain]\n        choice = selected["selected"][domain]\n        role = choice["role"]\n        spec = {**task["models"][role], "role": role}\n        record = read_json(stage1 / domain / role / "complete.json")\n        initial = {"path": f"{domain}/{role}/initial.keras", "sha256": spec["model_sha256"]}\n        adapted = {"path": f"{domain}/{role}/best.keras", "sha256": record["best_model"]["sha256"]}\n        if (\n            choice["initial_checkpoint"] != initial\n            or choice["stage1_checkpoint"] != adapted\n            or choice["model_config"] != spec\n            or record["model_config"] != spec\n            or record["initial"]["sha256"] != initial["sha256"]\n        ):\n            raise ValueError(f"{domain}\xec\x9d\x98 1\xec\xb0\xa8 \xec\x84\xa0\xed\x83\x9d \xea\xb8\xb0\xeb\xa1\x9d\xea\xb3\xbc \xec\xb6\x9c\xeb\xb0\x9c \xeb\xaa\xa8\xeb\x8d\xb8\xec\x9d\xb4 \xec\x9d\xbc\xec\xb9\x98\xed\x95\x98\xec\xa7\x80 \xec\x95\x8a\xec\x8a\xb5\xeb\x8b\x88\xeb\x8b\xa4.")\n        plan[domain] = {"spec": spec, "stage1_best": adapted}\n    return config, plan\n\n\ndef require_stage2(folder, bundle_sha):\n    marker, summary = read_json(folder / "bundle.json"), read_json(folder / "summary.json")\n    if (\n        marker["stage"] != 2\n        or marker["sha256"] != bundle_sha\n        or set(summary["completed_domains"]) != set(colab.DOMAINS)\n        or summary["failures"]\n        or not summary["comparison_written"]\n    ):\n        raise ValueError("\xea\xb3\xb5\xed\x86\xb5 ZIP v3\xec\x99\x80 \xec\x9d\xbc\xec\xb9\x98\xed\x95\x98\xeb\x8a\x94 2\xec\xb0\xa8 \xec\x99\x84\xeb\xa3\x8c \xea\xb2\xb0\xea\xb3\xbc\xea\xb0\x80 \xed\x95\x84\xec\x9a\x94\xed\x95\xa9\xeb\x8b\x88\xeb\x8b\xa4.")\n\n\ndef run_domain(runtime, stage1, bundle_sha, domain, archive, output, local, *, stage=2, key=None):\n    import cv2\n    import keras\n    import tensorflow as tf\n\n    from mediflow_datasets import adaptation_stage1 as train\n\n    config, plan = starting_plan(runtime, stage1, bundle_sha)\n    task = config["tasks"][domain]\n    spec, previous = plan[domain]["spec"], plan[domain]["stage1_best"]\n    reference = read_json(runtime / "references" / f"{domain}.json")\n    profile = reference["extended_profile"]\n    if profile["schema"] != "measured_camera_extended_v1" or profile["domain"] != domain:\n        raise ValueError(f"Unexpected expanded profile: {domain}")\n    original_path = train.data.local_path(runtime, spec["model"])\n    previous_path = train.data.local_path(stage1, previous["path"])\n    if batch.digest(original_path) != spec["model_sha256"] or (\n        batch.digest(previous_path) != previous["sha256"]\n    ):\n        raise ValueError(f"{domain} \xeb\xaa\xa8\xeb\x8d\xb8 \xed\x8c\x8c\xec\x9d\xbc\xec\x9d\xb4 1\xec\xb0\xa8 \xea\xb8\xb0\xeb\xa1\x9d\xea\xb3\xbc \xec\x9d\xbc\xec\xb9\x98\xed\x95\x98\xec\xa7\x80 \xec\x95\x8a\xec\x8a\xb5\xeb\x8b\x88\xeb\x8b\xa4.")\n    source_manifest = stage1 / domain / "source_manifest.json"\n    identity = {\n        "bundle_sha256": bundle_sha,\n        "selection_sha256": batch.digest(stage1 / "selection.json"),\n        "source_manifest_sha256": batch.digest(source_manifest),\n        "reference_sha256": train.object_hash(reference),\n        "configuration_sha256": train.object_hash(config),\n        "code_sha256": {"stage2": batch.digest(__file__), "training": batch.digest(train.__file__)},\n    }\n    treatment = "expanded"\n    benchmarks_paths = {"initial": original_path, "stage1_lighting": previous_path}\n    if stage == 3:\n        from mediflow_datasets import adaptation_stage3 as aligned\n\n        stage2 = stage1.with_name("stage2")\n        require_stage2(stage2, bundle_sha)\n        prior_folder = stage2 / domain\n        prior = read_json(prior_folder / "complete.json")\n        prior_config = read_json(prior_folder / "run_configuration.json")\n        prior_path = prior_folder / "best.keras"\n        if (\n            prior["model_config"] != spec\n            or prior["stage1_best"] != previous\n            or prior_config["training_configuration"] != config\n            or prior_config["profile"] != profile\n            or read_json(prior_folder / "source_manifest.json") != read_json(source_manifest)\n            or batch.digest(prior_path) != prior["best_model"]["sha256"]\n        ):\n            raise ValueError(f"{domain}\xec\x9d\x98 2\xec\xb0\xa8 \xeb\x8c\x80\xec\xa1\xb0\xea\xb5\xb0\xec\x9d\xb4 \xed\x98\x84\xec\x9e\xac \xec\x8b\xa4\xed\x97\x98 \xec\x84\xa4\xec\xa0\x95\xea\xb3\xbc \xec\x9d\xbc\xec\xb9\x98\xed\x95\x98\xec\xa7\x80 \xec\x95\x8a\xec\x8a\xb5\xeb\x8b\x88\xeb\x8b\xa4.")\n        benchmarks_paths["stage2_expanded"] = prior_path\n        identity["stage2_best_sha256"] = prior["best_model"]["sha256"]\n        identity["code_sha256"]["alignment"] = batch.digest(aligned.__file__)\n        treatment = "aligned"\n    output.mkdir(parents=True, exist_ok=True)\n    marker = output / "identity.json"\n    if marker.exists() and read_json(marker) != identity:\n        raise ValueError(f"{domain} \xea\xb8\xb0\xec\xa1\xb4 {stage}\xec\xb0\xa8 \xea\xb2\xb0\xea\xb3\xbc\xec\x99\x80 \xec\x9d\xb4\xeb\xb2\x88 \xec\x9e\x85\xeb\xa0\xa5\xec\x9d\xb4 \xec\x9d\xbc\xec\xb9\x98\xed\x95\x98\xec\xa7\x80 \xec\x95\x8a\xec\x8a\xb5\xeb\x8b\x88\xeb\x8b\xa4.")\n    write_progress(marker, identity)\n    complete = output / "complete.json"\n    if complete.is_file():\n        result = read_json(complete)\n        if batch.digest(output / "best.keras") != result["best_model"]["sha256"]:\n            raise ValueError(f"{domain} \xec\xa0\x80\xec\x9e\xa5\xeb\x90\x9c {stage}\xec\xb0\xa8 \xeb\xaa\xa8\xeb\x8d\xb8\xec\x9d\xb4 \xec\x99\x84\xeb\xa3\x8c \xea\xb8\xb0\xeb\xa1\x9d\xea\xb3\xbc \xec\x9d\xbc\xec\xb9\x98\xed\x95\x98\xec\xa7\x80 \xec\x95\x8a\xec\x8a\xb5\xeb\x8b\x88\xeb\x8b\xa4.")\n        print(f"[{domain}] {stage}\xec\xb0\xa8 \xec\x99\x84\xeb\xa3\x8c \xea\xb2\xb0\xea\xb3\xbc \xec\x9e\xac\xec\x82\xac\xec\x9a\xa9", flush=True)\n        return\n\n    target = None\n    if stage == 3:\n        target = aligned.target_inputs(runtime, domain, spec, task, key)\n        print(f"[{domain}] \xec\x8b\xa4\xec\xa0\x9c \xec\xb4\xac\xec\x98\x81 \xec\x9e\x85\xeb\xa0\xa5 {len(target[\'rows\'])}\xec\x9e\xa5 \xed\x99\x95\xec\x9d\xb8 \xc2\xb7 \xeb\xb9\x84\xeb\x9d\xbc\xeb\xb2\xa8 \xec\xa0\x95\xeb\xa0\xac\xec\x9a\xa9", flush=True)\n\n    write_progress(\n        output / "run_configuration.json",\n        {\n            "stage": stage,\n            "identity": identity,\n            "training_configuration": config,\n            "model": spec,\n            "stage1_best": previous,\n            "treatment": treatment,\n            "profile": profile,\n            "validation_conditions": ["clean", "lighting", "expanded"],\n            "validation_seed": 1707,\n            "tensorflow": tf.__version__,\n            "keras": keras.__version__,\n            "numpy": train.np.__version__,\n            "opencv": cv2.__version__,\n            "gpu": [\n                tf.config.experimental.get_device_details(d)\n                for d in tf.config.list_physical_devices("GPU")\n            ],\n            "bundle_provenance": read_json(runtime / "bundle_provenance.json"),\n        },\n    )\n    cv2.setNumThreads(1)\n    rows, val, pairs = train.prepare_source(domain, archive, task, reference, output)\n    if read_json(output / "source_manifest.json") != read_json(source_manifest):\n        raise ValueError(f"{domain}\xec\x9d\x98 Train/Validation/Test \xeb\xaa\xa9\xeb\xa1\x9d\xec\x9d\xb4 1\xec\xb0\xa8\xec\x99\x80 \xeb\x8b\xa4\xeb\xa6\x85\xeb\x8b\x88\xeb\x8b\xa4.")\n    size = spec["size"]\n    cache, shifted, pair_indexes = train.image_cache(\n        local, archive, rows, val, pairs, task, size, profile["lighting"], expanded=profile\n    )\n    keras.utils.set_random_seed(config["seed"])\n    model = keras.models.load_model(original_path, compile=False)\n    count = len(task["classes"])\n    if tuple(model.input_shape[1:]) != (size, size, 3) or (\n        len(model.outputs) != (4 if spec["architecture"] == "pmg" else 1)\n        or any(tensor.shape[-1] != count for tensor in model.outputs)\n    ):\n        raise ValueError(f"{domain} \xec\x8b\x9c\xec\x9e\x91 \xeb\xaa\xa8\xeb\x8d\xb8\xec\x9d\x98 \xec\x9e\x85\xeb\xa0\xa5 \xeb\x98\x90\xeb\x8a\x94 \xed\x81\xb4\xeb\x9e\x98\xec\x8a\xa4 \xec\x84\xa4\xec\xa0\x95\xec\x9d\xb4 \xeb\x8b\xa4\xeb\xa6\x85\xeb\x8b\x88\xeb\x8b\xa4.")\n    train.exp.configure_partial(model)\n    alignment = None\n    if target is not None:\n        calibration_path = output / "alignment_setup.json"\n        if calibration_path.is_file():\n            setup = read_json(calibration_path)\n        else:\n            setup = aligned.calibrate(model, cache, rows, task, spec, target)\n            write_progress(calibration_path, setup)\n        if setup["target_sha256"] != target["sha256"]:\n            raise ValueError("Target calibration does not match the bundled images")\n        alignment = {**target, **setup}\n    print(f"[{domain} \xc2\xb7 {spec[\'role\']}] 1\xec\xb0\xa8 \xed\x95\x99\xec\x8a\xb5 \xec\xa0\x84 \xec\xa0\x80\xec\x9e\xa5\xeb\xb3\xb8\xec\x97\x90\xec\x84\x9c {treatment} \xec\x8b\x9c\xec\x9e\x91", flush=True)\n    best = train.phase(\n        model,\n        (config, task, rows, val, profile, cache, shifted, pair_indexes),\n        output / treatment,\n        local / "saving",\n        spec,\n        treatment,\n        config["epochs"],\n        config["learning_rate"],\n        alignment=alignment,\n    )\n    saved_model = train.export_model(model, output, "best", local / "saving")\n    diagnostics = {}\n    predicted = {}\n    if alignment is not None:\n        diagnostics["stage3_aligned"], predicted["stage3_aligned"] = aligned.diagnostic(\n            model, shifted["expanded"], val, task, spec, target, setup["reference"]\n        )\n    del model\n    datasets = train.validation_pipelines(cache, shifted, len(rows), val, task["batch_size"], size)\n    benchmarks = {}\n    for name, path in benchmarks_paths.items():\n        keras.backend.clear_session()\n        model = keras.models.load_model(path, compile=False)\n        benchmarks[name] = train.validate(\n            train.predictor(model, spec), datasets, val, count, f"[{domain} \xc2\xb7 {name}]"\n        )\n        if alignment is not None:\n            diagnostics[name], predicted[name] = aligned.diagnostic(\n                model, shifted["expanded"], val, task, spec, target, setup["reference"]\n            )\n        del model\n    if alignment is not None:\n        for name, diagnostic in diagnostics.items():\n            diagnostic["predicted_class_changes_vs_initial"] = int(\n                (predicted[name] != predicted["initial"]).sum()\n            )\n        write_progress(\n            output / "feature_comparison.json",\n            {\n                "scope": "Expanded Source Validation sample vs unlabeled Target training images",\n                "class_names": task["classes"],\n                "target_accuracy": None,\n                "models": diagnostics,\n            },\n        )\n    benchmarks[f"stage{stage}_{treatment}"] = {name: best[name] for name in datasets}\n    write_progress(\n        complete,\n        {\n            "best_model": saved_model,\n            "best_epoch": best["epoch"],\n            "model_config": spec,\n            "stage1_best": previous,\n            "validation": benchmarks,\n            "validation_scope": config["validation_scope"],\n            "source_test_evaluated": False,\n        },\n    )\n    print(f"[{domain}] {stage}\xec\xb0\xa8 \xec\x99\x84\xeb\xa3\x8c \xc2\xb7 \xec\xb5\x9c\xec\xa0\x81 {best[\'epoch\']}\xed\x9a\x8c\xec\xb0\xa8 \xc2\xb7 Drive \xec\xa0\x80\xec\x9e\xa5 \xec\x99\x84\xeb\xa3\x8c", flush=True)\n\n\ndef write_comparison(destination):\n    rows = []\n    for domain in colab.DOMAINS:\n        complete = read_json(destination / domain / "complete.json")\n        for model, metrics in complete["validation"].items():\n            rows.append(\n                {\n                    "domain": domain,\n                    "start_role": complete["model_config"]["role"],\n                    "model": model,\n                    "validation_scope": complete["validation_scope"],\n                    **{f"{name}_macro_f1": result["macro_f1"] for name, result in metrics.items()},\n                    **{f"{name}_accuracy": result["accuracy"] for name, result in metrics.items()},\n                }\n            )\n    path = destination / "validation_comparison.csv"\n    temporary = path.with_name(f".{uuid.uuid4().hex}.csv.tmp")\n    with temporary.open("x", encoding="utf-8-sig", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n    temporary.replace(path)\n    return path\n\n\ndef launch(runtime, work, project, bundle_sha, my_drive, *, stage=2, key=None):\n    import fcntl\n\n    parent = my_drive / "MediFlow_adaptation_runs" / bundle_sha[:16]\n    stage1, destination = parent / "stage1", parent / f"stage{stage}"\n    config, _ = starting_plan(runtime, stage1, bundle_sha)\n    if stage == 3:\n        require_stage2(parent / "stage2", bundle_sha)\n    lock = Path("/content") / f"mediflow_stage{stage}_{bundle_sha}.lock"\n    with lock.open("a") as handle:\n        fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)\n        python, versions = batch.prepare_python(work, executor=colab.run_live)\n        if not versions["gpu"]:\n            raise RuntimeError("Colab \xeb\x9f\xb0\xed\x83\x80\xec\x9e\x84 \xec\x9c\xa0\xed\x98\x95\xec\x97\x90\xec\x84\x9c GPU\xeb\xa5\xbc \xec\x84\xa0\xed\x83\x9d\xed\x95\x9c \xeb\x92\xa4 \xeb\x8b\xa4\xec\x8b\x9c \xec\x8b\xa4\xed\x96\x89\xed\x95\x98\xec\x84\xb8\xec\x9a\x94.")\n        env = batch.python_environment(python, runtime)\n        env.update(PYTHONIOENCODING="utf-8", PYTHONDONTWRITEBYTECODE="1")\n        command = [str(python), "-m", "mediflow_datasets.adaptation_stage2", "--stage", str(stage)]\n        key_input = None\n        if stage == 3:\n            crypto_probe = [\n                str(python),\n                "-c",\n                "from cryptography.hazmat.primitives.ciphers.aead import AESGCM",\n            ]\n            if subprocess.run(crypto_probe, env=env, capture_output=True).returncode:\n                crypto = work / "target_crypto"\n                colab.run_live(\n                    [\n                        str(python),\n                        "-m",\n                        "pip",\n                        "install",\n                        "--disable-pip-version-check",\n                        "--target",\n                        str(crypto),\n                        "cryptography>=42,<49",\n                    ],\n                    env,\n                    work / "target_crypto.log",\n                )\n                env["PYTHONPATH"] += os.pathsep + str(crypto)\n            key_input = key.strip() + "\\n"\n            colab.run_live(\n                command + ["--check-target", str(runtime), str(stage1), bundle_sha],\n                env,\n                work / "target_check.log",\n                input_text=key_input,\n            )\n        destination.mkdir(exist_ok=True)\n        write_progress(\n            destination / "bundle.json",\n            {\n                "sha256": bundle_sha,\n                "stage": stage,\n                "environment": versions,\n                "launcher_sha256": batch.digest(__file__),\n            },\n        )\n        print(f"{stage}\xec\xb0\xa8 \xea\xb2\xb0\xea\xb3\xbc \xec\xa0\x80\xec\x9e\xa5:", destination, flush=True)\n        completed, failures = [], {}\n        for index, domain in enumerate(colab.DOMAINS, 1):\n            output = destination / domain\n            log = destination / f"{domain}_{uuid.uuid4().hex[:10]}.log"\n            print(f"\\n[{index}/3 \xc2\xb7 {domain}] {stage}\xec\xb0\xa8 \xec\xa4\x80\xeb\xb9\x84", flush=True)\n            try:\n                with tempfile.TemporaryDirectory(prefix=f"{domain}_", dir=work) as folder:\n                    local = Path(folder)\n                    source = local / "source.zip"\n                    if not (output / "complete.json").is_file():\n                        drive_source = batch.find_source(\n                            project / "datasets", config["tasks"][domain]\n                        )\n                        colab.copy_source(drive_source, source)\n                    args = (runtime, stage1, bundle_sha, domain, source, output, local)\n                    colab.run_live(\n                        command + ["--domain", *map(str, args)], env, log, input_text=key_input\n                    )\n                completed.append(domain)\n            except Exception as error:\n                failures[domain] = {"error": str(error), "log": log.name if log.is_file() else None}\n                print(f"[{domain}] \xeb\xaf\xb8\xec\x99\x84\xeb\xa3\x8c: {error}", flush=True)\n        if not failures:\n            print("\xeb\xb9\x84\xea\xb5\x90\xed\x91\x9c:", write_comparison(destination), flush=True)\n        write_progress(\n            destination / "summary.json",\n            {\n                "completed_domains": completed,\n                "failures": failures,\n                "comparison_written": not failures,\n                "source_test_evaluated": False,\n            },\n        )\n        if failures:\n            raise RuntimeError(f"\xeb\xaf\xb8\xec\x99\x84\xeb\xa3\x8c \xeb\xb6\x84\xec\x95\xbc\xea\xb0\x80 \xec\x9e\x88\xec\x8a\xb5\xeb\x8b\x88\xeb\x8b\xa4. \xeb\xa1\x9c\xea\xb7\xb8\xec\x99\x80 \xec\x99\x84\xeb\xa3\x8c \xea\xb8\xb0\xeb\xa1\x9d: {destination}")\n        print(f"{stage}\xec\xb0\xa8 \xec\x84\xb8 \xeb\xb6\x84\xec\x95\xbc \xec\x99\x84\xeb\xa3\x8c", flush=True)\n        return destination\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    mode = parser.add_mutually_exclusive_group(required=True)\n    mode.add_argument("--launch", nargs=5)\n    mode.add_argument("--domain", nargs=7)\n    mode.add_argument("--check-target", nargs=3)\n    parser.add_argument("--stage", type=int, choices=(2, 3), default=2)\n    args = parser.parse_args()\n    key = sys.stdin.readline().strip() if args.stage == 3 else None\n    if args.check_target:\n        if args.stage != 3:\n            raise ValueError("Target key validation requires Stage 3")\n        from mediflow_datasets.adaptation_stage3 import target_inputs\n\n        runtime, stage1, sha = args.check_target\n        config, plan = starting_plan(Path(runtime), Path(stage1), sha)\n        for domain, model in plan.items():\n            target = target_inputs(\n                Path(runtime), domain, model["spec"], config["tasks"][domain], key\n            )\n            print(f"[{domain}] \xec\x95\x94\xed\x98\xb8\xed\x99\x94 \xec\xb4\xac\xec\x98\x81 \xec\x9e\x85\xeb\xa0\xa5 {len(target[\'rows\'])}\xec\x9e\xa5 \xed\x99\x95\xec\x9d\xb8", flush=True)\n        return\n    if args.launch:\n        runtime, work, project, sha, my_drive = args.launch\n        launch(\n            Path(runtime), Path(work), Path(project), sha, Path(my_drive), stage=args.stage, key=key\n        )\n        return\n    runtime, stage1, sha, domain, source, output, local = args.domain\n    run_domain(\n        Path(runtime),\n        Path(stage1),\n        sha,\n        domain,\n        Path(source),\n        Path(output),\n        Path(local),\n        stage=args.stage,\n        key=key,\n    )\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n'}.items():
        (runtime / "src/mediflow_datasets" / name).write_bytes(content)
    study_path = work / "study.json"
    study_path.write_bytes(b'{\n  "schema": "mediflow_measured_strength_v1",\n  "bundle_sha256": "4ae6e07eb4af122531439519ae2207aad57a1c8dfb8f158152a2093fa65a91fe",\n  "hypothesis": "Pixel-replayed measured pilot photometric fit versus previous Stage 2",\n  "prior_source_test_inspected": true,\n  "source_test_evaluated": false,\n  "target_accuracy": null,\n  "fixed_conditions": [\n    "starting checkpoint",\n    "split",\n    "5 epochs",\n    "seed 42",\n    "Adam 1e-5",\n    "batch sizes",\n    "augmentation probability 0.5"\n  ],\n  "changed_condition": "Measured camera photometric profile",\n  "accompanying_changes": "Joint light-strength fit followed by residual color/focus re-audit",\n  "profile_builder_sha256": "a089bf6a043f3cd96faeef9d5ca7ef70bb39f1db8522ee87a1e77a737f172573",\n  "git_commit": "08cb827aa3c6858e61c71faf7f6481f3a139530b",\n  "uncommitted_runtime_sha256": {\n    "adaptation_strength.py": "4f52fc89b9e56dab6842bd44de97d9b760a2e5b8335e7efd434db474f4f4ba15",\n    "adaptation_stage1.py": "6b5aae35bcbe1af85fe19b6a47ebce6ed57bfca780129a82e95dd9dd0c9d4c8a",\n    "adaptation_stage1_colab.py": "ddd976b6e3d7e605cb4f6b8af4a95f647cd60f819348542094b2e0e18c80e854",\n    "adaptation_stage2.py": "764f19cf835450eb31224079d33d625a5aa9ff890a56c04285f862dc275696b9"\n  },\n  "domains": {\n    "skin": {\n      "profile": {\n        "schema": "measured_camera_extended_v1",\n        "domain": "skin",\n        "lighting": {\n          "domain": "skin",\n          "profile_version": "bounded_source_target_strength_v1",\n          "brightness_offset": [\n            67.47361687519953,\n            71.01787172836876\n          ],\n          "contrast": [\n            0.25,\n            0.26459406014955544\n          ],\n          "scale_x": [\n            1.0,\n            1.0\n          ],\n          "scale_y": [\n            1.0,\n            1.0\n          ],\n          "left_light": [\n            0.0,\n            0.0\n          ],\n          "evidence": {\n            "method": "Source class-balanced median to equal participant/condition Target quartiles",\n            "anchors": {\n              "y_p50": 83.40784072875977,\n              "contrast": 70.15270042419434\n            },\n            "coverage": {\n              "y_p50": {\n                "class_medians": [\n                  96.70617294311523,\n                  77.98558807373047,\n                  79.60416030883789,\n                  104.08260345458984,\n                  73.82204055786133,\n                  87.21152114868164,\n                  109.7138786315918,\n                  77.95131301879883,\n                  55.02832794189453,\n                  92.87034606933594\n                ],\n                "coverage_by_class": [\n                  {\n                    "total": 723,\n                    "measured": 723\n                  },\n                  {\n                    "total": 722,\n                    "measured": 722\n                  },\n                  {\n                    "total": 709,\n                    "measured": 709\n                  },\n                  {\n                    "total": 714,\n                    "measured": 714\n                  },\n                  {\n                    "total": 691,\n                    "measured": 691\n                  },\n                  {\n                    "total": 725,\n                    "measured": 725\n                  },\n                  {\n                    "total": 703,\n                    "measured": 703\n                  },\n                  {\n                    "total": 519,\n                    "measured": 519\n                  },\n                  {\n                    "total": 702,\n                    "measured": 702\n                  },\n                  {\n                    "total": 545,\n                    "measured": 545\n                  }\n                ]\n              },\n              "contrast": {\n                "class_medians": [\n                  50.152610778808594,\n                  86.36319732666016,\n                  54.131675720214844,\n                  55.54798126220703,\n                  74.786376953125,\n                  84.3202133178711,\n                  74.34446716308594,\n                  67.56494903564453,\n                  72.74045181274414,\n                  44.280799865722656\n                ],\n                "coverage_by_class": [\n                  {\n                    "total": 723,\n                    "measured": 723\n                  },\n                  {\n                    "total": 722,\n                    "measured": 722\n                  },\n                  {\n                    "total": 709,\n                    "measured": 709\n                  },\n                  {\n                    "total": 714,\n                    "measured": 714\n                  },\n                  {\n                    "total": 691,\n                    "measured": 691\n                  },\n                  {\n                    "total": 725,\n                    "measured": 725\n                  },\n                  {\n                    "total": 703,\n                    "measured": 703\n                  },\n                  {\n                    "total": 519,\n                    "measured": 519\n                  },\n                  {\n                    "total": 702,\n                    "measured": 702\n                  },\n                  {\n                    "total": 545,\n                    "measured": 545\n                  }\n                ]\n              }\n            }\n          },\n          "bounds": {\n            "max_brightness_offset": 160,\n            "contrast": [\n              0.25,\n              1.5\n            ],\n            "max_left_light": 64\n          },\n          "held_back": [\n            "color: unmatched disease color",\n            "blur/noise: texture/scale confounding",\n            "hair mask: not validated and could change class evidence",\n            "geometry: no rescaling onto artificial padding; composition is an input policy"\n          ],\n          "photometric_operator": "bounded_additive_v3",\n          "input_policy": "legacy_resize",\n          "source_sha256": "db6cf59087b68d4534ff857bdb0eacc1a1a09ed5703772fefc75f038d4570e4b",\n          "reference_model_sha256": "0b2a757320a113dbc1b285adab8c7745b2f7edc2cb29cc402d06f7882edc9f7a",\n          "calibration_fingerprint": "659c8bb3edf406ba89fe782781b4fe9b7ac065cfc87411d26c895aff7a4b75ea",\n          "source_subset_sha256": "59766e93e57214d1922caa5579fe78f8790a56a9fa5cdc7b439c43a93efc529b",\n          "train_visual_review": "class_brightness_extremes_reviewed",\n          "review_note": "Class brightness extremes inspected on before/previous/fitted contact sheets; all retained Train previews measured; no clinical preservation validation.",\n          "calibration_version": "pixel_replayed_photometric_refit_v1"\n        },\n        "color_a": [\n          -0.9609375,\n          0.0\n        ],\n        "color_b": [\n          0.0,\n          0.92578125\n        ],\n        "texture_gain": [\n          1,\n          1\n        ],\n        "blur_sigma_roi64": [\n          0,\n          0\n        ],\n        "blur_input_scale": 1.75,\n        "evidence": {\n          "chroma": {\n            "lab_a": {\n              "participant_changes": [\n                0.15625,\n                0.6875,\n                -0.515625,\n                -1.78125\n              ],\n              "range": [\n                -0.9609375,\n                0.0\n              ]\n            },\n            "lab_b": {\n              "participant_changes": [\n                -1.453125,\n                0.75,\n                0.09375,\n                -0.515625\n              ],\n              "range": [\n                0.0,\n                0.92578125\n              ]\n            }\n          },\n          "normalized_detail_ratios": [\n            1.5315814159199996,\n            0.9506550225759118,\n            0.93334946929935,\n            0.8952716478170912\n          ],\n          "focus_fits": [\n            {\n              "person": "P01",\n              "condition": "alt",\n              "sigma_roi64": 0.6000000000000001,\n              "reference_path": "skin/P01/S01/P01_S01_skin_alt_20261108T152804327485.png",\n              "compared_paths": [\n                "skin/P01/S01/P01_S01_skin_alt_20261108T152757692299.png",\n                "skin/P01/S01/P01_S01_skin_alt_20261108T152759879125.png",\n                "skin/P01/S01/P01_S01_skin_alt_20261108T152802271030.png",\n                "skin/P01/S01/P01_S01_skin_alt_20261108T152804327485.png",\n                "skin/P01/S01/P01_S01_skin_alt_20261108T152806871550.png"\n              ],\n              "normalized_detail_reference": 0.14585232734680176,\n              "normalized_detail_target_q25": 0.09714946150779724,\n              "fit_at_search_limit": false\n            },\n            {\n              "person": "P01",\n              "condition": "ref",\n              "sigma_roi64": 0.55,\n              "reference_path": "skin/P01/S01/P01_S01_skin_ref_20261108T152702885605.png",\n              "compared_paths": [\n                "skin/P01/S01/P01_S01_skin_ref_20261108T152654406408.png",\n                "skin/P01/S01/P01_S01_skin_ref_20261108T152656085726.png",\n                "skin/P01/S01/P01_S01_skin_ref_20261108T152700575134.png",\n                "skin/P01/S01/P01_S01_skin_ref_20261108T152702885605.png",\n                "skin/P01/S01/P01_S01_skin_ref_20261108T152726981437.png"\n              ],\n              "normalized_detail_reference": 0.0943775475025177,\n              "normalized_detail_target_q25": 0.06878045946359634,\n              "fit_at_search_limit": false\n            },\n            {\n              "person": "P02",\n              "condition": "alt",\n              "sigma_roi64": 0.55,\n              "reference_path": "skin/P02/S01/P02_S01_skin_ref_20261108T155710902205.png",\n              "compared_paths": [\n                "skin/P02/S01/P02_S01_skin_ref_20261108T155705604551.png",\n                "skin/P02/S01/P02_S01_skin_ref_20261108T155707918391.png",\n                "skin/P02/S01/P02_S01_skin_ref_20261108T155710902205.png",\n                "skin/P02/S01/P02_S01_skin_ref_20261108T155714224403.png",\n                "skin/P02/S01/P02_S01_skin_ref_20261108T155717083061.png"\n              ],\n              "normalized_detail_reference": 0.1551624834537506,\n              "normalized_detail_target_q25": 0.11530275642871857,\n              "fit_at_search_limit": false\n            },\n            {\n              "person": "P02",\n              "condition": "ref",\n              "sigma_roi64": 0.35000000000000003,\n              "reference_path": "skin/P02/S01/P02_S01_skin_ref_20261108T155640349143.png",\n              "compared_paths": [\n                "skin/P02/S01/P02_S01_skin_ref_20261108T155640349143.png",\n                "skin/P02/S01/P02_S01_skin_ref_20261108T155643620743.png",\n                "skin/P02/S01/P02_S01_skin_ref_20261108T155647691054.png",\n                "skin/P02/S01/P02_S01_skin_ref_20261108T155648665676.png",\n                "skin/P02/S01/P02_S01_skin_ref_20261108T155649721131.png",\n                "skin/P02/S01/P02_S01_skin_ref_20261108T155650848730.png"\n              ],\n              "normalized_detail_reference": 0.12822462618350983,\n              "normalized_detail_target_q25": 0.12492126226425171,\n              "fit_at_search_limit": false\n            },\n            {\n              "person": "P03",\n              "condition": "alt",\n              "sigma_roi64": 0.45,\n              "reference_path": "skin/P03/S01/P03_S01_skin_alt_20261108T160340079280.png",\n              "compared_paths": [\n                "skin/P03/S01/P03_S01_skin_alt_20261108T160327738793.png",\n                "skin/P03/S01/P03_S01_skin_alt_20261108T160329708329.png",\n                "skin/P03/S01/P03_S01_skin_alt_20261108T160331984806.png",\n                "skin/P03/S01/P03_S01_skin_alt_20261108T160335180900.png",\n                "skin/P03/S01/P03_S01_skin_alt_20261108T160340079280.png"\n              ],\n              "normalized_detail_reference": 0.1409766525030136,\n              "normalized_detail_target_q25": 0.1246052235364914,\n              "fit_at_search_limit": false\n            },\n            {\n              "person": "P03",\n              "condition": "ref",\n              "sigma_roi64": 0.4,\n              "reference_path": "skin/P03/S01/P03_S01_skin_ref_20261108T160307547385.png",\n              "compared_paths": [\n                "skin/P03/S01/P03_S01_skin_ref_20261108T160300453688.png",\n                "skin/P03/S01/P03_S01_skin_ref_20261108T160302282924.png",\n                "skin/P03/S01/P03_S01_skin_ref_20261108T160304846565.png",\n                "skin/P03/S01/P03_S01_skin_ref_20261108T160307547385.png",\n                "skin/P03/S01/P03_S01_skin_ref_20261108T160313317627.png"\n              ],\n              "normalized_detail_reference": 0.14165307581424713,\n              "normalized_detail_target_q25": 0.12634940445423126,\n              "fit_at_search_limit": false\n            },\n            {\n              "person": "P04",\n              "condition": "alt",\n              "sigma_roi64": 0.6000000000000001,\n              "reference_path": "skin/P04/S01/P04_S01_skin_alt_20261108T161536584800.png",\n              "compared_paths": [\n                "skin/P04/S01/P04_S01_skin_alt_20261108T161524502585.png",\n                "skin/P04/S01/P04_S01_skin_alt_20261108T161527705996.png",\n                "skin/P04/S01/P04_S01_skin_alt_20261108T161531183868.png",\n                "skin/P04/S01/P04_S01_skin_alt_20261108T161533594835.png",\n                "skin/P04/S01/P04_S01_skin_alt_20261108T161536584800.png"\n              ],\n              "normalized_detail_reference": 0.11807460337877274,\n              "normalized_detail_target_q25": 0.08239267021417618,\n              "fit_at_search_limit": false\n            },\n            {\n              "person": "P04",\n              "condition": "ref",\n              "sigma_roi64": 0.7000000000000001,\n              "reference_path": "skin/P04/S01/P04_S01_skin_ref_20261108T161422697239.png",\n              "compared_paths": [\n                "skin/P04/S01/P04_S01_skin_ref_20261108T161420722968.png",\n                "skin/P04/S01/P04_S01_skin_ref_20261108T161422697239.png",\n                "skin/P04/S01/P04_S01_skin_ref_20261108T161424373063.png",\n                "skin/P04/S01/P04_S01_skin_ref_20261108T161429837877.png",\n                "skin/P04/S01/P04_S01_skin_ref_20261108T161432163032.png",\n                "skin/P04/S01/P04_S01_skin_ref_20261108T161435199939.png"\n              ],\n              "normalized_detail_reference": 0.12333618104457855,\n              "normalized_detail_target_q25": 0.07842539250850677,\n              "fit_at_search_limit": false\n            }\n          ],\n          "source_train_statistics": {\n            "width": {\n              "n": 6753,\n              "mean": 512.0,\n              "var": 0.0,\n              "q05": 512.0,\n              "q25": 512.0,\n              "q50": 512.0,\n              "q75": 512.0,\n              "q95": 512.0\n            },\n            "height": {\n              "n": 6753,\n              "mean": 512.0,\n              "var": 0.0,\n              "q05": 512.0,\n              "q25": 512.0,\n              "q50": 512.0,\n              "q75": 512.0,\n              "q95": 512.0\n            },\n            "aspect": {\n              "n": 6753,\n              "mean": 1.0,\n              "var": 0.0,\n              "q05": 1.0,\n              "q25": 1.0,\n              "q50": 1.0,\n              "q75": 1.0,\n              "q95": 1.0\n            },\n            "frame_y_mean": {\n              "n": 6753,\n              "mean": 114.47958635284726,\n              "var": 87.24607925917684,\n              "q05": 98.81668624877929,\n              "q25": 108.35001754760742,\n              "q50": 114.64883804321289,\n              "q75": 120.7322006225586,\n              "q95": 129.66388626098632\n            },\n            "y_mean": {\n              "n": 6753,\n              "mean": 90.30464909543784,\n              "var": 314.8961629560137,\n              "q05": 59.69416427612305,\n              "q25": 77.81758880615234,\n              "q50": 91.09374237060547,\n              "q75": 103.08687591552734,\n              "q95": 118.37844696044921\n            },\n            "y_std": {\n              "n": 6753,\n              "mean": 27.32309096134558,\n              "var": 59.87403125311148,\n              "q05": 15.252275657653808,\n              "q25": 21.501598358154297,\n              "q50": 27.000465393066406,\n              "q75": 32.88024139404297,\n              "q95": 40.44438552856445\n            },\n            "y_p10": {\n              "n": 6753,\n              "mean": 58.886400914315765,\n              "var": 380.82845967317195,\n              "q05": 29.144909973144532,\n              "q25": 43.101741790771484,\n              "q50": 58.33773231506348,\n              "q75": 74.27768020629883,\n              "q95": 90.3705628967285\n            },\n            "y_p50": {\n              "n": 6753,\n              "mean": 86.80831658029916,\n              "var": 406.0092309856499,\n              "q05": 51.69489097595215,\n              "q25": 72.77033233642578,\n              "q50": 88.37580490112305,\n              "q75": 101.51531600952148,\n              "q95": 117.89541549682617\n            },\n            "y_p90": {\n              "n": 6753,\n              "mean": 125.98163479816716,\n              "var": 397.3568733147368,\n              "q05": 93.61960815429688,\n              "q25": 114.22539291381838,\n              "q50": 125.44330825805665,\n              "q75": 137.827294921875,\n              "q95": 160.09986907958984\n            },\n            "contrast": {\n              "n": 6753,\n              "mean": 67.09523358282188,\n              "var": 458.93342413074026,\n              "q05": 34.41724853515625,\n              "q25": 50.24510955810547,\n              "q50": 66.11698150634766,\n              "q75": 82.55947875976562,\n              "q95": 103.58233337402343\n            },\n            "very_bright": {\n              "n": 6753,\n              "mean": 1.7931845105878867e-05,\n              "var": 1.0437930924882913e-08,\n              "q05": 0.0,\n              "q25": 0.0,\n              "q50": 0.0,\n              "q75": 0.0,\n              "q95": 7.971938775510203e-05\n            },\n            "very_dark": {\n              "n": 6753,\n              "mean": 0.00015975751140082868,\n              "var": 2.2884004608760086e-06,\n              "q05": 0.0,\n              "q25": 0.0,\n              "q50": 0.0,\n              "q75": 0.0,\n              "q95": 0.00015943877551020407\n            },\n            "detail": {\n              "n": 6753,\n              "mean": 5.901556804200269,\n              "var": 6.615591828532237,\n              "q05": 2.219536018371582,\n              "q25": 3.957143783569336,\n              "q50": 5.602933883666992,\n              "q75": 7.533182144165039,\n              "q95": 10.604405212402344\n            },\n            "lab_a": {\n              "n": 6753,\n              "mean": 18.31927245298386,\n              "var": 76.4908259028224,\n              "q05": 5.015625,\n              "q25": 11.890625,\n              "q50": 17.34375,\n              "q75": 24.9921875,\n              "q95": 33.5\n            },\n            "lab_b": {\n              "n": 6753,\n              "mean": 13.335503665037761,\n              "var": 30.128186772458058,\n              "q05": 3.8125,\n              "q25": 9.875,\n              "q50": 13.390625,\n              "q75": 16.6875,\n              "q95": 22.734375\n            },\n            "lighting_range": {\n              "n": 6753,\n              "mean": 26.869026558886063,\n              "var": 282.25989583423865,\n              "q05": 6.5154876708984375,\n              "q25": 14.311271667480469,\n              "q50": 23.121639251708984,\n              "q75": 35.64080047607422,\n              "q95": 60.97312698364257\n            },\n            "r_p10": {\n              "n": 6753,\n              "mean": 92.25889119671201,\n              "var": 738.8185138834799,\n              "q05": 37.77306228637696,\n              "q25": 78.43979797363282,\n              "q50": 96.28571319580078,\n              "q75": 111.57449417114258,\n              "q95": 130.64938873291015\n            },\n            "r_p50": {\n              "n": 6753,\n              "mean": 121.62911342624804,\n              "var": 429.5406325190565,\n              "q05": 79.82091979980468,\n              "q25": 110.66836547851562,\n              "q50": 124.61734771728516,\n              "q75": 135.83163452148438,\n              "q95": 150.2193817138672\n            },\n            "r_p90": {\n              "n": 6753,\n              "mean": 153.09953683395307,\n              "var": 266.527168117746,\n              "q05": 125.96499908447267,\n              "q25": 142.97449798583983,\n              "q50": 153.19898529052733,\n              "q75": 163.71429443359375,\n              "q95": 179.5838781738281\n            },\n            "g_p10": {\n              "n": 6753,\n              "mean": 44.17473054689141,\n              "var": 422.4624981550124,\n              "q05": 14.131632347106935,\n              "q25": 27.714284896850586,\n              "q50": 42.1275520324707,\n              "q75": 59.944896697998054,\n              "q95": 78.92408203125\n            },\n            "g_p50": {\n              "n": 6753,\n              "mean": 73.4532289986573,\n              "var": 547.4737761948525,\n              "q05": 30.594898223876953,\n              "q25": 57.85714340209961,\n              "q50": 74.06887817382812,\n              "q75": 89.97193908691406,\n              "q95": 111.20306320190429\n            },\n            "g_p90": {\n              "n": 6753,\n              "mean": 117.08673418831088,\n              "var": 548.7626369136294,\n              "q05": 80.03285888671877,\n              "q25": 103.84948806762698,\n              "q50": 116.17857360839844,\n              "q75": 130.30510864257815,\n              "q95": 158.40153076171876\n            },\n            "b_p10": {\n              "n": 6753,\n              "mean": 38.451087429021065,\n              "var": 226.14975422575247,\n              "q05": 16.55806118011475,\n              "q25": 27.408163642883302,\n              "q50": 36.846427536010744,\n              "q75": 47.909695053100585,\n              "q95": 65.53969284057614\n            },\n            "b_p50": {\n              "n": 6753,\n              "mean": 65.42372289799097,\n              "var": 291.69216518860287,\n              "q05": 38.85867347717285,\n              "q25": 52.68877601623535,\n              "q50": 64.63010025024414,\n              "q75": 76.96683883666992,\n              "q95": 94.93622589111327\n            },\n            "b_p90": {\n              "n": 6753,\n              "mean": 107.51134333289255,\n              "var": 428.51867497457596,\n              "q05": 75.50520294189452,\n              "q25": 93.04846801757814,\n              "q50": 106.0,\n              "q75": 120.47193756103518,\n              "q95": 144.04938293457027\n            }\n          },\n          "target_statistics": {\n            "width": {\n              "n": 42,\n              "mean": 640.0,\n              "var": 0.0,\n              "q05": 640.0,\n              "q25": 640.0,\n              "q50": 640.0,\n              "q75": 640.0,\n              "q95": 640.0\n            },\n            "height": {\n              "n": 42,\n              "mean": 480.0,\n              "var": 0.0,\n              "q05": 480.0,\n              "q25": 480.0,\n              "q50": 480.0,\n              "q75": 480.0,\n              "q95": 480.0\n            },\n            "aspect": {\n              "n": 42,\n              "mean": 1.3333333333333333,\n              "var": 0.0,\n              "q05": 1.3333333333333333,\n              "q25": 1.3333333333333333,\n              "q50": 1.3333333333333333,\n              "q75": 1.3333333333333333,\n              "q95": 1.3333333333333333\n            },\n            "frame_y_mean": {\n              "n": 42,\n              "mean": 141.80286993117556,\n              "var": 4.310451009794682,\n              "q05": 138.94495605468748,\n              "q25": 140.46587158203124,\n              "q50": 141.25586100260415,\n              "q75": 143.70915283203126,\n              "q95": 145.35050097656247\n            },\n            "y_mean": {\n              "n": 42,\n              "mean": 155.47330329531715,\n              "var": 5.008440635400553,\n              "q05": 152.89668960571288,\n              "q25": 153.63165283203125,\n              "q50": 154.96517181396484,\n              "q75": 157.4436798095703,\n              "q95": 159.37026138305663\n            },\n            "y_std": {\n              "n": 42,\n              "mean": 9.428914728618803,\n              "var": 2.829139449333622,\n              "q05": 7.127552080154419,\n              "q25": 8.250247478485107,\n              "q50": 9.326010704040527,\n              "q75": 10.292496919631958,\n              "q95": 13.22414555549621\n            },\n            "y_p10": {\n              "n": 42,\n              "mean": 144.35247751871745,\n              "var": 5.533187754240107,\n              "q05": 140.94701385498047,\n              "q25": 142.9382297515869,\n              "q50": 143.66613082885743,\n              "q75": 146.12437324523927,\n              "q95": 148.2756336212158\n            },\n            "y_p50": {\n              "n": 42,\n              "mean": 154.95250393095472,\n              "var": 6.147326018129242,\n              "q05": 151.74092483520508,\n              "q25": 153.26583099365234,\n              "q50": 154.2252426147461,\n              "q75": 157.12484169006348,\n              "q95": 159.48736534118652\n            },\n            "y_p90": {\n              "n": 42,\n              "mean": 167.9121224176316,\n              "var": 8.209270375926142,\n              "q05": 162.84856636047365,\n              "q25": 165.90727386474612,\n              "q50": 168.01725158691406,\n              "q75": 169.62824020385742,\n              "q95": 172.11037048339844\n            },\n            "contrast": {\n              "n": 42,\n              "mean": 23.559644426618302,\n              "var": 9.826000824754386,\n              "q05": 19.020677947998045,\n              "q25": 21.123985290527344,\n              "q50": 23.768699645996094,\n              "q75": 26.311141967773438,\n              "q95": 27.667374420166013\n            },\n            "very_bright": {\n              "n": 42,\n              "mean": 0.0,\n              "var": 0.0,\n              "q05": 0.0,\n              "q25": 0.0,\n              "q50": 0.0,\n              "q75": 0.0,\n              "q95": 0.0\n            },\n            "very_dark": {\n              "n": 42,\n              "mean": 0.0,\n              "var": 0.0,\n              "q05": 0.0,\n              "q25": 0.0,\n              "q50": 0.0,\n              "q75": 0.0,\n              "q95": 0.0\n            },\n            "detail": {\n              "n": 42,\n              "mean": 2.5531061859357926,\n              "var": 0.515576507486669,\n              "q05": 1.3356806814670563,\n              "q25": 2.046028733253479,\n              "q50": 2.5456701517105103,\n              "q75": 3.2130183577537537,\n              "q95": 3.5456066250801084\n            },\n            "lab_a": {\n              "n": 42,\n              "mean": 2.4094122023809526,\n              "var": 1.3908315299589888,\n              "q05": 0.5265625,\n              "q25": 1.58984375,\n              "q50": 2.1953125,\n              "q75": 3.20703125,\n              "q95": 4.363281249999999\n            },\n            "lab_b": {\n              "n": 42,\n              "mean": 30.124441964285715,\n              "var": 5.113811363168314,\n              "q05": 27.596875,\n              "q25": 28.54296875,\n              "q50": 30.4765625,\n              "q75": 31.60546875,\n              "q95": 33.81015625\n            },\n            "lighting_range": {\n              "n": 42,\n              "mean": 4.645108904157366,\n              "var": 5.420522390871126,\n              "q05": 2.0537261962890625,\n              "q25": 2.7186660766601562,\n              "q50": 4.137176513671875,\n              "q75": 5.598941802978516,\n              "q95": 8.73028793334961\n            },\n            "r_p10": {\n              "n": 42,\n              "mean": 169.24217747279576,\n              "var": 14.438573350575775,\n              "q05": 164.11377334594727,\n              "q25": 165.52041244506836,\n              "q50": 168.3040802001953,\n              "q75": 172.53316116333008,\n              "q95": 175.86770347595214\n            },\n            "r_p50": {\n              "n": 42,\n              "mean": 178.81146712530227,\n              "var": 15.664006627342044,\n              "q05": 174.13571166992188,\n              "q25": 174.9285774230957,\n              "q50": 178.6556167602539,\n              "q75": 182.05101776123047,\n              "q95": 185.90255012512208\n            },\n            "r_p90": {\n              "n": 42,\n              "mean": 190.1097420102074,\n              "var": 15.070765466794745,\n              "q05": 183.20698570251466,\n              "q25": 188.30867385864258,\n              "q50": 189.88265228271484,\n              "q75": 192.12499618530273,\n              "q95": 196.22515365600586\n            },\n            "g_p10": {\n              "n": 42,\n              "mean": 142.0577498299735,\n              "var": 6.571455264308846,\n              "q05": 138.49030227661132,\n              "q25": 140.42857360839844,\n              "q50": 141.3484680175781,\n              "q75": 143.60025291442872,\n              "q95": 146.84218849182128\n            },\n            "g_p50": {\n              "n": 42,\n              "mean": 153.0891640072777,\n              "var": 6.768668294293346,\n              "q05": 149.02831802368163,\n              "q25": 151.22066688537598,\n              "q50": 152.82142639160156,\n              "q75": 154.6364803314209,\n              "q95": 157.30229568481445\n            },\n            "g_p90": {\n              "n": 42,\n              "mean": 166.57378619966053,\n              "var": 9.316482722943652,\n              "q05": 161.804598236084,\n              "q25": 164.26811180114748,\n              "q50": 166.43214645385743,\n              "q75": 168.77551651000977,\n              "q95": 171.23229400634764\n            },\n            "b_p10": {\n              "n": 42,\n              "mean": 88.87910610380628,\n              "var": 23.33520545854283,\n              "q05": 82.05377723693849,\n              "q25": 85.81964302062988,\n              "q50": 87.89948806762695,\n              "q75": 91.59413394927978,\n              "q95": 96.33178791046141\n            },\n            "b_p50": {\n              "n": 42,\n              "mean": 102.36479586646671,\n              "var": 16.101650183758434,\n              "q05": 97.44540672302246,\n              "q25": 99.79591369628906,\n              "q50": 102.03571319580078,\n              "q75": 105.5076551437378,\n              "q95": 107.72602291107178\n            },\n            "b_p90": {\n              "n": 42,\n              "mean": 118.02538942609515,\n              "var": 19.96475796479221,\n              "q05": 111.144030418396,\n              "q25": 115.1709213256836,\n              "q50": 118.38265609741211,\n              "q75": 121.53239612579345,\n              "q95": 123.92857360839844\n            }\n          }\n        },\n        "engineering_bounds": {\n          "chroma_limit_lab": 3,\n          "chroma_budget_policy": "q75 absolute within-person change; includes zero",\n          "texture_gain_limits": [\n            1,\n            1\n          ],\n          "blur_max_roi64": 0,\n          "lighting": {\n            "max_brightness_offset": 160,\n            "contrast": [\n              0.25,\n              1.5\n            ],\n            "max_left_light": 64\n          },\n          "pixel_refit_policy": {\n            "brightness_offset_limit": 160.0,\n            "contrast_limits": [\n              0.25,\n              1.5\n            ],\n            "side_light_limit": 64.0,\n            "probe_per_class": 4,\n            "full_replay_finalists": 3,\n            "spatial_correlation_q05_minimum": 0.95,\n            "chroma_limit_lab": 3.0,\n            "color_budget": "Observed absolute within-person condition change q75; zero-inclusive",\n            "hair_contrast": "Identity: scalp coverage and hair content confound global contrast",\n            "selection": "Minimize worst remaining primary W1 fraction, then mean fraction",\n            "limits_interpretation": "Engineering search limits, not clinically validated thresholds"\n          }\n        },\n        "limitations": [\n          "Within-person variation includes pose/content; not an isolated camera calibration",\n          "Texture and focus proxies are not independent physical measurements",\n          "Global disease colors/textures are not matched to unlabeled participants",\n          "Four participants are not a general camera population or independent Target Test",\n          "Source/Target brightness and contrast also contain disease and composition differences",\n          "Source face-ROI coverage varies by class; anchors describe successfully measured rows",\n          "Bounds are engineering choices, not validated physical camera calibration",\n          "This profile changes direction and calibration policy as well as amplitude",\n          "The existing Source Test has already been inspected; select on Validation only",\n          "Revised chroma direction remains confounded with disease, person and body region",\n          "Consistency across omissions is a sensitivity check, not independent validation",\n          "Focus gates are content-dependent proxies; no physical camera equivalence established",\n          "Photometric proxy fit to four calibration participants; no physical equivalence established",\n          "Spatial correlation is an engineering guard, not clinical lesion preservation validation",\n          "Source previews, not the entire original Source dataset, enter pixel replay"\n        ],\n        "previous_visual_review": {\n          "profile_version": "quarantined_within_person_bounded_v4",\n          "train_visual_review": "approved",\n          "review_note": "Inspected class-stratified retained Train previews and worst observed clipping examples. No gross lesion-boundary loss or artificial geometry observed at preview scale. All retained preview metrics and parameter corners were calculated. Some highlights approach bounds; this is a limited pilot approval, not image information or clinical accuracy validation. No face identity assessment performed. Source ROI missingness does not set brightness; within-person Target lighting differences are used. Whole Source/Target appearance matching is not claimed."\n        },\n        "calibration_version": "pixel_replayed_photometric_refit_v1",\n        "calibration_evidence": {\n          "method": "Retained Train pixel replay against unlabeled pilot captures",\n          "proposal": {\n            "source_class_balanced_luminance_anchor": 83.326003074646,\n            "target_equal_group_luminance_iqr": [\n              153.3482494354248,\n              156.10825729370117\n            ],\n            "smooth_operator_inverse_offsets": [\n              89.96482250026604,\n              94.69049563782501\n            ],\n            "target_to_source_contrast_iqr_ratio": [\n              0.2952120927340417,\n              0.3527920801994072\n            ],\n            "candidate_count": 20\n          },\n          "decision_file_sha256": "43d73ccc5516ad53f76cf67149b12d9fac2fe4092b44a5129bad2647112abf36",\n          "target_accuracy": null,\n          "full_training_executed": false\n        },\n        "revision_evidence": {\n          "color_direction": {\n            "a": {\n              "source_after_lighting_class_balanced_anchor": 15.72265625,\n              "target_equal_group_medians": {\n                "P01/alt": 3.046875,\n                "P01/ref": 2.90625,\n                "P02/ref": 1.5703125,\n                "P02/alt": 2.234375,\n                "P03/alt": 1.828125,\n                "P03/ref": 2.3125,\n                "P04/alt": 1.640625,\n                "P04/ref": 3.359375\n              },\n              "full_direction": -1,\n              "leave_one_person_out_directions": {\n                "P01": -1,\n                "P02": -1,\n                "P03": -1,\n                "P04": -1\n              },\n              "direction_stable": true,\n              "within_person_absolute_q75_limit": 0.9609375,\n              "proposed_range": [\n                -0.9609375,\n                0.0\n              ],\n              "policy": "Zero-inclusive bounded shift; direction stable across four omissions"\n            },\n            "b": {\n              "source_after_lighting_class_balanced_anchor": 11.990234375,\n              "target_equal_group_medians": {\n                "P01/alt": 30.046875,\n                "P01/ref": 31.453125,\n                "P02/ref": 27.828125,\n                "P02/alt": 28.59375,\n                "P03/alt": 31.0,\n                "P03/ref": 30.890625,\n                "P04/alt": 31.78125,\n                "P04/ref": 32.3203125\n              },\n              "full_direction": 1,\n              "leave_one_person_out_directions": {\n                "P01": 1,\n                "P02": 1,\n                "P03": 1,\n                "P04": 1\n              },\n              "direction_stable": true,\n              "within_person_absolute_q75_limit": 0.92578125,\n              "proposed_range": [\n                0.0,\n                0.92578125\n              ],\n              "policy": "Zero-inclusive bounded shift; direction stable across four omissions"\n            }\n          },\n          "color_choice": "both",\n          "focus_choice": "none",\n          "source_preview_count": 223,\n          "target_count": 42,\n          "seeds": [\n            42,\n            707,\n            1707\n          ],\n          "selection_basis": "Unlabeled Train/calibration proxy checks, not classifier scores"\n        },\n        "previous_engineering_bounds": {\n          "measured_change_fraction": 1.0,\n          "chroma_limit_lab": 3,\n          "texture_gain_limits": [\n            0.85,\n            1.15\n          ],\n          "focus_fit_grid_roi64": [\n            0,\n            1.5,\n            0.05\n          ],\n          "blur_max_roi64": 0.75,\n          "brightness_offset_limit": 32,\n          "contrast_limits": [\n            0.8,\n            1.25\n          ]\n        },\n        "pre_refit_engineering_bounds": {\n          "chroma_limit_lab": 3,\n          "chroma_budget_policy": "q75 absolute within-person change; includes zero",\n          "texture_gain_limits": [\n            1.0,\n            1.0\n          ],\n          "blur_max_roi64": 0.0,\n          "lighting": {\n            "max_brightness_offset": 32,\n            "contrast": [\n              0.8,\n              1.25\n            ]\n          }\n        },\n        "parent_calibration_evidence": {\n          "source_role": "retained_original_train",\n          "target_role": "unlabeled_calibration",\n          "source_class_balanced_anchors": {\n            "y_p50": 83.40784072875977,\n            "contrast": 70.15270042419434\n          },\n          "source_measurement_coverage": {\n            "y_p50": {\n              "class_medians": [\n                96.70617294311523,\n                77.98558807373047,\n                79.60416030883789,\n                104.08260345458984,\n                73.82204055786133,\n                87.21152114868164,\n                109.7138786315918,\n                77.95131301879883,\n                55.02832794189453,\n                92.87034606933594\n              ],\n              "coverage_by_class": [\n                {\n                  "total": 723,\n                  "measured": 723\n                },\n                {\n                  "total": 722,\n                  "measured": 722\n                },\n                {\n                  "total": 709,\n                  "measured": 709\n                },\n                {\n                  "total": 714,\n                  "measured": 714\n                },\n                {\n                  "total": 691,\n                  "measured": 691\n                },\n                {\n                  "total": 725,\n                  "measured": 725\n                },\n                {\n                  "total": 703,\n                  "measured": 703\n                },\n                {\n                  "total": 519,\n                  "measured": 519\n                },\n                {\n                  "total": 702,\n                  "measured": 702\n                },\n                {\n                  "total": 545,\n                  "measured": 545\n                }\n              ]\n            },\n            "contrast": {\n              "class_medians": [\n                50.152610778808594,\n                86.36319732666016,\n                54.131675720214844,\n                55.54798126220703,\n                74.786376953125,\n                84.3202133178711,\n                74.34446716308594,\n                67.56494903564453,\n                72.74045181274414,\n                44.280799865722656\n              ],\n              "coverage_by_class": [\n                {\n                  "total": 723,\n                  "measured": 723\n                },\n                {\n                  "total": 722,\n                  "measured": 722\n                },\n                {\n                  "total": 709,\n                  "measured": 709\n                },\n                {\n                  "total": 714,\n                  "measured": 714\n                },\n                {\n                  "total": 691,\n                  "measured": 691\n                },\n                {\n                  "total": 725,\n                  "measured": 725\n                },\n                {\n                  "total": 703,\n                  "measured": 703\n                },\n                {\n                  "total": 519,\n                  "measured": 519\n                },\n                {\n                  "total": 702,\n                  "measured": 702\n                },\n                {\n                  "total": 545,\n                  "measured": 545\n                }\n              ]\n            }\n          },\n          "target_equal_participant_condition_medians": {\n            "y_p50": [\n              155.1225357055664,\n              157.67981719970703,\n              151.86930084228516,\n              153.47462463378906,\n              152.83219146728516,\n              155.95436096191406,\n              155.48592376708984,\n              156.50391387939453\n            ],\n            "contrast": [\n              23.243118286132812,\n              24.954025268554688,\n              21.043075561523438,\n              26.485572814941406,\n              23.47039794921875,\n              25.83660888671875,\n              19.079574584960938,\n              20.986839294433594\n            ]\n          },\n          "raw_brightness_offsets": [\n            69.90617561340332,\n            72.68390846252441\n          ],\n          "raw_contrast_ratios": [\n            0.29976061317090036,\n            0.35885534014900783\n          ],\n          "brightness_limit": 32,\n          "contrast_limits": [\n            0.8,\n            1.25\n          ],\n          "hair_global_contrast": "identity; coverage and content remain confounded",\n          "revision_policy": "direction_and_focus_audited_v1"\n        }\n      },\n      "previous_reference_sha256": "1837fd554de1363d4695b01190d0625420e5fdde9775313d732fb93e69876172",\n      "source_manifest_sha256": "36f79b61a42e9ed634ffa7d4c490efefe5e5031a8bbdce1daf630b1e01e7e27d",\n      "previous_best_sha256": "9687fa503d8287b880e58e5014cae0a0efd47e750f5ececcc11d813367e7ce6b",\n      "measurement_provenance": {\n        "source": "camera_capture/_review/domain_adaptation_colab_20261109/skin/analysis/analysis.json",\n        "source_sha256": "0d701074947a030c66c1fb53ee6450a94515641183645b2ed7e636d7c9ae1eba",\n        "target": "camera_capture/_review/training_reference_inputs_20261110/skin/calibration.json",\n        "target_sha256": "6617cf541d62372720fc611baa2b3d0472bcdcdd543a7059bf0dd31db09bbbd2",\n        "capture_fingerprint": "659c8bb3edf406ba89fe782781b4fe9b7ac065cfc87411d26c895aff7a4b75ea",\n        "retained_train_count": 6753,\n        "target_count": 42\n      }\n    },\n    "web_skin": {\n      "profile": {\n        "schema": "measured_camera_extended_v1",\n        "domain": "web_skin",\n        "lighting": {\n          "domain": "web_skin",\n          "profile_version": "bounded_source_target_strength_v1",\n          "brightness_offset": [\n            -99.7760789846667,\n            -56.06404023208951\n          ],\n          "contrast": [\n            1.0,\n            1.0\n          ],\n          "scale_x": [\n            1.0,\n            1.0\n          ],\n          "scale_y": [\n            1.0,\n            1.0\n          ],\n          "left_light": [\n            3.5654401022290427,\n            64.0\n          ],\n          "evidence": {\n            "method": "Source class-balanced median to equal participant/condition Target quartiles",\n            "anchors": {\n              "y_p50": 167.1248779296875,\n              "contrast": 27.026893615722656\n            },\n            "coverage": {\n              "y_p50": {\n                "class_medians": [\n                  164.75593948364258,\n                  166.1425018310547,\n                  167.1248779296875,\n                  193.63400268554688,\n                  168.03762435913086\n                ],\n                "coverage_by_class": [\n                  {\n                    "total": 714,\n                    "measured": 356\n                  },\n                  {\n                    "total": 712,\n                    "measured": 595\n                  },\n                  {\n                    "total": 687,\n                    "measured": 643\n                  },\n                  {\n                    "total": 720,\n                    "measured": 719\n                  },\n                  {\n                    "total": 702,\n                    "measured": 670\n                  }\n                ]\n              },\n              "contrast": {\n                "class_medians": [\n                  27.026893615722656,\n                  24.09050750732422,\n                  22.20507049560547,\n                  28.38959503173828,\n                  30.2370548248291\n                ],\n                "coverage_by_class": [\n                  {\n                    "total": 714,\n                    "measured": 356\n                  },\n                  {\n                    "total": 712,\n                    "measured": 595\n                  },\n                  {\n                    "total": 687,\n                    "measured": 643\n                  },\n                  {\n                    "total": 720,\n                    "measured": 719\n                  },\n                  {\n                    "total": 702,\n                    "measured": 670\n                  }\n                ]\n              }\n            }\n          },\n          "bounds": {\n            "max_brightness_offset": 160,\n            "contrast": [\n              0.25,\n              1.5\n            ],\n            "max_left_light": 64\n          },\n          "held_back": [\n            "color: unmatched disease color",\n            "blur/noise: texture/scale confounding",\n            "hair mask: not validated and could change class evidence",\n            "geometry: no rescaling onto artificial padding; composition is an input policy"\n          ],\n          "photometric_operator": "bounded_additive_v3",\n          "input_policy": "center_square",\n          "source_sha256": "f8908af3d54e521ad14c37a44b569d33fe92be3b8b9b66a8d80faf4ba964072d",\n          "reference_model_sha256": "83e659dd09a9355135ee0de0197037e81ece962777f59dedae8d9a37b49a3fc4",\n          "calibration_fingerprint": "20dc5244d16d7e28a2eede6ad323a97a5db98fba2f5bb06420b0941dd1e481df",\n          "source_subset_sha256": "4377568a2b483f5b2efa38349f5469e658c74923c15dac924e1b3606ad4e0993",\n          "train_visual_review": "class_brightness_extremes_reviewed",\n          "review_note": "Class brightness extremes inspected on before/previous/fitted contact sheets; all retained Train previews measured; no clinical preservation validation.",\n          "calibration_version": "pixel_replayed_photometric_refit_v1"\n        },\n        "color_a": [\n          0,\n          0\n        ],\n        "color_b": [\n          0,\n          0\n        ],\n        "texture_gain": [\n          0.85,\n          0.9145815899839365\n        ],\n        "blur_sigma_roi64": [\n          0,\n          0.4\n        ],\n        "blur_input_scale": 1.9375,\n        "evidence": {\n          "chroma": {\n            "lab_a": {\n              "participant_changes": [\n                -0.53125,\n                -0.1640625,\n                -2.2734375,\n                -0.97265625\n              ],\n              "range": [\n                0.0,\n                0.0\n              ]\n            },\n            "lab_b": {\n              "participant_changes": [\n                -2.703125,\n                1.63671875,\n                -0.265625,\n                0.32421875\n              ],\n              "range": [\n                -1.9033203125,\n                0.0\n              ]\n            }\n          },\n          "normalized_detail_ratios": [\n            0.9041781600546774,\n            0.7888216863779229,\n            0.8610762805143408,\n            0.9457918797717134\n          ],\n          "focus_fits": [\n            {\n              "person": "P01",\n              "condition": "alt",\n              "sigma_roi64": 0.35000000000000003,\n              "reference_path": "web_skin/P01/S01/P01_S01_web_skin_alt_20261108T150143660730.png",\n              "compared_paths": [\n                "web_skin/P01/S01/P01_S01_web_skin_alt_20261108T150146243557.png",\n                "web_skin/P01/S01/P01_S01_web_skin_alt_20261108T150145477935.png",\n                "web_skin/P01/S01/P01_S01_web_skin_alt_20261108T150143660730.png"\n              ],\n              "normalized_detail_reference": 0.04072413966059685,\n              "normalized_detail_target_q25": 0.03876550309360027,\n              "fit_at_search_limit": false\n            },\n            {\n              "person": "P01",\n              "condition": "ref",\n              "sigma_roi64": 0.4,\n              "reference_path": "web_skin/P01/S01/P01_S01_web_skin_ref_20261108T150117359310.png",\n              "compared_paths": [\n                "web_skin/P01/S01/P01_S01_web_skin_ref_20261108T150117359310.png",\n                "web_skin/P01/S01/P01_S01_web_skin_ref_20261108T150121743228.png",\n                "web_skin/P01/S01/P01_S01_web_skin_ref_20261108T150120048313.png"\n              ],\n              "normalized_detail_reference": 0.042286016047000885,\n              "normalized_detail_target_q25": 0.03858809173107147,\n              "fit_at_search_limit": false\n            },\n            {\n              "person": "P02",\n              "condition": "alt",\n              "sigma_roi64": 0.35000000000000003,\n              "reference_path": "web_skin/P02/S01/P02_S01_web_skin_alt_20261108T155549996145.png",\n              "compared_paths": [\n                "web_skin/P02/S01/P02_S01_web_skin_alt_20261108T155546795247.png",\n                "web_skin/P02/S01/P02_S01_web_skin_alt_20261108T155549996145.png",\n                "web_skin/P02/S01/P02_S01_web_skin_alt_20261108T155547726859.png"\n              ],\n              "normalized_detail_reference": 0.03815416991710663,\n              "normalized_detail_target_q25": 0.03712081536650658,\n              "fit_at_search_limit": false\n            },\n            {\n              "person": "P02",\n              "condition": "ref",\n              "sigma_roi64": 0.4,\n              "reference_path": "web_skin/P02/S01/P02_S01_web_skin_ref_20261108T155523943430.png",\n              "compared_paths": [\n                "web_skin/P02/S01/P02_S01_web_skin_ref_20261108T155520318017.png",\n                "web_skin/P02/S01/P02_S01_web_skin_ref_20261108T155523943430.png",\n                "web_skin/P02/S01/P02_S01_web_skin_ref_20261108T155520566900.png"\n              ],\n              "normalized_detail_reference": 0.032488320022821426,\n              "normalized_detail_target_q25": 0.02975787501782179,\n              "fit_at_search_limit": false\n            },\n            {\n              "person": "P03",\n              "condition": "alt",\n              "sigma_roi64": 0.4,\n              "reference_path": "web_skin/P03/S01/P03_S01_web_skin_alt_20261108T160213812778.png",\n              "compared_paths": [\n                "web_skin/P03/S01/P03_S01_web_skin_alt_20261108T160217426731.png",\n                "web_skin/P03/S01/P03_S01_web_skin_alt_20261108T160213812778.png",\n                "web_skin/P03/S01/P03_S01_web_skin_alt_20261108T160214841649.png"\n              ],\n              "normalized_detail_reference": 0.04655195027589798,\n              "normalized_detail_target_q25": 0.04139178246259689,\n              "fit_at_search_limit": false\n            },\n            {\n              "person": "P03",\n              "condition": "ref",\n              "sigma_roi64": 0.4,\n              "reference_path": "web_skin/P03/S01/P03_S01_web_skin_ref_20261108T160141027206.png",\n              "compared_paths": [\n                "web_skin/P03/S01/P03_S01_web_skin_ref_20261108T160139114564.png",\n                "web_skin/P03/S01/P03_S01_web_skin_ref_20261108T160135334160.png",\n                "web_skin/P03/S01/P03_S01_web_skin_ref_20261108T160141027206.png"\n              ],\n              "normalized_detail_reference": 0.049063827842473984,\n              "normalized_detail_target_q25": 0.04606209881603718,\n              "fit_at_search_limit": false\n            },\n            {\n              "person": "P04",\n              "condition": "alt",\n              "sigma_roi64": 0.30000000000000004,\n              "reference_path": "web_skin/P04/S01/P04_S01_web_skin_alt_20261108T161326825741.png",\n              "compared_paths": [\n                "web_skin/P04/S01/P04_S01_web_skin_alt_20261108T161326825741.png",\n                "web_skin/P04/S01/P04_S01_web_skin_alt_20261108T161323856383.png",\n                "web_skin/P04/S01/P04_S01_web_skin_alt_20261108T161325776274.png"\n              ],\n              "normalized_detail_reference": 0.045572228729724884,\n              "normalized_detail_target_q25": 0.04498937726020813,\n              "fit_at_search_limit": false\n            },\n            {\n              "person": "P04",\n              "condition": "ref",\n              "sigma_roi64": 0.4,\n              "reference_path": "web_skin/P04/S01/P04_S01_web_skin_ref_20261108T161259907934.png",\n              "compared_paths": [\n                "web_skin/P04/S01/P04_S01_web_skin_ref_20261108T161258616795.png",\n                "web_skin/P04/S01/P04_S01_web_skin_ref_20261108T161259907934.png",\n                "web_skin/P04/S01/P04_S01_web_skin_ref_20261108T161317619759.png"\n              ],\n              "normalized_detail_reference": 0.058970071375370026,\n              "normalized_detail_target_q25": 0.05298882909119129,\n              "fit_at_search_limit": false\n            }\n          ],\n          "source_train_statistics": {\n            "width": {\n              "n": 3535,\n              "mean": 1024.0,\n              "var": 0.0,\n              "q05": 1024.0,\n              "q25": 1024.0,\n              "q50": 1024.0,\n              "q75": 1024.0,\n              "q95": 1024.0\n            },\n            "height": {\n              "n": 3535,\n              "mean": 1024.0,\n              "var": 0.0,\n              "q05": 1024.0,\n              "q25": 1024.0,\n              "q50": 1024.0,\n              "q75": 1024.0,\n              "q95": 1024.0\n            },\n            "aspect": {\n              "n": 3535,\n              "mean": 1.0,\n              "var": 0.0,\n              "q05": 1.0,\n              "q25": 1.0,\n              "q50": 1.0,\n              "q75": 1.0,\n              "q95": 1.0\n            },\n            "frame_y_mean": {\n              "n": 3535,\n              "mean": 122.4840412606613,\n              "var": 174.63957111668591,\n              "q05": 100.79979181289673,\n              "q25": 113.60491180419922,\n              "q50": 122.60625743865967,\n              "q75": 131.76815176010132,\n              "q95": 144.24287633895872\n            },\n            "face_w": {\n              "n": 2983,\n              "mean": 0.7038756730849816,\n              "var": 0.010719641525178694,\n              "q05": 0.4990234375,\n              "q25": 0.658203125,\n              "q50": 0.734375,\n              "q75": 0.77490234375,\n              "q95": 0.818359375\n            },\n            "face_h": {\n              "n": 2983,\n              "mean": 0.7038782920926919,\n              "var": 0.010721216919545137,\n              "q05": 0.4990234375,\n              "q25": 0.658203125,\n              "q50": 0.734375,\n              "q75": 0.775390625,\n              "q95": 0.818359375\n            },\n            "face_x": {\n              "n": 2983,\n              "mean": 0.4952846403054811,\n              "var": 0.0005910972975787344,\n              "q05": 0.46875,\n              "q25": 0.4873046875,\n              "q50": 0.49755859375,\n              "q75": 0.50537109375,\n              "q95": 0.51904296875\n            },\n            "face_y": {\n              "n": 2983,\n              "mean": 0.49806962762948376,\n              "var": 0.002680517827346623,\n              "q05": 0.41943359375,\n              "q25": 0.45947265625,\n              "q50": 0.5009765625,\n              "q75": 0.5302734375,\n              "q95": 0.57958984375\n            },\n            "y_mean": {\n              "n": 2983,\n              "mean": 171.60286215700003,\n              "var": 340.5626552189052,\n              "q05": 142.94990234375,\n              "q25": 160.08666229248047,\n              "q50": 170.08668518066406,\n              "q75": 182.89017486572266,\n              "q95": 204.19006652832033\n            },\n            "y_std": {\n              "n": 2983,\n              "mean": 13.824026793711065,\n              "var": 28.611544611011293,\n              "q05": 6.904114818572998,\n              "q25": 10.07047176361084,\n              "q50": 12.990439414978027,\n              "q75": 16.47877597808838,\n              "q95": 23.480698585510254\n            },\n            "y_p10": {\n              "n": 2983,\n              "mean": 153.8254084425647,\n              "var": 433.1489020992675,\n              "q05": 119.6391700744629,\n              "q25": 141.92668838500975,\n              "q50": 153.04942779541017,\n              "q75": 165.74359970092775,\n              "q95": 190.13076248168946\n            },\n            "y_p50": {\n              "n": 2983,\n              "mean": 172.59530727630727,\n              "var": 360.93161945805053,\n              "q05": 143.72868576049805,\n              "q25": 160.80518341064453,\n              "q50": 170.97125244140625,\n              "q75": 184.3146858215332,\n              "q95": 206.0211311340332\n            },\n            "y_p90": {\n              "n": 2983,\n              "mean": 188.0014141879464,\n              "var": 336.62722341229323,\n              "q05": 159.26213500976564,\n              "q25": 176.1097557067871,\n              "q50": 186.53913269042968,\n              "q75": 200.18789443969726,\n              "q95": 219.2217042541504\n            },\n            "contrast": {\n              "n": 2983,\n              "mean": 28.567295564248024,\n              "var": 145.80308910326926,\n              "q05": 14.091455078125,\n              "q25": 20.190940856933594,\n              "q50": 26.279129028320312,\n              "q75": 33.947147369384766,\n              "q95": 51.409949493408206\n            },\n            "very_bright": {\n              "n": 2983,\n              "mean": 0.0,\n              "var": 0.0,\n              "q05": 0.0,\n              "q25": 0.0,\n              "q50": 0.0,\n              "q75": 0.0,\n              "q95": 0.0\n            },\n            "very_dark": {\n              "n": 2983,\n              "mean": 2.257461191420293e-06,\n              "var": 1.5196662733752344e-08,\n              "q05": 0.0,\n              "q25": 0.0,\n              "q50": 0.0,\n              "q75": 0.0,\n              "q95": 0.0\n            },\n            "detail": {\n              "n": 2983,\n              "mean": 0.49890150976165293,\n              "var": 0.09083875806631915,\n              "q05": 0.12404881790280342,\n              "q25": 0.262665294110775,\n              "q50": 0.44187019765377045,\n              "q75": 0.6757000982761383,\n              "q95": 1.082399696111679\n            },\n            "lab_a": {\n              "n": 2983,\n              "mean": 14.891709269192088,\n              "var": 23.005682807687716,\n              "q05": 7.0,\n              "q25": 12.0,\n              "q50": 14.671875,\n              "q75": 17.81640625,\n              "q95": 23.07734375\n            },\n            "lab_b": {\n              "n": 2983,\n              "mean": 20.73640473097553,\n              "var": 50.364169986750426,\n              "q05": 7.71875,\n              "q25": 16.07421875,\n              "q50": 21.6875,\n              "q75": 26.0,\n              "q95": 30.9671875\n            },\n            "lighting_range": {\n              "n": 2983,\n              "mean": 30.582603060715638,\n              "var": 227.63935227652632,\n              "q05": 11.45445556640625,\n              "q25": 19.742637634277344,\n              "q50": 27.928878784179688,\n              "q75": 38.418434143066406,\n              "q95": 58.480264282226564\n            },\n            "r_p10": {\n              "n": 2983,\n              "mean": 195.33571069393227,\n              "var": 558.134331727566,\n              "q05": 153.7725,\n              "q25": 182.25,\n              "q50": 196.0,\n              "q75": 209.25,\n              "q95": 235.46\n            },\n            "r_p50": {\n              "n": 2983,\n              "mean": 210.31352665102247,\n              "var": 460.8270383672918,\n              "q05": 173.275,\n              "q25": 197.0625,\n              "q50": 210.0,\n              "q75": 223.8125,\n              "q95": 246.875\n            },\n            "r_p90": {\n              "n": 2983,\n              "mean": 221.05195273214886,\n              "var": 391.97067943356865,\n              "q05": 187.0,\n              "q25": 208.25,\n              "q50": 221.0,\n              "q75": 235.5,\n              "q95": 252.5\n            },\n            "g_p10": {\n              "n": 2983,\n              "mean": 140.27132919879315,\n              "var": 466.92841134922116,\n              "q05": 104.975,\n              "q25": 127.25,\n              "q50": 139.75,\n              "q75": 153.6875,\n              "q95": 177.5225\n            },\n            "g_p50": {\n              "n": 2983,\n              "mean": 160.78247569560844,\n              "var": 389.1172563835024,\n              "q05": 131.0,\n              "q25": 148.0,\n              "q50": 159.25,\n              "q75": 173.1875,\n              "q95": 195.25\n            },\n            "g_p90": {\n              "n": 2983,\n              "mean": 178.013794837412,\n              "var": 373.13305894818654,\n              "q05": 148.6825,\n              "q25": 165.5,\n              "q50": 176.70000000000002,\n              "q75": 191.0,\n              "q95": 211.25\n            },\n            "b_p10": {\n              "n": 2983,\n              "mean": 112.22356687898088,\n              "var": 468.43108514864485,\n              "q05": 78.055,\n              "q25": 98.0,\n              "q50": 111.125,\n              "q75": 126.0,\n              "q95": 149.09\n            },\n            "b_p50": {\n              "n": 2983,\n              "mean": 134.49773717733825,\n              "var": 425.0373524056142,\n              "q05": 102.0,\n              "q25": 120.75,\n              "q50": 133.75,\n              "q75": 148.6875,\n              "q95": 168.625\n            },\n            "b_p90": {\n              "n": 2983,\n              "mean": 155.19481226952732,\n              "var": 436.5251266660647,\n              "q05": 120.75,\n              "q25": 141.7,\n              "q50": 155.575,\n              "q75": 168.8,\n              "q95": 189.5\n            },\n            "side_light": {\n              "n": 2983,\n              "mean": -2.789991009846814,\n              "var": 216.62861919520685,\n              "q05": -27.705178833007814,\n              "q25": -11.624557495117188,\n              "q50": -2.094879150390625,\n              "q75": 7.0645599365234375,\n              "q95": 19.366043090820312\n            }\n          },\n          "target_statistics": {\n            "width": {\n              "n": 42,\n              "mean": 720.0,\n              "var": 0.0,\n              "q05": 720.0,\n              "q25": 720.0,\n              "q50": 720.0,\n              "q75": 720.0,\n              "q95": 720.0\n            },\n            "height": {\n              "n": 42,\n              "mean": 720.0,\n              "var": 0.0,\n              "q05": 720.0,\n              "q25": 720.0,\n              "q50": 720.0,\n              "q75": 720.0,\n              "q95": 720.0\n            },\n            "aspect": {\n              "n": 42,\n              "mean": 1.0,\n              "var": 0.0,\n              "q05": 1.0,\n              "q25": 1.0,\n              "q50": 1.0,\n              "q75": 1.0,\n              "q95": 1.0\n            },\n            "frame_y_mean": {\n              "n": 42,\n              "mean": 117.96577284501763,\n              "var": 168.44828486838938,\n              "q05": 97.55988040123458,\n              "q25": 106.15853973765432,\n              "q50": 121.94068769290124,\n              "q75": 125.36228298611111,\n              "q95": 141.28436178626544\n            },\n            "native_width": {\n              "n": 42,\n              "mean": 1280.0,\n              "var": 0.0,\n              "q05": 1280.0,\n              "q25": 1280.0,\n              "q50": 1280.0,\n              "q75": 1280.0,\n              "q95": 1280.0\n            },\n            "native_height": {\n              "n": 42,\n              "mean": 720.0,\n              "var": 0.0,\n              "q05": 720.0,\n              "q25": 720.0,\n              "q50": 720.0,\n              "q75": 720.0,\n              "q95": 720.0\n            },\n            "face_w": {\n              "n": 42,\n              "mean": 0.47837301587301584,\n              "var": 0.002169876438229613,\n              "q05": 0.39458333333333334,\n              "q25": 0.4472222222222222,\n              "q50": 0.48333333333333334,\n              "q75": 0.5083333333333333,\n              "q95": 0.5541666666666666\n            },\n            "face_h": {\n              "n": 42,\n              "mean": 0.47837301587301584,\n              "var": 0.002169876438229613,\n              "q05": 0.39458333333333334,\n              "q25": 0.4472222222222222,\n              "q50": 0.48333333333333334,\n              "q75": 0.5083333333333333,\n              "q95": 0.5541666666666666\n            },\n            "face_x": {\n              "n": 42,\n              "mean": 0.5650462962962963,\n              "var": 0.003975043785518322,\n              "q05": 0.4625694444444445,\n              "q25": 0.5145833333333333,\n              "q50": 0.5916666666666667,\n              "q75": 0.6246527777777777,\n              "q95": 0.6442361111111111\n            },\n            "face_y": {\n              "n": 42,\n              "mean": 0.5393849206349207,\n              "var": 0.005078981245275889,\n              "q05": 0.4222222222222222,\n              "q25": 0.46215277777777775,\n              "q50": 0.5625,\n              "q75": 0.5934027777777777,\n              "q95": 0.6263194444444444\n            },\n            "y_mean": {\n              "n": 42,\n              "mean": 108.4215832664853,\n              "var": 471.73526077400675,\n              "q05": 81.75565910339355,\n              "q25": 97.35028457641602,\n              "q50": 101.1187858581543,\n              "q75": 124.03936767578125,\n              "q95": 144.38405075073243\n            },\n            "y_std": {\n              "n": 42,\n              "mean": 12.71238576798212,\n              "var": 27.17134695743536,\n              "q05": 6.027664399147033,\n              "q25": 7.170705556869507,\n              "q50": 12.186700344085693,\n              "q75": 17.962547779083252,\n              "q95": 19.73787727355957\n            },\n            "y_p10": {\n              "n": 42,\n              "mean": 92.96419746762231,\n              "var": 309.3590120859504,\n              "q05": 73.76079399108887,\n              "q25": 76.78439102172851,\n              "q50": 89.62044029235841,\n              "q75": 106.92601470947267,\n              "q95": 123.44529014587403\n            },\n            "y_p50": {\n              "n": 42,\n              "mean": 106.60154070172992,\n              "var": 415.21468403277703,\n              "q05": 81.44257717132568,\n              "q25": 95.33713245391846,\n              "q50": 100.76734733581543,\n              "q75": 115.55965232849121,\n              "q95": 139.64438972473144\n            },\n            "y_p90": {\n              "n": 42,\n              "mean": 126.24842383975076,\n              "var": 824.4180391542692,\n              "q05": 88.9665768814087,\n              "q25": 109.57465324401855,\n              "q50": 118.08759078979493,\n              "q75": 155.39628486633302,\n              "q95": 172.2517123413086\n            },\n            "contrast": {\n              "n": 42,\n              "mean": 23.63250796000163,\n              "var": 31.77594674984333,\n              "q05": 15.582783126831055,\n              "q25": 18.345415115356445,\n              "q50": 24.028310775756836,\n              "q75": 27.55977439880371,\n              "q95": 32.18445816040039\n            },\n            "very_bright": {\n              "n": 42,\n              "mean": 0.0,\n              "var": 0.0,\n              "q05": 0.0,\n              "q25": 0.0,\n              "q50": 0.0,\n              "q75": 0.0,\n              "q95": 0.0\n            },\n            "very_dark": {\n              "n": 42,\n              "mean": 0.0,\n              "var": 0.0,\n              "q05": 0.0,\n              "q25": 0.0,\n              "q50": 0.0,\n              "q75": 0.0,\n              "q95": 0.0\n            },\n            "detail": {\n              "n": 42,\n              "mean": 0.10692421950045086,\n              "var": 0.00038114416721783337,\n              "q05": 0.07504560090601445,\n              "q25": 0.09076076932251453,\n              "q50": 0.10582415945827961,\n              "q75": 0.12221345119178295,\n              "q95": 0.13948970828205345\n            },\n            "lab_a": {\n              "n": 42,\n              "mean": 15.566778273809524,\n              "var": 1.9094418549483594,\n              "q05": 13.469140625,\n              "q25": 14.6953125,\n              "q50": 15.45703125,\n              "q75": 16.59375,\n              "q95": 17.8703125\n            },\n            "lab_b": {\n              "n": 42,\n              "mean": 13.899925595238095,\n              "var": 13.184434402016015,\n              "q05": 8.048046875,\n              "q25": 10.611328125,\n              "q50": 13.7265625,\n              "q75": 17.736328125,\n              "q95": 19.006249999999998\n            },\n            "lighting_range": {\n              "n": 42,\n              "mean": 31.957427796863374,\n              "var": 247.98203200329277,\n              "q05": 12.51398811340332,\n              "q25": 17.754268646240234,\n              "q50": 29.24612808227539,\n              "q75": 45.80515670776367,\n              "q95": 54.02673950195312\n            },\n            "r_p10": {\n              "n": 42,\n              "mean": 124.67250046502977,\n              "var": 344.8074200952891,\n              "q05": 103.9420166015625,\n              "q25": 106.6775146484375,\n              "q50": 121.81464843750001,\n              "q75": 137.7626953125,\n              "q95": 155.14917480468748\n            },\n            "r_p50": {\n              "n": 42,\n              "mean": 138.02106584821428,\n              "var": 424.9906201200421,\n              "q05": 112.015625,\n              "q25": 126.3297119140625,\n              "q50": 131.5458984375,\n              "q75": 147.0126953125,\n              "q95": 171.9563720703125\n            },\n            "r_p90": {\n              "n": 42,\n              "mean": 155.45787527901786,\n              "var": 878.8976339353021,\n              "q05": 118.23587890625,\n              "q25": 137.4837890625,\n              "q50": 147.3919921875,\n              "q75": 181.68759765625,\n              "q95": 204.9484033203125\n            },\n            "g_p10": {\n              "n": 42,\n              "mean": 81.36427408854168,\n              "var": 282.89784631271215,\n              "q05": 62.83041015625,\n              "q25": 67.665625,\n              "q50": 76.99370117187499,\n              "q75": 95.308740234375,\n              "q95": 110.8648828125\n            },\n            "g_p50": {\n              "n": 42,\n              "mean": 95.46513439360119,\n              "var": 395.1578419555882,\n              "q05": 71.2409912109375,\n              "q25": 85.3331298828125,\n              "q50": 88.64208984375,\n              "q75": 103.992431640625,\n              "q95": 127.4594970703125\n            },\n            "g_p90": {\n              "n": 42,\n              "mean": 115.66137230282737,\n              "var": 778.3190722293864,\n              "q05": 79.87362304687501,\n              "q25": 97.6484375,\n              "q50": 108.1435546875,\n              "q75": 145.5100341796875,\n              "q95": 159.31440429687498\n            },\n            "b_p10": {\n              "n": 42,\n              "mean": 68.77881556919644,\n              "var": 385.041113648366,\n              "q05": 46.137138671875,\n              "q25": 47.9603515625,\n              "q50": 70.10048828125,\n              "q75": 81.8400390625,\n              "q95": 101.8196240234375\n            },\n            "b_p50": {\n              "n": 42,\n              "mean": 82.75233677455357,\n              "var": 532.924830329256,\n              "q05": 54.620068359375,\n              "q25": 63.442626953125,\n              "q50": 82.416259765625,\n              "q75": 92.9210205078125,\n              "q95": 120.99169921875\n            },\n            "b_p90": {\n              "n": 42,\n              "mean": 104.50118815104166,\n              "var": 955.9511582215247,\n              "q05": 63.3044384765625,\n              "q25": 88.7904296875,\n              "q50": 97.46328125,\n              "q75": 134.455322265625,\n              "q95": 153.45194824218748\n            },\n            "side_light": {\n              "n": 42,\n              "mean": 13.002259753999256,\n              "var": 216.15893591643612,\n              "q05": -6.440453720092774,\n              "q25": 0.09168052673339844,\n              "q50": 10.742233276367188,\n              "q75": 27.831308364868164,\n              "q95": 30.24348678588867\n            }\n          }\n        },\n        "engineering_bounds": {\n          "chroma_limit_lab": 3,\n          "chroma_budget_policy": "q75 absolute within-person change; includes zero",\n          "texture_gain_limits": [\n            0.85,\n            0.9145815899839365\n          ],\n          "blur_max_roi64": 0.4,\n          "lighting": {\n            "max_brightness_offset": 160,\n            "contrast": [\n              0.25,\n              1.5\n            ],\n            "max_left_light": 64\n          },\n          "pixel_refit_policy": {\n            "brightness_offset_limit": 160.0,\n            "contrast_limits": [\n              0.25,\n              1.5\n            ],\n            "side_light_limit": 64.0,\n            "probe_per_class": 4,\n            "full_replay_finalists": 3,\n            "spatial_correlation_q05_minimum": 0.95,\n            "chroma_limit_lab": 3.0,\n            "color_budget": "Observed absolute within-person condition change q75; zero-inclusive",\n            "hair_contrast": "Identity: scalp coverage and hair content confound global contrast",\n            "selection": "Minimize worst remaining primary W1 fraction, then mean fraction",\n            "limits_interpretation": "Engineering search limits, not clinically validated thresholds"\n          }\n        },\n        "limitations": [\n          "Within-person variation includes pose/content; not an isolated camera calibration",\n          "Texture and focus proxies are not independent physical measurements",\n          "Global disease colors/textures are not matched to unlabeled participants",\n          "Four participants are not a general camera population or independent Target Test",\n          "Source/Target brightness and contrast also contain disease and composition differences",\n          "Source face-ROI coverage varies by class; anchors describe successfully measured rows",\n          "Bounds are engineering choices, not validated physical camera calibration",\n          "This profile changes direction and calibration policy as well as amplitude",\n          "The existing Source Test has already been inspected; select on Validation only",\n          "Revised chroma direction remains confounded with disease, person and body region",\n          "Consistency across omissions is a sensitivity check, not independent validation",\n          "Focus gates are content-dependent proxies; no physical camera equivalence established",\n          "Photometric proxy fit to four calibration participants; no physical equivalence established",\n          "Spatial correlation is an engineering guard, not clinical lesion preservation validation",\n          "Source previews, not the entire original Source dataset, enter pixel replay"\n        ],\n        "previous_visual_review": {\n          "profile_version": "quarantined_within_person_bounded_v4",\n          "train_visual_review": "approved",\n          "review_note": "Inspected class-stratified retained Train previews and worst observed clipping examples. No gross lesion-boundary loss or artificial geometry observed at preview scale. All retained preview metrics and parameter corners were calculated. Some highlights approach bounds; this is a limited pilot approval, not image information or clinical accuracy validation. No face identity assessment performed. Source ROI missingness does not set brightness; within-person Target lighting differences are used. Whole Source/Target appearance matching is not claimed."\n        },\n        "calibration_version": "pixel_replayed_photometric_refit_v1",\n        "calibration_evidence": {\n          "method": "Retained Train pixel replay against unlabeled pilot captures",\n          "proposal": {\n            "source_class_balanced_luminance_anchor": 168.30550384521484,\n            "target_equal_group_luminance_iqr": [\n              93.03293895721436,\n              120.62324714660645\n            ],\n            "smooth_operator_inverse_offsets": [\n              -99.7760789846667,\n              -56.06404023208951\n            ],\n            "target_to_source_contrast_iqr_ratio": [\n              0.8046412357482071,\n              1.0740764648944454\n            ],\n            "candidate_count": 40\n          },\n          "decision_file_sha256": "43d73ccc5516ad53f76cf67149b12d9fac2fe4092b44a5129bad2647112abf36",\n          "target_accuracy": null,\n          "full_training_executed": false\n        },\n        "revision_evidence": {\n          "color_direction": {\n            "a": {\n              "source_after_lighting_class_balanced_anchor": 16.5234375,\n              "target_equal_group_medians": {\n                "P01/alt": 15.0234375,\n                "P01/ref": 15.40625,\n                "P02/alt": 16.546875,\n                "P02/ref": 16.80859375,\n                "P03/alt": 15.6796875,\n                "P03/ref": 17.90625,\n                "P04/alt": 13.53125,\n                "P04/ref": 14.421875\n              },\n              "full_direction": 0,\n              "leave_one_person_out_directions": {\n                "P01": 0,\n                "P02": -1,\n                "P03": -1,\n                "P04": 0\n              },\n              "direction_stable": false,\n              "within_person_absolute_q75_limit": 1.2978515625,\n              "proposed_range": [\n                0.0,\n                0.0\n              ],\n              "policy": "Zero-inclusive bounded shift; direction stable across four omissions"\n            },\n            "b": {\n              "source_after_lighting_class_balanced_anchor": 20.734375,\n              "target_equal_group_medians": {\n                "P01/alt": 12.8984375,\n                "P01/ref": 15.53125,\n                "P02/alt": 9.8515625,\n                "P02/ref": 8.171875,\n                "P03/alt": 13.625,\n                "P03/ref": 13.890625,\n                "P04/alt": 18.84375,\n                "P04/ref": 18.484375\n              },\n              "full_direction": -1,\n              "leave_one_person_out_directions": {\n                "P01": -1,\n                "P02": -1,\n                "P03": -1,\n                "P04": -1\n              },\n              "direction_stable": true,\n              "within_person_absolute_q75_limit": 1.9033203125,\n              "proposed_range": [\n                -1.9033203125,\n                0.0\n              ],\n              "policy": "Zero-inclusive bounded shift; direction stable across four omissions"\n            }\n          },\n          "color_choice": "both",\n          "focus_choice": "both",\n          "source_preview_count": 118,\n          "target_count": 42,\n          "seeds": [\n            42,\n            707,\n            1707\n          ],\n          "selection_basis": "Unlabeled Train/calibration proxy checks, not classifier scores"\n        },\n        "previous_engineering_bounds": {\n          "measured_change_fraction": 1.0,\n          "chroma_limit_lab": 3,\n          "texture_gain_limits": [\n            0.85,\n            1.15\n          ],\n          "focus_fit_grid_roi64": [\n            0,\n            1.5,\n            0.05\n          ],\n          "blur_max_roi64": 0.75,\n          "brightness_offset_limit": 32,\n          "contrast_limits": [\n            0.8,\n            1.25\n          ]\n        },\n        "pre_refit_engineering_bounds": {\n          "chroma_limit_lab": 3,\n          "chroma_budget_policy": "q75 absolute within-person change; includes zero",\n          "texture_gain_limits": [\n            0.85,\n            0.9145815899839365\n          ],\n          "blur_max_roi64": 0.4,\n          "lighting": {\n            "max_brightness_offset": 32,\n            "contrast": [\n              0.8,\n              1.25\n            ]\n          }\n        },\n        "parent_calibration_evidence": {\n          "source_role": "retained_original_train",\n          "target_role": "unlabeled_calibration",\n          "source_class_balanced_anchors": {\n            "y_p50": 167.1248779296875,\n            "contrast": 27.026893615722656\n          },\n          "source_measurement_coverage": {\n            "y_p50": {\n              "class_medians": [\n                164.75593948364258,\n                166.1425018310547,\n                167.1248779296875,\n                193.63400268554688,\n                168.03762435913086\n              ],\n              "coverage_by_class": [\n                {\n                  "total": 714,\n                  "measured": 356\n                },\n                {\n                  "total": 712,\n                  "measured": 595\n                },\n                {\n                  "total": 687,\n                  "measured": 643\n                },\n                {\n                  "total": 720,\n                  "measured": 719\n                },\n                {\n                  "total": 702,\n                  "measured": 670\n                }\n              ]\n            },\n            "contrast": {\n              "class_medians": [\n                27.026893615722656,\n                24.09050750732422,\n                22.20507049560547,\n                28.38959503173828,\n                30.2370548248291\n              ],\n              "coverage_by_class": [\n                {\n                  "total": 714,\n                  "measured": 356\n                },\n                {\n                  "total": 712,\n                  "measured": 595\n                },\n                {\n                  "total": 687,\n                  "measured": 643\n                },\n                {\n                  "total": 720,\n                  "measured": 719\n                },\n                {\n                  "total": 702,\n                  "measured": 670\n                }\n              ]\n            }\n          },\n          "target_equal_participant_condition_medians": {\n            "y_p50": [\n              114.7862548828125,\n              82.22571182250977,\n              139.51790618896484,\n              100.60310363769531,\n              136.71737670898438,\n              102.7280044555664,\n              95.67724990844727,\n              85.4756031036377\n            ],\n            "contrast": [\n              26.097904205322266,\n              16.046756744384766,\n              31.983932495117188,\n              21.72176170349121,\n              31.467559814453125,\n              24.71625518798828,\n              24.18914794921875,\n              15.956871032714844\n            ]\n          },\n          "raw_brightness_offsets": [\n            -73.99803972244263,\n            -46.85584259033203\n          ],\n          "raw_contrast_ratios": [\n            0.7512150953191121,\n            1.0152967817078993\n          ],\n          "brightness_limit": 32,\n          "contrast_limits": [\n            0.8,\n            1.25\n          ],\n          "hair_global_contrast": "identity; coverage and content remain confounded",\n          "revision_policy": "direction_and_focus_audited_v1"\n        }\n      },\n      "previous_reference_sha256": "239245cc7a0804c931ddd45b5b9408474fb8cce5838a00b00d65471b87221b4e",\n      "source_manifest_sha256": "7c8b41878883193e5dd340ab8a4b7f804a3ab7f37c8159093b663937d997a738",\n      "previous_best_sha256": "5e181c1ff064eb1d501c57ccecba91703cfe59b14d02b258152b3ca737c46981",\n      "measurement_provenance": {\n        "source": "camera_capture/_review/domain_adaptation_colab_20261109/web_skin/analysis/analysis.json",\n        "source_sha256": "9f4c9d26e6915e61b08da96955a806e29c2ab157af3b8982111508bad2f36dde",\n        "target": "camera_capture/_review/training_reference_inputs_20261110/web_skin/calibration.json",\n        "target_sha256": "5a68556d2692571057e834370a9087f7d9e2a769459a3289898f819d5bc00646",\n        "capture_fingerprint": "20dc5244d16d7e28a2eede6ad323a97a5db98fba2f5bb06420b0941dd1e481df",\n        "retained_train_count": 3535,\n        "target_count": 42\n      }\n    },\n    "hair": {\n      "profile": {\n        "schema": "measured_camera_extended_v1",\n        "domain": "hair",\n        "lighting": {\n          "domain": "hair",\n          "profile_version": "bounded_source_target_strength_v1",\n          "brightness_offset": [\n            15.043113708496094,\n            26.26080894470215\n          ],\n          "contrast": [\n            1.0,\n            1.0\n          ],\n          "scale_x": [\n            1.0,\n            1.0\n          ],\n          "scale_y": [\n            1.0,\n            1.0\n          ],\n          "left_light": [\n            0.0,\n            0.0\n          ],\n          "evidence": {\n            "method": "Source class-balanced median to equal participant/condition Target quartiles",\n            "anchors": {\n              "y_p50": 160.440185546875,\n              "contrast": 70.79170608520508\n            },\n            "coverage": {\n              "y_p50": {\n                "class_medians": [\n                  150.06185150146484,\n                  160.9347801208496,\n                  160.440185546875,\n                  157.4539794921875,\n                  165.97833251953125\n                ],\n                "coverage_by_class": [\n                  {\n                    "total": 1990,\n                    "measured": 1990\n                  },\n                  {\n                    "total": 2050,\n                    "measured": 2050\n                  },\n                  {\n                    "total": 1929,\n                    "measured": 1929\n                  },\n                  {\n                    "total": 1965,\n                    "measured": 1965\n                  },\n                  {\n                    "total": 2098,\n                    "measured": 2098\n                  }\n                ]\n              },\n              "contrast": {\n                "class_medians": [\n                  58.59663391113281,\n                  70.79170608520508,\n                  83.7009048461914,\n                  35.322113037109375,\n                  71.07597732543945\n                ],\n                "coverage_by_class": [\n                  {\n                    "total": 1990,\n                    "measured": 1990\n                  },\n                  {\n                    "total": 2050,\n                    "measured": 2050\n                  },\n                  {\n                    "total": 1929,\n                    "measured": 1929\n                  },\n                  {\n                    "total": 1965,\n                    "measured": 1965\n                  },\n                  {\n                    "total": 2098,\n                    "measured": 2098\n                  }\n                ]\n              }\n            }\n          },\n          "bounds": {\n            "max_brightness_offset": 160,\n            "contrast": [\n              0.25,\n              1.5\n            ],\n            "max_left_light": 64\n          },\n          "held_back": [\n            "color: unmatched disease color",\n            "blur/noise: texture/scale confounding",\n            "hair mask: not validated and could change class evidence",\n            "geometry: no rescaling onto artificial padding; composition is an input policy",\n            "contrast/highlights: hair coverage and scalp visibility changed"\n          ],\n          "source_sha256": "2ac7260663cf69835ba50edb6ae8c7e7ac13be9c73b9f7ea24f60e0342e7e156",\n          "model_sha256": "9faa33b8f79f45ef6e7be415473e2ee5225f6195fff04d098828733db23ed3c2",\n          "calibration_fingerprint": "2d2ab5106e7f079147d4cd1e2e6e0ce1c27e8494d554085b4ad4947e3cd2b22c",\n          "input_policy": "legacy_resize",\n          "train_visual_review": "class_brightness_extremes_reviewed",\n          "selected_from": "verified_train_and_target_calibration",\n          "photometric_operator": "bounded_additive_v3",\n          "review_note": "Class brightness extremes inspected on before/previous/fitted contact sheets; all retained Train previews measured; no clinical preservation validation.",\n          "calibration_version": "pixel_replayed_photometric_refit_v1"\n        },\n        "color_a": [\n          0,\n          0\n        ],\n        "color_b": [\n          0,\n          0\n        ],\n        "texture_gain": [\n          1,\n          1\n        ],\n        "blur_sigma_roi64": [\n          0,\n          0\n        ],\n        "blur_input_scale": 3.0,\n        "evidence": {\n          "chroma": {\n            "lab_a": {\n              "participant_changes": [\n                -1.265625,\n                0.59375,\n                -0.8125,\n                -1.140625\n              ],\n              "range": [\n                0.0,\n                0.0\n              ]\n            },\n            "lab_b": {\n              "participant_changes": [\n                -0.640625,\n                -2.21875,\n                0.390625,\n                -0.359375\n              ],\n              "range": [\n                0.0,\n                0.0\n              ]\n            }\n          },\n          "normalized_detail_ratios": [\n            0.7956155523836431,\n            0.903396231148771,\n            0.8178386205760373,\n            1.00576106234326\n          ],\n          "focus_fits": [\n            {\n              "person": "P01",\n              "condition": "alt",\n              "sigma_roi64": 0.5,\n              "reference_path": "hair/P01/S01/P01_S01_hair_alt_20261108T153239675963.png",\n              "compared_paths": [\n                "hair/P01/S01/P01_S01_hair_alt_20261108T153159444214.png",\n                "hair/P01/S01/P01_S01_hair_alt_20261108T153239675963.png",\n                "hair/P01/S01/P01_S01_hair_alt_20261108T153151051139.png"\n              ],\n              "normalized_detail_reference": 0.06333822011947632,\n              "normalized_detail_target_q25": 0.048252301290631294,\n              "fit_at_search_limit": false\n            },\n            {\n              "person": "P01",\n              "condition": "ref",\n              "sigma_roi64": 0.7000000000000001,\n              "reference_path": "hair/P01/S01/P01_S01_hair_ref_20261108T153045788746.png",\n              "compared_paths": [\n                "hair/P01/S01/P01_S01_hair_ref_20261108T153045788746.png",\n                "hair/P01/S01/P01_S01_hair_ref_20261108T153121869582.png",\n                "hair/P01/S01/P01_S01_hair_ref_20261108T153051029359.png"\n              ],\n              "normalized_detail_reference": 0.06347747892141342,\n              "normalized_detail_target_q25": 0.04593171179294586,\n              "fit_at_search_limit": false\n            },\n            {\n              "person": "P02",\n              "condition": "alt",\n              "sigma_roi64": 0.5,\n              "reference_path": "hair/P02/S01/P02_S01_hair_alt_20261108T155856007554.png",\n              "compared_paths": [\n                "hair/P02/S01/P02_S01_hair_alt_20261108T155902899143.png",\n                "hair/P02/S01/P02_S01_hair_alt_20261108T155856007554.png",\n                "hair/P02/S01/P02_S01_hair_alt_20261108T155847429457.png"\n              ],\n              "normalized_detail_reference": 0.0768783763051033,\n              "normalized_detail_target_q25": 0.06241852976381779,\n              "fit_at_search_limit": false\n            },\n            {\n              "person": "P02",\n              "condition": "ref",\n              "sigma_roi64": 0.4,\n              "reference_path": "hair/P02/S01/P02_S01_hair_ref_20261108T155823882222.png",\n              "compared_paths": [\n                "hair/P02/S01/P02_S01_hair_ref_20261108T155823882222.png",\n                "hair/P02/S01/P02_S01_hair_ref_20261108T155807961945.png",\n                "hair/P02/S01/P02_S01_hair_ref_20261108T155819402127.png"\n              ],\n              "normalized_detail_reference": 0.07383783161640167,\n              "normalized_detail_target_q25": 0.06689455173909664,\n              "fit_at_search_limit": false\n            },\n            {\n              "person": "P03",\n              "condition": "alt",\n              "sigma_roi64": 0.65,\n              "reference_path": "hair/P03/S01/P03_S01_hair_alt_20261108T160524791795.png",\n              "compared_paths": [\n                "hair/P03/S01/P03_S01_hair_alt_20261108T160524791795.png",\n                "hair/P03/S01/P03_S01_hair_alt_20261108T160528077321.png",\n                "hair/P03/S01/P03_S01_hair_alt_20261108T160534515213.png"\n              ],\n              "normalized_detail_reference": 0.04810452461242676,\n              "normalized_detail_target_q25": 0.034384097903966904,\n              "fit_at_search_limit": false\n            },\n            {\n              "person": "P03",\n              "condition": "ref",\n              "sigma_roi64": 0.8,\n              "reference_path": "hair/P03/S01/P03_S01_hair_ref_20261108T160432797868.png",\n              "compared_paths": [\n                "hair/P03/S01/P03_S01_hair_ref_20261108T160447415708.png",\n                "hair/P03/S01/P03_S01_hair_ref_20261108T160414107183.png",\n                "hair/P03/S01/P03_S01_hair_ref_20261108T160432797868.png"\n              ],\n              "normalized_detail_reference": 0.08795812726020813,\n              "normalized_detail_target_q25": 0.05506112985312939,\n              "fit_at_search_limit": false\n            },\n            {\n              "person": "P04",\n              "condition": "alt",\n              "sigma_roi64": 0.4,\n              "reference_path": "hair/P04/S01/P04_S01_hair_alt_20261108T161821606621.png",\n              "compared_paths": [\n                "hair/P04/S01/P04_S01_hair_alt_20261108T161818945430.png",\n                "hair/P04/S01/P04_S01_hair_alt_20261108T161821606621.png",\n                "hair/P04/S01/P04_S01_hair_alt_20261108T161755328976.png"\n              ],\n              "normalized_detail_reference": 0.0599982924759388,\n              "normalized_detail_target_q25": 0.05631589889526367,\n              "fit_at_search_limit": false\n            },\n            {\n              "person": "P04",\n              "condition": "ref",\n              "sigma_roi64": 0.75,\n              "reference_path": "hair/P04/S01/P04_S01_hair_ref_20261108T161741046903.png",\n              "compared_paths": [\n                "hair/P04/S01/P04_S01_hair_ref_20261108T161656311169.png",\n                "hair/P04/S01/P04_S01_hair_ref_20261108T161717832940.png",\n                "hair/P04/S01/P04_S01_hair_ref_20261108T161741046903.png"\n              ],\n              "normalized_detail_reference": 0.06583444774150848,\n              "normalized_detail_target_q25": 0.04240403510630131,\n              "fit_at_search_limit": false\n            }\n          ],\n          "source_train_statistics": {\n            "width": {\n              "n": 10032,\n              "mean": 640.0,\n              "var": 0.0,\n              "q05": 640.0,\n              "q25": 640.0,\n              "q50": 640.0,\n              "q75": 640.0,\n              "q95": 640.0\n            },\n            "height": {\n              "n": 10032,\n              "mean": 480.0,\n              "var": 0.0,\n              "q05": 480.0,\n              "q25": 480.0,\n              "q50": 480.0,\n              "q75": 480.0,\n              "q95": 480.0\n            },\n            "aspect": {\n              "n": 10032,\n              "mean": 1.3333333333333333,\n              "var": 0.0,\n              "q05": 1.3333333333333333,\n              "q25": 1.3333333333333333,\n              "q50": 1.3333333333333333,\n              "q75": 1.3333333333333333,\n              "q95": 1.3333333333333333\n            },\n            "frame_y_mean": {\n              "n": 10032,\n              "mean": 151.29554827449297,\n              "var": 45.89208796768109,\n              "q05": 140.40998828125,\n              "q25": 146.89392415364583,\n              "q50": 151.2693359375,\n              "q75": 155.68573893229166,\n              "q95": 162.4778154296875\n            },\n            "y_mean": {\n              "n": 10032,\n              "mean": 155.6826136944967,\n              "var": 130.13233446255578,\n              "q05": 136.8667984008789,\n              "q25": 147.92795181274414,\n              "q50": 155.83558654785156,\n              "q75": 163.58118057250977,\n              "q95": 174.12792510986327\n            },\n            "y_std": {\n              "n": 10032,\n              "mean": 29.953676802880075,\n              "var": 73.51660025014611,\n              "q05": 15.744802856445313,\n              "q25": 23.93495750427246,\n              "q50": 30.09425640106201,\n              "q75": 35.900136947631836,\n              "q95": 44.008992195129395\n            },\n            "y_p10": {\n              "n": 10032,\n              "mean": 118.73763829463977,\n              "var": 560.5050425503916,\n              "q05": 73.80268222808837,\n              "q25": 102.88904209136963,\n              "q50": 123.861079788208,\n              "q75": 137.00472869873047,\n              "q95": 150.05238952636716\n            },\n            "y_p50": {\n              "n": 10032,\n              "mean": 158.98320664249158,\n              "var": 145.1415514473159,\n              "q05": 139.57371711730957,\n              "q25": 150.6834716796875,\n              "q50": 158.9146385192871,\n              "q75": 167.0275421142578,\n              "q95": 179.19177665710447\n            },\n            "y_p90": {\n              "n": 10032,\n              "mean": 186.38956678153227,\n              "var": 287.0178060840724,\n              "q05": 160.2778648376465,\n              "q25": 173.7174224853516,\n              "q50": 185.64989242553713,\n              "q75": 197.79513931274414,\n              "q95": 215.30248764038083\n            },\n            "contrast": {\n              "n": 10032,\n              "mean": 67.65192802785116,\n              "var": 869.3538685090921,\n              "q05": 26.21061782836914,\n              "q25": 44.13491439819336,\n              "q50": 63.77436447143555,\n              "q75": 88.79049301147461,\n              "q95": 120.14251708984374\n            },\n            "very_bright": {\n              "n": 10032,\n              "mean": 5.6784435639287606e-08,\n              "var": 1.537151928674455e-12,\n              "q05": 0.0,\n              "q25": 0.0,\n              "q50": 0.0,\n              "q75": 0.0,\n              "q95": 0.0\n            },\n            "very_dark": {\n              "n": 10032,\n              "mean": 0.0,\n              "var": 0.0,\n              "q05": 0.0,\n              "q25": 0.0,\n              "q50": 0.0,\n              "q75": 0.0,\n              "q95": 0.0\n            },\n            "detail": {\n              "n": 10032,\n              "mean": 7.9486075215980385,\n              "var": 6.836524631497128,\n              "q05": 3.89287109375,\n              "q25": 6.066689610481262,\n              "q50": 7.8241846561431885,\n              "q75": 9.658596277236938,\n              "q95": 12.415809583663938\n            },\n            "lab_a": {\n              "n": 10032,\n              "mean": 1.548149048046252,\n              "var": 93.81593158000446,\n              "q05": -11.36640625,\n              "q25": -5.875,\n              "q50": -0.546875,\n              "q75": 8.275390625,\n              "q95": 19.3125\n            },\n            "lab_b": {\n              "n": 10032,\n              "mean": 6.404843563098086,\n              "var": 47.66745941111487,\n              "q05": -4.296875,\n              "q25": 1.71875,\n              "q50": 6.078125,\n              "q75": 10.546875,\n              "q95": 18.710156249999983\n            },\n            "lighting_range": {\n              "n": 10032,\n              "mean": 20.021714983184182,\n              "var": 145.52417187959475,\n              "q05": 5.645606994628906,\n              "q25": 11.416168212890625,\n              "q50": 17.636642456054688,\n              "q75": 25.657127380371094,\n              "q95": 42.85272674560545\n            },\n            "r_p10": {\n              "n": 10032,\n              "mean": 122.83078115849592,\n              "var": 843.9024052340699,\n              "q05": 72.25,\n              "q25": 102.3291681289673,\n              "q50": 125.08958549499512,\n              "q75": 144.0,\n              "q95": 167.6666717529297\n            },\n            "r_p50": {\n              "n": 10032,\n              "mean": 167.46787159979058,\n              "var": 243.81335583496457,\n              "q05": 143.625,\n              "q25": 156.58333587646484,\n              "q50": 166.5833282470703,\n              "q75": 177.5416660308838,\n              "q95": 194.83333511352538\n            },\n            "r_p90": {\n              "n": 10032,\n              "mean": 189.73281665053665,\n              "var": 229.85646193840952,\n              "q05": 165.92083053588868,\n              "q25": 179.375,\n              "q50": 188.9104209899903,\n              "q75": 199.5833282470703,\n              "q95": 215.875\n            },\n            "g_p10": {\n              "n": 10032,\n              "mean": 117.43019380143575,\n              "var": 560.2681108291806,\n              "q05": 73.98125114440919,\n              "q25": 101.54166412353516,\n              "q50": 120.58958549499512,\n              "q75": 135.05416870117188,\n              "q95": 151.6854171752929\n            },\n            "g_p50": {\n              "n": 10032,\n              "mean": 157.09712873729603,\n              "var": 315.9570630042419,\n              "q05": 127.93958282470703,\n              "q25": 144.0416717529297,\n              "q50": 157.4166717529297,\n              "q75": 170.0416660308838,\n              "q95": 185.9166717529297\n            },\n            "g_p90": {\n              "n": 10032,\n              "mean": 187.1766518092422,\n              "var": 489.9436178101222,\n              "q05": 151.18270950317392,\n              "q25": 170.3333282470703,\n              "q50": 187.75,\n              "q75": 203.21562080383305,\n              "q95": 223.20145492553718\n            },\n            "b_p10": {\n              "n": 10032,\n              "mean": 104.9410540757567,\n              "var": 536.7216484610998,\n              "q05": 64.375,\n              "q25": 89.625,\n              "q50": 106.29166412353516,\n              "q75": 121.375,\n              "q95": 140.7916717529297\n            },\n            "b_p50": {\n              "n": 10032,\n              "mean": 147.03057508643545,\n              "var": 382.9799729408593,\n              "q05": 115.95833244323731,\n              "q25": 133.25,\n              "q50": 146.625,\n              "q75": 160.42187881469727,\n              "q95": 180.0\n            },\n            "b_p90": {\n              "n": 10032,\n              "mean": 182.3092101091784,\n              "var": 615.5828331807069,\n              "q05": 141.0,\n              "q25": 164.66667060852052,\n              "q50": 182.8479209899903,\n              "q75": 200.375,\n              "q95": 222.91250457763664\n            },\n            "dark_ridge": {\n              "n": 10032,\n              "mean": 0.12943713039281243,\n              "var": 0.0035930118071247218,\n              "q05": 0.036717393663194445,\n              "q25": 0.0853610568576389,\n              "q50": 0.125244140625,\n              "q75": 0.16943359375,\n              "q95": 0.23448350694444445\n            }\n          },\n          "target_statistics": {\n            "width": {\n              "n": 40,\n              "mean": 640.0,\n              "var": 0.0,\n              "q05": 640.0,\n              "q25": 640.0,\n              "q50": 640.0,\n              "q75": 640.0,\n              "q95": 640.0\n            },\n            "height": {\n              "n": 40,\n              "mean": 480.0,\n              "var": 0.0,\n              "q05": 480.0,\n              "q25": 480.0,\n              "q50": 480.0,\n              "q75": 480.0,\n              "q95": 480.0\n            },\n            "aspect": {\n              "n": 40,\n              "mean": 1.3333333333333333,\n              "var": 0.0,\n              "q05": 1.3333333333333333,\n              "q25": 1.3333333333333333,\n              "q50": 1.3333333333333333,\n              "q75": 1.3333333333333333,\n              "q95": 1.3333333333333333\n            },\n            "frame_y_mean": {\n              "n": 40,\n              "mean": 129.3877598470052,\n              "var": 227.04159751607412,\n              "q05": 108.03277636718751,\n              "q25": 127.07451578776042,\n              "q50": 133.569970703125,\n              "q75": 137.4266650390625,\n              "q95": 142.24065006510418\n            },\n            "y_mean": {\n              "n": 40,\n              "mean": 150.14063701629638,\n              "var": 448.04424967062334,\n              "q05": 105.40777053833008,\n              "q25": 142.20550918579102,\n              "q50": 154.28058624267578,\n              "q75": 165.46496200561523,\n              "q95": 171.9572799682617\n            },\n            "y_std": {\n              "n": 40,\n              "mean": 63.410143756866454,\n              "var": 84.45266875594243,\n              "q05": 47.934397315979005,\n              "q25": 59.69694900512695,\n              "q50": 64.1294059753418,\n              "q75": 69.64577865600586,\n              "q95": 77.99204025268554\n            },\n            "y_p10": {\n              "n": 40,\n              "mean": 41.933959406614306,\n              "var": 958.1521278850385,\n              "q05": 2.6368062615394594,\n              "q25": 18.843915605545057,\n              "q50": 32.772714614868164,\n              "q75": 61.99440174102783,\n              "q95": 109.34296669006349\n            },\n            "y_p50": {\n              "n": 40,\n              "mean": 173.0009801864624,\n              "var": 645.5022337511291,\n              "q05": 124.96294689178467,\n              "q25": 163.11760330200195,\n              "q50": 181.49776458740234,\n              "q75": 188.64525032043457,\n              "q95": 193.22096557617186\n            },\n            "y_p90": {\n              "n": 40,\n              "mean": 210.17519733428952,\n              "var": 295.3613638653209,\n              "q05": 183.68099594116222,\n              "q25": 208.27590675354006,\n              "q50": 213.28709335327147,\n              "q75": 216.74776496887208,\n              "q95": 233.09896827697756\n            },\n            "contrast": {\n              "n": 40,\n              "mean": 168.2412368774414,\n              "var": 1074.1118161923218,\n              "q05": 99.46612167358398,\n              "q25": 158.83081817626953,\n              "q50": 173.03665924072266,\n              "q75": 189.48519897460938,\n              "q95": 211.66148376464844\n            },\n            "very_bright": {\n              "n": 40,\n              "mean": 0.004967583550347222,\n              "var": 0.00026568657040228086,\n              "q05": 0.0,\n              "q25": 0.0,\n              "q50": 0.00029839409722222225,\n              "q75": 0.002163357204861111,\n              "q95": 0.018929036458333288\n            },\n            "very_dark": {\n              "n": 40,\n              "mean": 0.04101901584201389,\n              "var": 0.0017355358596017327,\n              "q05": 0.0003350151909722223,\n              "q25": 0.014112684461805556,\n              "q50": 0.023491753472222224,\n              "q75": 0.05970594618055555,\n              "q95": 0.14106445312499996\n            },\n            "detail": {\n              "n": 40,\n              "mean": 10.617775154113769,\n              "var": 15.142611563012437,\n              "q05": 4.960670614242554,\n              "q25": 8.365247249603271,\n              "q50": 10.384588241577148,\n              "q75": 11.909588098526001,\n              "q95": 16.43147802352905\n            },\n            "lab_a": {\n              "n": 40,\n              "mean": -3.1611328125,\n              "var": 1.1246114730834962,\n              "q05": -4.60703125,\n              "q25": -3.75,\n              "q50": -3.328125,\n              "q75": -2.48046875,\n              "q95": -1.2265625000000004\n            },\n            "lab_b": {\n              "n": 40,\n              "mean": 14.0974609375,\n              "var": 7.331787071228026,\n              "q05": 10.19765625,\n              "q25": 11.80078125,\n              "q50": 13.8203125,\n              "q75": 16.4296875,\n              "q95": 18.154687499999998\n            },\n            "lighting_range": {\n              "n": 40,\n              "mean": 56.37495832443237,\n              "var": 835.1875440829211,\n              "q05": 22.528936004638673,\n              "q25": 34.36559295654297,\n              "q50": 54.25605773925781,\n              "q75": 70.88351821899414,\n              "q95": 116.78360519409178\n            },\n            "r_p10": {\n              "n": 40,\n              "mean": 48.72812503933907,\n              "var": 966.3664832597153,\n              "q05": 7.367291672229768,\n              "q25": 26.8125,\n              "q50": 39.375,\n              "q75": 67.0510410308838,\n              "q95": 113.98833583831788\n            },\n            "r_p50": {\n              "n": 40,\n              "mean": 179.26197872161865,\n              "var": 661.7000408236996,\n              "q05": 132.64375,\n              "q25": 169.18749618530273,\n              "q50": 188.64583587646484,\n              "q75": 194.12499618530273,\n              "q95": 200.77708358764647\n            },\n            "r_p90": {\n              "n": 40,\n              "mean": 214.31541709899903,\n              "var": 282.4515446650902,\n              "q05": 187.3104118347168,\n              "q25": 210.65625,\n              "q50": 216.91041564941415,\n              "q75": 221.28125381469727,\n              "q95": 240.01041717529296\n            },\n            "g_p10": {\n              "n": 40,\n              "mean": 41.64218744665385,\n              "var": 1019.8722548788097,\n              "q05": 0.48687498867512474,\n              "q25": 17.690624952316284,\n              "q50": 32.35416603088379,\n              "q75": 62.35000019073487,\n              "q95": 110.67562660217285\n            },\n            "g_p50": {\n              "n": 40,\n              "mean": 175.17812471389772,\n              "var": 661.0801369134093,\n              "q05": 126.98125,\n              "q25": 164.6041717529297,\n              "q50": 184.14583206176758,\n              "q75": 191.56250762939453,\n              "q95": 195.47708282470703\n            },\n            "g_p90": {\n              "n": 40,\n              "mean": 212.95385398864747,\n              "var": 299.5880323434888,\n              "q05": 185.92291641235352,\n              "q25": 211.34374618530273,\n              "q50": 216.0833282470703,\n              "q75": 219.43750381469727,\n              "q95": 235.725\n            },\n            "b_p10": {\n              "n": 40,\n              "mean": 25.2124998831749,\n              "var": 711.475031435818,\n              "q05": 0.0,\n              "q25": 4.603124964237219,\n              "q50": 14.979166507720947,\n              "q75": 41.250000953674316,\n              "q95": 88.94229175567627\n            },\n            "b_p50": {\n              "n": 40,\n              "mean": 145.1359375,\n              "var": 583.7990686353174,\n              "q05": 100.77291908264161,\n              "q25": 138.78125,\n              "q50": 150.83333587646484,\n              "q75": 161.79166412353516,\n              "q95": 166.6375\n            },\n            "b_p90": {\n              "n": 40,\n              "mean": 186.04812545776372,\n              "var": 340.0548890277736,\n              "q05": 159.0110397338869,\n              "q25": 181.09375,\n              "q50": 188.01458740234384,\n              "q75": 197.07291793823242,\n              "q95": 209.48958816528318\n            },\n            "dark_ridge": {\n              "n": 40,\n              "mean": 0.14239705403645836,\n              "var": 0.00478912869307362,\n              "q05": 0.05533989800347222,\n              "q25": 0.09922960069444445,\n              "q50": 0.1377902560763889,\n              "q75": 0.1697048611111111,\n              "q95": 0.24119873046874996\n            }\n          }\n        },\n        "engineering_bounds": {\n          "chroma_limit_lab": 3,\n          "chroma_budget_policy": "q75 absolute within-person change; includes zero",\n          "texture_gain_limits": [\n            1,\n            1\n          ],\n          "blur_max_roi64": 0,\n          "lighting": {\n            "max_brightness_offset": 160,\n            "contrast": [\n              0.25,\n              1.5\n            ],\n            "max_left_light": 64\n          },\n          "pixel_refit_policy": {\n            "brightness_offset_limit": 160.0,\n            "contrast_limits": [\n              0.25,\n              1.5\n            ],\n            "side_light_limit": 64.0,\n            "probe_per_class": 4,\n            "full_replay_finalists": 3,\n            "spatial_correlation_q05_minimum": 0.95,\n            "chroma_limit_lab": 3.0,\n            "color_budget": "Observed absolute within-person condition change q75; zero-inclusive",\n            "hair_contrast": "Identity: scalp coverage and hair content confound global contrast",\n            "selection": "Minimize worst remaining primary W1 fraction, then mean fraction",\n            "limits_interpretation": "Engineering search limits, not clinically validated thresholds"\n          }\n        },\n        "limitations": [\n          "Within-person variation includes pose/content; not an isolated camera calibration",\n          "Texture and focus proxies are not independent physical measurements",\n          "Global disease colors/textures are not matched to unlabeled participants",\n          "Four participants are not a general camera population or independent Target Test",\n          "Source/Target brightness and contrast also contain disease and composition differences",\n          "Source face-ROI coverage varies by class; anchors describe successfully measured rows",\n          "Bounds are engineering choices, not validated physical camera calibration",\n          "This profile changes direction and calibration policy as well as amplitude",\n          "The existing Source Test has already been inspected; select on Validation only",\n          "Revised chroma direction remains confounded with disease, person and body region",\n          "Consistency across omissions is a sensitivity check, not independent validation",\n          "Focus gates are content-dependent proxies; no physical camera equivalence established",\n          "Photometric proxy fit to four calibration participants; no physical equivalence established",\n          "Spatial correlation is an engineering guard, not clinical lesion preservation validation",\n          "Source previews, not the entire original Source dataset, enter pixel replay"\n        ],\n        "previous_visual_review": {\n          "profile_version": "measured_bounded_v3",\n          "train_visual_review": "approved",\n          "review_note": "Codex inspected all 120 fixed Train before/after previews (pages 1-15). No gross hair/scale boundary loss or artificial geometry observed at preview resolution. Bounded brightness operator retained for a limited pilot, not a clinical validation. The full Source leakage audit is repeated before training; model inputs and augmentation use float32 RGB 0-255."\n        },\n        "calibration_version": "pixel_replayed_photometric_refit_v1",\n        "calibration_evidence": {\n          "method": "Retained Train pixel replay against unlabeled pilot captures",\n          "proposal": {\n            "source_class_balanced_luminance_anchor": 157.0885009765625,\n            "target_equal_group_luminance_iqr": [\n              175.44100952148438,\n              186.6767463684082\n            ],\n            "smooth_operator_inverse_offsets": [\n              20.323023242164584,\n              35.22991124016547\n            ],\n            "target_to_source_contrast_iqr_ratio": [\n              2.3470476631624733,\n              2.4512758553626286\n            ],\n            "candidate_count": 4\n          },\n          "decision_file_sha256": "43d73ccc5516ad53f76cf67149b12d9fac2fe4092b44a5129bad2647112abf36",\n          "target_accuracy": null,\n          "full_training_executed": false\n        },\n        "revision_evidence": {\n          "color_direction": {\n            "a": {\n              "source_after_lighting_class_balanced_anchor": -2.96875,\n              "target_equal_group_medians": {\n                "P01/alt": -2.984375,\n                "P01/ref": -1.671875,\n                "P02/alt": -3.625,\n                "P02/ref": -4.25,\n                "P03/alt": -3.640625,\n                "P03/ref": -2.828125,\n                "P04/alt": -3.640625,\n                "P04/ref": -2.46875\n              },\n              "full_direction": 0,\n              "leave_one_person_out_directions": {\n                "P01": -1,\n                "P02": 0,\n                "P03": 0,\n                "P04": 0\n              },\n              "direction_stable": false,\n              "within_person_absolute_q75_limit": 1.171875,\n              "proposed_range": [\n                0.0,\n                0.0\n              ],\n              "policy": "Zero-inclusive bounded shift; direction stable across four omissions"\n            },\n            "b": {\n              "source_after_lighting_class_balanced_anchor": 5.4453125,\n              "target_equal_group_medians": {\n                "P01/alt": 12.46875,\n                "P01/ref": 13.109375,\n                "P02/alt": 10.671875,\n                "P02/ref": 12.875,\n                "P03/alt": 16.328125,\n                "P03/ref": 15.9375,\n                "P04/alt": 14.5625,\n                "P04/ref": 14.9375\n              },\n              "full_direction": 1,\n              "leave_one_person_out_directions": {\n                "P01": 1,\n                "P02": 1,\n                "P03": 1,\n                "P04": 1\n              },\n              "direction_stable": true,\n              "within_person_absolute_q75_limit": 1.03515625,\n              "proposed_range": [\n                0.0,\n                1.03515625\n              ],\n              "policy": "Zero-inclusive bounded shift; direction stable across four omissions"\n            }\n          },\n          "color_choice": "none",\n          "focus_choice": "none",\n          "source_preview_count": 120,\n          "target_count": 40,\n          "seeds": [\n            42,\n            707,\n            1707\n          ],\n          "selection_basis": "Unlabeled Train/calibration proxy checks, not classifier scores"\n        },\n        "previous_engineering_bounds": {\n          "measured_change_fraction": 1.0,\n          "chroma_limit_lab": 3,\n          "texture_gain_limits": [\n            0.85,\n            1.15\n          ],\n          "focus_fit_grid_roi64": [\n            0,\n            1.5,\n            0.05\n          ],\n          "blur_max_roi64": 0.75,\n          "brightness_offset_limit": 32,\n          "contrast_limits": [\n            0.8,\n            1.25\n          ]\n        },\n        "pre_refit_engineering_bounds": {\n          "chroma_limit_lab": 3,\n          "chroma_budget_policy": "q75 absolute within-person change; includes zero",\n          "texture_gain_limits": [\n            1.0,\n            1.0\n          ],\n          "blur_max_roi64": 0.0,\n          "lighting": {\n            "max_brightness_offset": 32,\n            "contrast": [\n              0.8,\n              1.25\n            ]\n          }\n        },\n        "parent_calibration_evidence": {\n          "source_role": "retained_original_train",\n          "target_role": "unlabeled_calibration",\n          "source_class_balanced_anchors": {\n            "y_p50": 160.440185546875,\n            "contrast": 70.79170608520508\n          },\n          "source_measurement_coverage": {\n            "y_p50": {\n              "class_medians": [\n                150.06185150146484,\n                160.9347801208496,\n                160.440185546875,\n                157.4539794921875,\n                165.97833251953125\n              ],\n              "coverage_by_class": [\n                {\n                  "total": 1990,\n                  "measured": 1990\n                },\n                {\n                  "total": 2050,\n                  "measured": 2050\n                },\n                {\n                  "total": 1929,\n                  "measured": 1929\n                },\n                {\n                  "total": 1965,\n                  "measured": 1965\n                },\n                {\n                  "total": 2098,\n                  "measured": 2098\n                }\n              ]\n            },\n            "contrast": {\n              "class_medians": [\n                58.59663391113281,\n                70.79170608520508,\n                83.7009048461914,\n                35.322113037109375,\n                71.07597732543945\n              ],\n              "coverage_by_class": [\n                {\n                  "total": 1990,\n                  "measured": 1990\n                },\n                {\n                  "total": 2050,\n                  "measured": 2050\n                },\n                {\n                  "total": 1929,\n                  "measured": 1929\n                },\n                {\n                  "total": 1965,\n                  "measured": 1965\n                },\n                {\n                  "total": 2098,\n                  "measured": 2098\n                }\n              ]\n            }\n          },\n          "target_equal_participant_condition_medians": {\n            "y_p50": [\n              188.3468475341797,\n              162.00440216064453,\n              179.97626495361328,\n              188.29984283447266,\n              181.40353393554688,\n              180.3260040283203,\n              186.1680450439453,\n              161.74720764160156\n            ],\n            "contrast": [\n              145.20181274414062,\n              173.9080047607422,\n              174.50856018066406,\n              177.37515258789062,\n              172.16531372070312,\n              183.07321166992188,\n              168.57965087890625,\n              189.6483917236328\n            ]\n          },\n          "raw_brightness_offsets": [\n            15.043113708496094,\n            26.26080894470215\n          ],\n          "raw_contrast_ratios": [\n            2.4193356465249507,\n            2.5257149070993528\n          ],\n          "brightness_limit": 32,\n          "contrast_limits": [\n            0.8,\n            1.25\n          ],\n          "hair_global_contrast": "identity; coverage and content remain confounded",\n          "revision_policy": "direction_and_focus_audited_v1"\n        }\n      },\n      "previous_reference_sha256": "1a44a143fe6f5ef3021865b75f4797ffff16b1e21bae7dc7ca838eb70a09a81f",\n      "source_manifest_sha256": "c6320fa7d2ef25332d508b3da868d2731d61ce7c8cd3ed2ebc505e6cd0b54717",\n      "previous_best_sha256": "fa2c59ae6a0b1342b8c693014d0a951c59064a4c8439a313f2520e1ce8c224fd",\n      "measurement_provenance": {\n        "source": "camera_capture/_review/domain_adaptation_colab_20261109/hair/analysis/analysis.json",\n        "source_sha256": "e006e425e2711f0d6da7eef2856c88fa7128fd256888ad13dd23409ee85e4e2d",\n        "target": "camera_capture/_review/training_reference_inputs_20261110/hair/calibration.json",\n        "target_sha256": "16b6d45f549f2e6f533eb0b65f22085e6c2865833111d32b2d6ae09187193183",\n        "capture_fingerprint": "2d2ab5106e7f079147d4cd1e2e6e0ce1c27e8494d554085b4ad4947e3cd2b22c",\n        "retained_train_count": 10032,\n        "target_count": 40\n      }\n    }\n  },\n  "configuration_sha256": "ee9604c652ace8a6d7d0df5656b70ef7512f0ae40f7ef2211f081432b75d9366",\n  "parent_study_sha256": "d1054f86cf155c7cc2fe8a20a386ca9cf206f6a7e47fa53798bc179e002d6be5",\n  "revision_schema": "train_preview_pilot_pixel_replay_v1",\n  "revision_decision_sha256": "21f5faa3679f5b91806aa6d6b52a370d409846b6948bcc23cfcace882f79710d",\n  "parent_runtime_sha256": {\n    "adaptation_strength.py": "4f52fc89b9e56dab6842bd44de97d9b760a2e5b8335e7efd434db474f4f4ba15",\n    "adaptation_stage1.py": "6b5aae35bcbe1af85fe19b6a47ebce6ed57bfca780129a82e95dd9dd0c9d4c8a",\n    "adaptation_stage1_colab.py": "ddd976b6e3d7e605cb4f6b8af4a95f647cd60f819348542094b2e0e18c80e854",\n    "adaptation_stage2.py": "764f19cf835450eb31224079d33d625a5aa9ff890a56c04285f862dc275696b9"\n  },\n  "pretraining_check_revision": "early_required_inputs_and_full_cache_space_v1",\n  "revision_decision_file_sha256": "43d73ccc5516ad53f76cf67149b12d9fac2fe4092b44a5129bad2647112abf36",\n  "revision_summary_file_sha256": "9232d23cbcb3e9ab24283888d5d16adc797ef0a944609e10050115333a5cd3f6",\n  "notebook_builder_sha256": "56de759444b2b0dbe8d142e69d24479cbd3dbd2d0553f2cd15f5f69e217066d8",\n  "refit_schema": "train_preview_pilot_pixel_replay_v1",\n  "refit_parent_study_sha256": "492ee01afb4579b6402fca29459c59d56618243589642fc022db18bbbe45a165",\n  "refit_policy": {\n    "brightness_offset_limit": 160.0,\n    "contrast_limits": [\n      0.25,\n      1.5\n    ],\n    "side_light_limit": 64.0,\n    "probe_per_class": 4,\n    "full_replay_finalists": 3,\n    "spatial_correlation_q05_minimum": 0.95,\n    "chroma_limit_lab": 3.0,\n    "color_budget": "Observed absolute within-person condition change q75; zero-inclusive",\n    "hair_contrast": "Identity: scalp coverage and hair content confound global contrast",\n    "selection": "Minimize worst remaining primary W1 fraction, then mean fraction",\n    "limits_interpretation": "Engineering search limits, not clinically validated thresholds"\n  },\n  "refit_visual_review_sha256": "fafa68cc60b0bd13f463148fd77e6bf8b87165b14db0d7fc8ec44f5887f61c27",\n  "parent_revision_schema": "camera_direction_and_focus_audit_v1"\n}\n')
    env = {**os.environ, "PYTHONPATH": str(runtime / "src"), "MPLBACKEND": "Agg",
           "PYTHONIOENCODING": "utf-8", "PYTHONUNBUFFERED": "1",
           "PYTHONDONTWRITEBYTECODE": "1"}
    command = [sys.executable, "-u", "-m", "mediflow_datasets.adaptation_strength",
               "--launch", str(runtime), str(work), str(project), str(study_path), str(my_drive)]
    with (
        subprocess.Popen(command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, encoding="utf-8", errors="replace", bufsize=1,
                         start_new_session=True) as process,
        log_path.open("x", encoding="utf-8") as log,
    ):
        try:
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line, end="", flush=True)
            if process.wait():
                raise RuntimeError(f"후속 학습 미완료 · 로그를 확인하세요. {log_path}")
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
            for diagnostic in work.glob("*.log"):
                shutil.copyfile(diagnostic, logs / f"{run_id}_{diagnostic.name}")
    names = ["study.json", "summary.json", "selection.json", "validation_comparison.csv",
             "plots/learning_curves.png"]
    names.append("startup_preflight.json")
    names.extend("code/" + name for name in (
        "adaptation_strength.py", "adaptation_stage1.py", "camera_extended.py"))
    for domain in ("skin", "web_skin", "hair"):
        names.extend(f"{domain}/{name}" for name in (
            "best.keras", "complete.json", "run_configuration.json", "data_audit.json",
            "source_manifest.json", "identity.json", "expanded/history.json"))
    local_zip = Path("/content/MediFlow_Revised_Camera_Results_20261111.zip")
    with ZipFile(local_zip, "w", compression=8) as archive:
        for name in names:
            archive.write(out / name, name)
    shutil.copyfile(local_zip, out / local_zip.name)
print("세 분야 후속 학습 완료", out, flush=True)
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv(out / "validation_comparison.csv", dtype=str, keep_default_na=False))
display(Image(filename=str(out / "plots/learning_curves.png")))
files.download(str(local_zip))
